# Hospital Readmission within 30 Days
## Final master notebook, reproducibility checks and full experiment archive


### Read this before running

This notebook has **two parts**:

1. **The active reproduction (Section 1–12):** which is the selected original feature-building and training cells in the order of dependancies, starting from the competition CSVs and ending with the three actual selected submission files (`FINAL_A`, `FINAL_B`, `FINAL_C`), a default `submission.csv`, model comparisons, leakage checks and a feature inventory. It is intended to run top-to-bottom in **Kaggle with the original competition dataset attached**.
2. **CODE ARCHIVE:** all 47 original cells from the master notebook and all six original NOVA cells, preserved as-is inside disabled Python blocks. These document V1–V6, V8–V12, NOVA and failed/intermediate attempts without rerunning duplicate or superseded experiments. **No V7 or V13 was created**

**Methodological caveat:** this preserves some original train/test-wide *unsupervised category-vocabulary selection* (in V8–V11) and original exploratory model-selection procedures. These are disclosed below; they are not equivalent to fully nested, fold-isolated preprocessing. V5's XGBoost imputers and ordinal encoder were correctly fitted on each fold's training subset, but V5 also used three V4 lab-unit summary features that lacked an admission-to-discharge filter. Final blends retain some V5 predictions; this historical pipeline must not be called fully leakage-free. The reported leaderboard results are historical observations, not new measurements from this file.

## 1. Environment, input files and reproducibility controls

Used fixed random seeds and check all required CSVs before doing expensive work. The notebook uses the Kaggle competition input directory and writes output files to `/kaggle/working/`. Do not install external data or load any labels from the test set.

In [ ]:

import os, sys, platform, random
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DATA_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    'train.csv','test.csv','index_encounters.csv','patients.csv',
    'hospitals.csv','prior_utilization.csv','diagnoses.csv',
    'discharge_planning.csv','medications.csv','procedures.csv',
    'unit_transfers.csv','lab_results.csv','vitals.csv','icd10_lookup.csv'
]
file_locations = {p.name.lower():p for p in DATA_ROOT.rglob('*.csv')}
missing_files = [f for f in required_files if f not in file_locations]
if missing_files:
    raise FileNotFoundError(
        'Attach the original Kaggle competition dataset under /kaggle/input. '
        'Missing files: ' + ', '.join(missing_files)
    )

import sklearn, catboost, xgboost, lightgbm
print('Python:',sys.version.split()[0], '| pandas:',pd.__version__,
      '| sklearn:',sklearn.__version__, '| catboost:',catboost.__version__,
      '| xgboost:',xgboost.__version__, '| lightgbm:',lightgbm.__version__)
print('Dataset files found:',len(file_locations))
print('Outputs:',OUTPUT_DIR)


## 2. Input audit and V1: One row per encounter

**Why:** The raw dataset has one-to-many tables. Joining event tables directly would duplicate encounter rows and invalidate training. So I started from the unique encounter index and join patient/hospital information using validated keys. Previous utilization is filtered to visits **before the index admission**. Length of stay and age are checked for implausible values. Discharge information is permitted only because the prediction is made at discharge.

### 2.1 Inspect files, IDs and target prevalence

Check train/test counts, duplicates, key coverage and the positive-label fraction before building features.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 1.

In [ ]:

from pathlib import Path
import pandas as pd
import csv

# Find the competition's CSV files
files = {}
for path in Path("/kaggle/input").rglob("*.csv"):
    files[path.name.lower()] = path

expected = [
    "train.csv", "test.csv", "sample_submission.csv",
    "index_encounters.csv", "patients.csv",
    "hospitals.csv", "diagnoses.csv",
    "procedures.csv", "lab_results.csv",
    "vitals.csv", "medications.csv",
    "unit_transfers.csv", "prior_utilization.csv",
    "discharge_planning.csv", "icd10_lookup.csv",
    "data_dictionary.csv"
]

print("DATASET AUDIT")
print("=" * 55)

for name in expected:
    if name not in files:
        print("MISSING:", name)
        continue

    path = files[name]

    # Count CSV records without loading the full file
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = sum(1 for row in reader)

    print(f"{name}: {rows:,} rows")
    print("  Columns:", header)

print("\nTRAIN / TEST CHECK")
print("=" * 55)

train = pd.read_csv(files["train.csv"])
test = pd.read_csv(files["test.csv"])
enc = pd.read_csv(files["index_encounters.csv"])
patients = pd.read_csv(files["patients.csv"])

print("Train:", train.shape)
print("Test:", test.shape)
print("Encounters:", enc.shape)
print("Patients:", patients.shape)

print("\nReadmission rate:")
print(train["readmit_30d"].value_counts(normalize=True))

print("\nDuplicate encounter IDs:")
print("Train:", train.encounter_id.duplicated().sum())
print("Test:", test.encounter_id.duplicated().sum())
print("Index:", enc.encounter_id.duplicated().sum())

print("\nTrain/test ID overlap:",
      len(set(train.encounter_id) & set(test.encounter_id)))

print("Train IDs missing from index:",
      (~train.encounter_id.isin(enc.encounter_id)).sum())

print("Test IDs missing from index:",
      (~test.encounter_id.isin(enc.encounter_id)).sum())

print("\nAUDIT COMPLETE")


### 2.2 Construct V1 encounter-level table

Use `index_encounters` as the spine, join `patients` and `hospitals`, calculate age/stay and summarize historical visits and diagnoses. All rows are aligned back to the original train/test encounter order.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 2.

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# Find the competition data files
files = {
    p.name.lower(): p
    for p in Path("/kaggle/input").rglob("*.csv")
}

def load(name):
    return pd.read_csv(files[name])

# 1. Load the main tables
train = load("train.csv")
test = load("test.csv")
enc = load("index_encounters.csv")
patients = load("patients.csv")
hospitals = load("hospitals.csv")

# 2. Start with all encounters
master = enc.copy()

# 3. Add patient and hospital information
master = master.merge(
    patients, on="patient_id", how="left",
    validate="one_to_one"
)

master = master.merge(
    hospitals, on="hospital_id", how="left",
    validate="many_to_one"
)

# 4. Convert dates into real date values
master["admit_dt"] = pd.to_datetime(
    master["admit_dt"], errors="coerce"
)

master["discharge_dt"] = pd.to_datetime(
    master["discharge_dt"], errors="coerce"
)

# 5. Calculate age and length of stay
master["age"] = (
    master["admit_dt"].dt.year -
    master["birth_year"]
)

master["stay_days"] = (
    master["discharge_dt"] -
    master["admit_dt"]
).dt.total_seconds() / 86400

# Keep missing values for impossible dates
master.loc[
    (master["stay_days"] < 0) |
    (master["stay_days"] > 365),
    "stay_days"
] = np.nan

master.loc[
    (master["age"] < 18) |
    (master["age"] > 110),
    "age"
] = np.nan

# 6. Add historical visit counts
prior = load("prior_utilization.csv")
prior["visit_dt"] = pd.to_datetime(
    prior["visit_dt"], errors="coerce"
)

prior = prior.merge(
    master[["patient_id", "admit_dt"]],
    on="patient_id",
    how="inner",
    validate="many_to_one"
)

# Only count visits before current admission
prior = prior[
    prior["visit_dt"] < prior["admit_dt"]
].copy()

prior["days_before_admit"] = (
    prior["admit_dt"] - prior["visit_dt"]
).dt.total_seconds() / 86400

prior["within_90d"] = (
    prior["days_before_admit"] <= 90
).astype(int)

visit_counts = prior.groupby("patient_id").agg(
    previous_visits=("visit_dt", "count"),
    visits_90d=("within_90d", "sum"),
    days_since_last_visit=("days_before_admit", "min"),
    hospitals_visited=("hospital_id", "nunique")
).reset_index()

master = master.merge(
    visit_counts, on="patient_id",
    how="left", validate="one_to_one"
)

master["previous_visits"] = (
    master["previous_visits"].fillna(0)
)

master["visits_90d"] = (
    master["visits_90d"].fillna(0)
)

master["hospitals_visited"] = (
    master["hospitals_visited"].fillna(0)
)

# 7. Count diagnoses per encounter
diag = load("diagnoses.csv")

diag_counts = diag.groupby("encounter_id").agg(
    diagnosis_count=("icd10_code", "count"),
    unique_diagnoses=("icd10_code", "nunique")
).reset_index()

master = master.merge(
    diag_counts, on="encounter_id",
    how="left", validate="one_to_one"
)

for col in ["diagnosis_count", "unique_diagnoses"]:
    master[col] = master[col].fillna(0)

# 8. Add discharge-planning information
discharge = load("discharge_planning.csv")

master = master.merge(
    discharge, on="encounter_id",
    how="left", validate="one_to_one"
)

# 9. Create simple date features
master["admit_month"] = master["admit_dt"].dt.month
master["discharge_weekday"] = (
    master["discharge_dt"].dt.dayofweek
)

# 10. Remove identifiers and raw date columns
drop_cols = [
    "encounter_id", "patient_id",
    "attending_id", "admit_dt",
    "discharge_dt", "birth_year"
]

features = master.drop(columns=drop_cols)

# 11. Standardize categorical text
cat_cols = features.select_dtypes(
    include=["object", "string"]
).columns.tolist()

for col in cat_cols:
    features[col] = (
        features[col]
        .fillna("Missing")
        .astype(str)
        .str.strip()
        .str.lower()
    )

# 12. Separate training and test data
train_index = master["encounter_id"].isin(
    train["encounter_id"]
)

test_index = master["encounter_id"].isin(
    test["encounter_id"]
)

X_train = features.loc[train_index].copy()
X_test = features.loc[test_index].copy()

# Put rows in the same order as train/test
train_order = master.loc[
    train_index, "encounter_id"
]

test_order = master.loc[
    test_index, "encounter_id"
]

X_train.index = train_order
X_test.index = test_order

X_train = X_train.loc[
    train["encounter_id"]
].reset_index(drop=True)

X_test = X_test.loc[
    test["encounter_id"]
].reset_index(drop=True)

y_train = train["readmit_30d"]

# 13. Final checks
assert len(X_train) == len(train)
assert len(X_test) == len(test)
assert X_train.columns.equals(X_test.columns)

print("MASTER TABLE CREATED SUCCESSFULLY")
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Categorical features:", len(cat_cols))
print("Total features:", X_train.shape[1])
print("Missing numeric values:",
      X_train.select_dtypes(include="number")
      .isna().sum().sum())


## 3. V2: Clinical and utilization summaries

**Why:** Readmission is influenced by prior emergency/inpatient visits, comorbidity burden, medication burden, procedures and intensive-care exposure. Each event table is aggregated to **one row per encounter** before joining. Prior visits must precede admission; future visits must never enter the features.

### 3.1 Add V2 historical, diagnosis, medication, procedure and transfer features

Uses only raw competition files; chronic-diagnosis flags come from the supplied ICD-10 lookup. Categorical features are standardized and missing counts are handled.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 6.

In [ ]:

# additional V2 features

import pandas as pd
import numpy as np

extra = master[["encounter_id", "patient_id"]].copy()

# A. PREVIOUS HOSPITAL VISITS

prior2 = load("prior_utilization.csv")

prior2["visit_dt"] = pd.to_datetime(
    prior2["visit_dt"], errors="coerce"
)

prior2 = prior2.merge(
    master[["patient_id", "admit_dt"]],
    on="patient_id",
    how="inner",
    validate="many_to_one"
)

# Only visits BEFORE the index admission
prior2 = prior2[
    prior2["visit_dt"] < prior2["admit_dt"]
].copy()

prior2["days_ago"] = (
    prior2["admit_dt"] - prior2["visit_dt"]
).dt.total_seconds() / 86400

visit_type = (
    prior2["visit_type"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
)

prior2["is_ed"] = visit_type.isin(
    ["ED", "ER", "EMERGENCY"]
).astype(int)

prior2["is_ip"] = visit_type.isin(
    ["IP", "INPATIENT"]
).astype(int)

prior2["is_op"] = visit_type.isin(
    ["OP", "OUTPATIENT"]
).astype(int)

for days in [30, 90, 365]:
    recent = prior2["days_ago"] <= days

    prior2[f"visits_{days}d_v2"] = recent.astype(int)
    prior2[f"ed_{days}d"] = (
        recent & (prior2["is_ed"] == 1)
    ).astype(int)

    prior2[f"ip_{days}d"] = (
        recent & (prior2["is_ip"] == 1)
    ).astype(int)

visit_features = prior2.groupby("patient_id").agg({
    "is_ed": "sum",
    "is_ip": "sum",
    "is_op": "sum",
    **{
        f"{prefix}_{days}d": "sum"
        for days in [30, 90, 365]
        for prefix in ["visits", "ed", "ip"]
        if prefix != "visits"
    },
    **{
        f"visits_{days}d_v2": "sum"
        for days in [30, 90, 365]
    }
}).reset_index()

visit_features = visit_features.rename(columns={
    "is_ed": "total_ed_visits",
    "is_ip": "total_ip_visits",
    "is_op": "total_op_visits"
})

extra = extra.merge(
    visit_features,
    on="patient_id",
    how="left",
    validate="one_to_one"
)

# B. DIAGNOSIS FEATURES


diag2 = load("diagnoses.csv")
lookup = load("icd10_lookup.csv")

# Match diagnosis codes to lookup descriptions
diag2["icd10_code"] = (
    diag2["icd10_code"].astype("string")
    .str.strip().str.upper()
)

lookup["icd10_code"] = (
    lookup["icd10_code"].astype("string")
    .str.strip().str.upper()
)

lookup = lookup.drop_duplicates("icd10_code")

diag2 = diag2.merge(
    lookup[["icd10_code", "chapter", "chronic_flag"]],
    on="icd10_code",
    how="left",
    validate="many_to_one"
)

diag2["is_chronic"] = (
    diag2["chronic_flag"]
    .astype("string")
    .str.upper()
    .isin(["1", "Y", "YES", "TRUE"])
    .astype(int)
)

diag2["primary_num"] = pd.to_numeric(
    diag2["is_primary"], errors="coerce"
).fillna(0)

diag2["seq_num"] = pd.to_numeric(
    diag2["seq"], errors="coerce"
)

diag_features = diag2.groupby("encounter_id").agg(
    chronic_diagnoses=("is_chronic", "sum"),
    diagnosis_chapters=("chapter", "nunique"),
    primary_diagnosis_count=("primary_num", "sum")
).reset_index()

# the primary diagnosis
primary_diag = diag2.sort_values(
    ["encounter_id", "primary_num", "seq_num"],
    ascending=[True, False, True]
).drop_duplicates("encounter_id")

primary_diag = primary_diag[
    ["encounter_id", "icd10_code", "chapter"]
].rename(columns={
    "icd10_code": "primary_icd_code",
    "chapter": "primary_icd_chapter"
})

diag_features = diag_features.merge(
    primary_diag,
    on="encounter_id",
    how="left",
    validate="one_to_one"
)

extra = extra.merge(
    diag_features,
    on="encounter_id",
    how="left",
    validate="one_to_one"
)

# C. MEDICATION FEATURES

med = load("medications.csv")

med["drug_class"] = (
    med["drug_class"]
    .fillna("Unknown")
    .astype(str)
    .str.upper()
    .str.strip()
)

med["continued_num"] = pd.to_numeric(
    med["continued_at_discharge"],
    errors="coerce"
)

med_features = med.groupby("encounter_id").agg(
    medication_count=("drug_name", "count"),
    medication_classes=("drug_class", "nunique"),
    continued_medications=("continued_num", "sum")
).reset_index()

extra = extra.merge(
    med_features,
    on="encounter_id",
    how="left",
    validate="one_to_one"
)


# D. PROCEDURE FEATURES

proc = load("procedures.csv")

proc_features = proc.groupby("encounter_id").agg(
    procedure_count=("code", "count"),
    unique_procedures=("code", "nunique")
).reset_index()

extra = extra.merge(
    proc_features,
    on="encounter_id",
    how="left",
    validate="one_to_one"
)


# E. HOSPITAL UNIT TRANSFERS


transfers = load("unit_transfers.csv")

transfers["in_dt"] = pd.to_datetime(
    transfers["in_dt"], errors="coerce"
)

transfers["out_dt"] = pd.to_datetime(
    transfers["out_dt"], errors="coerce"
)

transfers["unit"] = (
    transfers["unit"]
    .fillna("Unknown")
    .astype(str)
    .str.upper()
    .str.strip()
)

transfers["unit_hours"] = (
    transfers["out_dt"] - transfers["in_dt"]
).dt.total_seconds() / 3600

# Remove impossible durations
transfers.loc[
    (transfers["unit_hours"] < 0) |
    (transfers["unit_hours"] > 8760),
    "unit_hours"
] = np.nan

transfers["icu_flag"] = (
    transfers["unit"].str.contains(
        r"\bICU\b", regex=True
    )
).astype(int)

transfers["icu_hours"] = np.where(
    transfers["icu_flag"] == 1,
    transfers["unit_hours"],
    0
)

transfer_features = transfers.groupby("encounter_id").agg(
    unit_records=("unit", "count"),
    unique_units=("unit", "nunique"),
    icu_exposure=("icu_flag", "max"),
    total_icu_hours=("icu_hours", "sum"),
    total_unit_hours=("unit_hours", "sum")
).reset_index()

extra = extra.merge(
    transfer_features,
    on="encounter_id",
    how="left",
    validate="one_to_one"
)


# F. COMBINE WITH ORIGINAL V1 FEATURES


# Keep original features in their original row order
v2_features = features.reset_index(drop=True).copy()

extra_features = extra.drop(
    columns=["encounter_id", "patient_id"]
).reset_index(drop=True)

v2_features = pd.concat(
    [v2_features, extra_features],
    axis=1
)

# Count-based missing values mean zero records
count_columns = [
    c for c in extra_features.columns
    if c not in [
        "primary_icd_code",
        "primary_icd_chapter"
    ]
]

v2_features[count_columns] = (
    v2_features[count_columns].fillna(0)
)

# Standardize categorical columns
cat_cols_v2 = v2_features.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

for col in cat_cols_v2:
    v2_features[col] = (
        v2_features[col]
        .fillna("Missing")
        .astype(str)
        .str.strip()
        .str.lower()
    )

# Use encounter IDs to guarantee correct row alignment
v2_features.index = master["encounter_id"]

X_train_v2 = v2_features.loc[
    train["encounter_id"]
].reset_index(drop=True)

X_test_v2 = v2_features.loc[
    test["encounter_id"]
].reset_index(drop=True)

# Final checks
assert len(X_train_v2) == 76975
assert len(X_test_v2) == 32812
assert X_train_v2.columns.is_unique
assert X_train_v2.columns.equals(X_test_v2.columns)

print("V2 FEATURE ENGINEERING COMPLETE")
print("V1 features:", X_train.shape[1])
print("V2 features:", X_train_v2.shape[1])
print("New features:",
      X_train_v2.shape[1] - X_train.shape[1])
print("Training rows:", len(X_train_v2))
print("Test rows:", len(X_test_v2))


## 4. V3: Laboratory and vital-sign trends

**Why:** Averages alone can hide deterioration. We summarize the first, last, minimum, maximum, standard deviation and change in clinical measurements. Observations are restricted to the interval **admission ≤ measurement time ≤ discharge**. Where specified, lab units are normalized before aggregation; incompatible or unknown units are excluded from numeric lab statistics.

### 4.1 Clean and aggregate laboratory measurements

Includes unit conversions, abnormal flags, test counts and first-to-last changes. Clinical time windows are applied before aggregation.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 9.

In [ ]:

# V3 LABORATORY FEATURES

import pandas as pd
import numpy as np

labs = pd.read_csv(
    files["lab_results.csv"],
    usecols=[
        "encounter_id", "lab_name",
        "collected_dt", "value",
        "unit", "abnormal_flag"
    ]
)

print("Original lab rows:", len(labs))

# Convert dates and numeric measurements
labs["collected_dt"] = pd.to_datetime(
    labs["collected_dt"], errors="coerce"
)
labs["value"] = pd.to_numeric(
    labs["value"], errors="coerce"
)

# Attach admission/discharge timestamps
labs = labs.merge(
    master[[
        "encounter_id", "admit_dt", "discharge_dt"
    ]],
    on="encounter_id",
    how="inner",
    validate="many_to_one"
)

# Only use measurements 
labs = labs[
    (labs["collected_dt"] >= labs["admit_dt"]) &
    (labs["collected_dt"] <= labs["discharge_dt"])
].copy()

# Standardize test names and units
labs["lab_key"] = (
    labs["lab_name"].astype("string")
    .str.strip().str.lower()
)

labs["unit_clean"] = (
    labs["unit"].astype("string")
    .str.strip().str.lower()
    .str.replace("µ", "u", regex=False)
    .str.replace("μ", "u", regex=False)
)

# Convert creatinine from umol/L to mg/dL
mask = (
    (labs["lab_key"] == "creatinine") &
    (labs["unit_clean"] == "umol/l")
)
labs.loc[mask, "value"] /= 88.4

# Convert hemoglobin from g/L to g/dL
mask = (
    (labs["lab_key"] == "hemoglobin") &
    (labs["unit_clean"] == "g/l")
)
labs.loc[mask, "value"] /= 10

# Convert BUN from mmol/L to mg/dL
mask = (
    (labs["lab_key"] == "bun") &
    (labs["unit_clean"] == "mmol/l")
)
labs.loc[mask, "value"] *= 2.801

# Convert glucose from mmol/L to mg/dL
mask = (
    (labs["lab_key"] == "glucose") &
    (labs["unit_clean"] == "mmol/l")
)
labs.loc[mask, "value"] *= 18.018

# Select the common tests
selected_labs = [
    "creatinine", "bun", "hemoglobin",
    "wbc", "potassium", "sodium", "glucose"
]

# Overall number of tests and abnormal flags
labs["flag_clean"] = (
    labs["abnormal_flag"].fillna("")
    .astype(str).str.strip().str.upper()
)

labs["abnormal_num"] = (
    labs["flag_clean"].isin(["H", "L"])
).astype(int)

labs["high_num"] = (
    labs["flag_clean"] == "H"
).astype(int)

labs["low_num"] = (
    labs["flag_clean"] == "L"
).astype(int)

lab_overall = labs.groupby("encounter_id").agg(
    lab_total_count=("value", "count"),
    lab_abnormal_count=("abnormal_num", "sum"),
    lab_high_count=("high_num", "sum"),
    lab_low_count=("low_num", "sum")
)

lab_overall["lab_abnormal_fraction"] = (
    lab_overall["lab_abnormal_count"] /
    lab_overall["lab_total_count"].replace(0, np.nan)
)

# Summarize individual laboratory tests
labs_selected = labs[
    labs["lab_key"].isin(selected_labs)
].copy()

# Avoid mixing incompatible units
allowed_units = {
    "creatinine": ["mg/dl", "umol/l"],
    "bun": ["mg/dl", "mmol/l"],
    "hemoglobin": ["g/dl", "g/l"],
    "wbc": ["10^3/ul", "10^9/l"],
    "potassium": ["mmol/l", "meq/l"],
    "sodium": ["mmol/l", "meq/l"],
    "glucose": ["mg/dl", "mmol/l"]
}

valid_units = pd.Series(False, index=labs_selected.index)

for test_name, units in allowed_units.items():
    valid_units |= (
        (labs_selected["lab_key"] == test_name) &
        (labs_selected["unit_clean"].isin(units))
    )

print("Selected test rows:", len(labs_selected))
print("Recognized unit rows:", int(valid_units.sum()))
print("Units not recognized:")
print(
    labs_selected.loc[~valid_units]
    .groupby(["lab_key", "unit_clean"])
    .size().head(20)
)

labs_selected = labs_selected[valid_units].copy()

labs_selected = labs_selected.sort_values(
    ["encounter_id", "lab_key", "collected_dt"]
)

lab_stats = (
    labs_selected
    .groupby(["encounter_id", "lab_key"])["value"]
    .agg(["count", "mean", "min", "max",
          "std", "first", "last"])
    .unstack("lab_key")
)

lab_stats.columns = [
    f"lab_{test}_{stat}"
    for stat, test in lab_stats.columns
]

# Calculate first-to-last changes
for test_name in selected_labs:
    first_col = f"lab_{test_name}_first"
    last_col = f"lab_{test_name}_last"

    if first_col in lab_stats and last_col in lab_stats:
        lab_stats[f"lab_{test_name}_change"] = (
            lab_stats[last_col] - lab_stats[first_col]
        )

lab_features = lab_overall.join(
    lab_stats, how="left"
)

print("\nLAB FEATURES CREATED")
print("Feature count:", lab_features.shape[1])
print("Encounters covered:", len(lab_features))


### 4.2 Clean and aggregate vital signs

Includes physiological-range checks, Fahrenheit-to-Celsius conversion, and time-ordered first/last measurements.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 10.

In [ ]:

# V3 VITAL SIGN FEATURES

vitals = pd.read_csv(
    files["vitals.csv"],
    usecols=[
        "encounter_id", "charttime",
        "vital_type", "value"
    ]
)

print("Original vital rows:", len(vitals))

# Convert dates and numeric values
vitals["charttime"] = pd.to_datetime(
    vitals["charttime"], errors="coerce"
)

vitals["value"] = pd.to_numeric(
    vitals["value"], errors="coerce"
)

vitals["vital_type"] = (
    vitals["vital_type"].astype("string")
    .str.strip().str.upper()
)

# Attach admission/discharge dates
vitals = vitals.merge(
    master[[
        "encounter_id", "admit_dt", "discharge_dt"
    ]],
    on="encounter_id",
    how="inner",
    validate="many_to_one"
)

# Keep observations during hospitalization
vitals = vitals[
    (vitals["charttime"] >= vitals["admit_dt"]) &
    (vitals["charttime"] <= vitals["discharge_dt"])
].copy()

# Convert Fahrenheit temp to Celsius
temp_mask = (
    (vitals["vital_type"] == "TEMP") &
    (vitals["value"] > 45) &
    (vitals["value"] <= 115)
)

vitals.loc[temp_mask, "value"] = (
    (vitals.loc[temp_mask, "value"] - 32) * 5 / 9
)

# Reject clearly impossible meas
valid_ranges = {
    "HR": (20, 250),
    "SBP": (40, 300),
    "DBP": (20, 200),
    "RR": (4, 70),
    "SPO2": (40, 100),
    "TEMP": (30, 43)
}

for vital_name, (minimum, maximum) in valid_ranges.items():
    invalid = (
        (vitals["vital_type"] == vital_name) &
        (~vitals["value"].between(minimum, maximum))
    )
    vitals.loc[invalid, "value"] = np.nan

# Keep the six relevant vital types
vitals = vitals[
    vitals["vital_type"].isin(valid_ranges)
].copy()

# Sort measurements in chronological order
vitals = vitals.sort_values(
    ["encounter_id", "vital_type", "charttime"]
)

# Calculate summary statistics
vital_stats = (
    vitals
    .groupby(["encounter_id", "vital_type"])["value"]
    .agg(["count", "mean", "min", "max",
          "std", "first", "last"])
    .unstack("vital_type")
)

vital_stats.columns = [
    f"vital_{vital.lower()}_{stat}"
    for stat, vital in vital_stats.columns
]

# Calculate changes between first and last readings
for vital_name in valid_ranges:
    name = vital_name.lower()

    first_col = f"vital_{name}_first"
    last_col = f"vital_{name}_last"

    if first_col in vital_stats and last_col in vital_stats:
        vital_stats[f"vital_{name}_change"] = (
            vital_stats[last_col] -
            vital_stats[first_col]
        )

# Additional feature: final pulse pressure
if (
    "vital_sbp_last" in vital_stats and
    "vital_dbp_last" in vital_stats
):
    vital_stats["pulse_pressure_last"] = (
        vital_stats["vital_sbp_last"] -
        vital_stats["vital_dbp_last"]
    )

vital_features = vital_stats.copy()

print("\nVITAL FEATURES CREATED")
print("Feature count:", vital_features.shape[1])
print("Encounters covered:", len(vital_features))


### 4.3 Assemble V3 feature table

Merge V2, laboratory and vital summaries on encounter ID; remove all-empty columns and verify the train/test column schemas.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 11.

In [ ]:

# V3  COMBINE ALL FEATURES

# Start with V2
v3_features = v2_features.copy()

# Attach laboratory features by encounter ID
v3_features = v3_features.join(
    lab_features, how="left", validate="one_to_one"
)

# Attach vital-sign features by encounter ID
v3_features = v3_features.join(
    vital_features, how="left", validate="one_to_one"
)

# Keep only columns that contain usable training data
train_ids = train["encounter_id"]
test_ids = test["encounter_id"]

X_train_v3 = v3_features.reindex(
    train_ids
).reset_index(drop=True)

X_test_v3 = v3_features.reindex(
    test_ids
).reset_index(drop=True)

# Drop features entirely missing in training
empty_cols = X_train_v3.columns[
    X_train_v3.isna().all()
].tolist()

X_train_v3 = X_train_v3.drop(columns=empty_cols)
X_test_v3 = X_test_v3.drop(columns=empty_cols)

cat_cols_v3 = [
    c for c in cat_cols_v2
    if c in X_train_v3.columns
]

# Ensure row and column consistency
assert len(X_train_v3) == len(train)
assert len(X_test_v3) == len(test)
assert X_train_v3.columns.equals(X_test_v3.columns)
assert X_train_v3.columns.is_unique

print("V3 DATASET READY")
print("V2 features:", X_train_v2.shape[1])
print("V3 features:", X_train_v3.shape[1])
print("New features:",
      X_train_v3.shape[1] - X_train_v2.shape[1])
print("Training rows:", len(X_train_v3))
print("Test rows:", len(X_test_v3))
print("All-missing columns removed:", len(empty_cols))


## 5. V4: More utilization history and consistent model inputs

**Why:** Recent utilization is often more informative than lifetime utilization. V4 adds windows of 30/90/365 days and type-specific ED/IP/OP counts. It also performs a raw laboratory-unit audit.

**Important limitation:** the original V4 raw-unit summary was calculated from `selected_raw` without its own explicit admission/discharge time filter. The later V8 **new model matrix** drops these three V4 lab-unit-count features (`labs_missing_units`, `labs_total_selected`, `labs_missing_units_fraction`). **However, V8 blends in 30% of the older V5 predictions, and V5 was trained with the original three unfiltered features. Consequently, the selected final ensembles still inherit some V5 predictions. A fully leakage-free final model is NOT established by these original notebooks.** This is a material limitation, not merely a theoretical one.

### 5.1 Raw laboratory-unit audit

Audit unit strings and missing laboratory values. This is a diagnostic step, not an outcome-label feature.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 17.

In [ ]:

# V4 - LAB AUDIT BEFORE FILTer

raw_labs = pd.read_csv(files["lab_results.csv"])

raw_labs["lab_key"] = (
    raw_labs["lab_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

raw_labs["unit_clean"] = (
    raw_labs["unit"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace("µ", "u", regex=False)
    .str.replace("μ", "u", regex=False)
)

selected_raw = raw_labs[
    raw_labs["lab_key"].isin(selected_labs)
].copy()

print("Original selected lab rows:", len(selected_raw))

print("\nUnit distribution before filtering:")
print(
    selected_raw.groupby(
        ["lab_key", "unit_clean"],
        dropna=False
    ).size().to_string()
)

print("\nMissing lab values:")
print(selected_raw["value"].isna().sum())


### 5.2 Construct prior-utilization windows

Use historical records only and calculate visit counts and time since the most recent previous visit.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 18.

In [ ]:

# patient's previous hospital visits

history = pd.read_csv(files["prior_utilization.csv"])

# dates are actual dates, not text
history["visit_dt"] = pd.to_datetime(
    history["visit_dt"],
    errors="coerce",
    format="mixed"
)

#  admission date 
admissions = master[
    ["encounter_id", "patient_id", "admit_dt"]
].copy()

admissions["admit_dt"] = pd.to_datetime(
    admissions["admit_dt"],
    errors="coerce",
    format="mixed"
)

history = history.merge(
    admissions,
    on="patient_id",
    how="inner",
    validate="many_to_one"
)

history["days_before"] = (
    history["admit_dt"] - history["visit_dt"]
).dt.total_seconds() / 86400

history = history[
    history["days_before"].ge(0)
].copy()

# Clean up the visit names
history["visit_type"] = (
    history["visit_type"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Make a separate table for new features
visit_features = pd.DataFrame(
    index=admissions["encounter_id"]
)

visit_features.index.name = "encounter_id"

# Count visits in different time windows
for days in [30, 90, 365]:
    recent = history[
        history["days_before"] <= days
    ]

    counts = recent.groupby("encounter_id").size()

    visit_features[f"visits_{days}d"] = counts

# Count emergency, inpatient and outpatient visits
for visit_type in ["ED", "IP", "OP"]:
    selected = history[
        history["visit_type"] == visit_type
    ]

    counts = selected.groupby("encounter_id").size()

    visit_features[f"visits_{visit_type.lower()}"] = counts

# time since most recent visit 
visit_features["days_since_last_visit"] = (
    history.groupby("encounter_id")["days_before"].min()
)

#  different hospitals per patient visitation
visit_features["different_hospitals_visited"] = (
    history.groupby("encounter_id")["hospital_id"].nunique()
)


count_columns = [
    c for c in visit_features.columns
    if c != "days_since_last_visit"
]

visit_features[count_columns] = (
    visit_features[count_columns].fillna(0)
)

print("PREVIOUS VISIT FEATURES READY")
print("New features:", visit_features.shape[1])
print("Patients with recorded previous visits:",
      visit_features["days_since_last_visit"].notna().sum())


### 5.3 Build V4 178-feature training and test matrices

Join by encounter ID, reject duplicate columns, prepare categorical features, and assert equal train/test feature sets.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 19.

In [ ]:

import pandas as pd

# V4 - rebuild the dataset

needed = [
    "v2_features", "lab_features", "vital_features",
    "visit_features", "selected_raw", "train", "test"
]

missing = [name for name in needed if name not in globals()]

if missing:
    raise RuntimeError("Missing source tables: " + str(missing))


# make sure all tables use encounter_id as the index
def fix_index(df):
    df = df.copy()

    if "encounter_id" in df.columns:
        df = df.set_index("encounter_id")

    df.index = df.index.astype(str)
    df.index.name = "encounter_id"

    if not df.index.is_unique:
        raise ValueError("Duplicate encounter ids found")

    if not df.columns.is_unique:
        raise ValueError("Duplicate columns found")

    return df


# recreate the missing lab unit features
if "unit_clean" not in selected_raw.columns:
    raise ValueError("unit_clean is missing from selected_raw")

if "encounter_id" in selected_raw.columns:
    lab_units = selected_raw[
        ["encounter_id", "unit_clean"]
    ].copy()

elif selected_raw.index.name == "encounter_id":
    lab_units = selected_raw[
        ["unit_clean"]
    ].reset_index()

else:
    raise ValueError("encounter_id is missing from selected_raw")


lab_units["unit_missing"] = (
    lab_units["unit_clean"].isna().astype(int)
)

missing_lab_features = lab_units.groupby(
    "encounter_id"
).agg(
    labs_missing_units=("unit_missing", "sum"),
    labs_total_selected=("unit_missing", "size")
)

missing_lab_features["labs_missing_units_fraction"] = (
    missing_lab_features["labs_missing_units"] /
    missing_lab_features["labs_total_selected"]
)


# rebuild all the v4 features
v4_features = fix_index(v2_features)

for table in [
    lab_features,
    vital_features,
    visit_features,
    missing_lab_features
]:
    extra = fix_index(table)

    # only add columns we don't already have
    new_cols = extra.columns.difference(
        v4_features.columns, sort=False
    )

    v4_features = v4_features.join(
        extra[new_cols],
        how="left",
        validate="one_to_one"
    )


# make sure the target isn't part of the features
v4_features = v4_features.drop(
    columns=["readmit_30d"],
    errors="ignore"
)

if v4_features.shape[1] != 178:
    raise ValueError(
        f"Expected 178 features, got {v4_features.shape[1]}"
    )


# separate the training and test rows
train_ids = train["encounter_id"].astype(str)
test_ids = test["encounter_id"].astype(str)

all_ids = pd.Index(
    pd.concat([train_ids, test_ids], ignore_index=True)
)

missing_ids = all_ids.difference(v4_features.index)

if len(missing_ids) > 0:
    raise ValueError(
        f"{len(missing_ids)} encounters are missing"
    )

X_train_v4 = v4_features.loc[
    train_ids
].reset_index(drop=True)

X_test_v4 = v4_features.loc[
    test_ids
].reset_index(drop=True)


# categorical columns from v4
known_cats = [
    "hospital_id", "admission_type", "admission_source",
    "service_line", "discharge_disposition",
    "attending_specialty", "sex", "race_ethnicity",
    "language", "marital_status", "insurance_type",
    "urban_rural", "ownership", "home_health_referral",
    "education_done", "primary_icd_code",
    "primary_icd_chapter"
]

cat_cols_v4 = [
    col for col in known_cats
    if col in X_train_v4.columns
]

for col in X_train_v4.columns:
    if (
        col not in cat_cols_v4
        and not pd.api.types.is_numeric_dtype(X_train_v4[col])
    ):
        cat_cols_v4.append(col)


# prepare the categorical values for training
X_train_v4_ready = X_train_v4.copy()
X_test_v4_ready = X_test_v4.copy()

for col in cat_cols_v4:
    for df in [X_train_v4_ready, X_test_v4_ready]:
        df[col] = (
            df[col]
            .astype(object)
            .where(df[col].notna(), "Missing")
            .astype(str)
        )


# check the final dataset
print("V4 DATASET READY")
print("Training shape:", X_train_v4_ready.shape)
print("Test shape:", X_test_v4_ready.shape)
print("Categorical columns:", len(cat_cols_v4))
print("Duplicate columns:", v4_features.columns.duplicated().sum())
print("Missing lab features:", list(missing_lab_features.columns))


## 6. V5: Three-fold model comparison (CatBoost and XGBoost)

**Why:** Compare boosted decision-tree models under the **same stratified three-fold splits**. CatBoost natively handles categories; the XGBoost branch uses a fold-fitted median imputer and ordinal encoder. Out-of-fold (OOF) predictions give one held-out prediction per training patient. Weighted ensembles are compared by ROC-AUC.

**Course connection:** decision trees, boosting, cross-validation, ensemble learning and overfitting control via early stopping. The final model uses **probabilities**, never 0/1 labels.

### 6.1 Train V5 models and compute OOF ensembles

Fits two CatBoost configurations and one XGBoost configuration; saves model-level and blend predictions.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 23.

In [ ]:
# V5 - compare models before using more submissions

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# use the V4 dataset already made
X = X_train_v4_ready.reset_index(drop=True).copy()
T = X_test_v4_ready.reset_index(drop=True).copy()
y = train["readmit_30d"].to_numpy(dtype=int)
cat_columns = list(cat_cols_v4)
num_columns = [col for col in X.columns if col not in cat_columns]

assert X.shape == (76975, 178), X.shape
assert T.shape == (32812, 178), T.shape
assert X.columns.equals(T.columns)
assert len(y) == len(X)
assert not X.columns.duplicated().any()
assert set(np.unique(y)) == {0, 1}

# keep the same folds for every model
folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
model_names = ["catboost_6", "catboost_5", "xgboost"]
oof = {name: np.zeros(len(X)) for name in model_names}
test_preds = {name: np.zeros(len(T)) for name in model_names}
fold_scores = {name: [] for name in model_names}

for fold, (tr, va) in enumerate(folds.split(X, y), start=1):
    print("\nFOLD", fold, "OF 3")

    for name, depth, rate, reg in [
        ("catboost_6", 6, 0.035, 5),
        ("catboost_5", 5, 0.05, 7)
    ]:
        model = CatBoostClassifier(
            iterations=900,
            depth=depth,
            learning_rate=rate,
            l2_leaf_reg=reg,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42 + fold,
            thread_count=-1,
            verbose=False,
            allow_writing_files=False
        )
        model.fit(
            X.iloc[tr], y[tr],
            cat_features=cat_columns,
            eval_set=(X.iloc[va], y[va]),
            early_stopping_rounds=90,
            verbose=False
        )
        oof[name][va] = model.predict_proba(X.iloc[va])[:, 1]
        test_preds[name] += model.predict_proba(T)[:, 1] / 3
        score = roc_auc_score(y[va], oof[name][va])
        fold_scores[name].append(score)
        print(name, "AUC:", round(score, 5), "rounds:", model.best_iteration_ + 1)

    # fit the XGBoost encoder only on this fold's training rows
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median", keep_empty_features=True), num_columns),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_columns)
    ])
    x_tr = prep.fit_transform(X.iloc[tr])
    x_va = prep.transform(X.iloc[va])
    x_te = prep.transform(T)

    xgb = XGBClassifier(
        n_estimators=1100,
        max_depth=4,
        learning_rate=0.035,
        min_child_weight=8,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=8,
        objective="binary:logistic",
        tree_method="hist",
        eval_metric="auc",
        early_stopping_rounds=90,
        random_state=42 + fold,
        n_jobs=-1
    )
    xgb.fit(x_tr, y[tr], eval_set=[(x_va, y[va])], verbose=False)
    oof["xgboost"][va] = xgb.predict_proba(x_va)[:, 1]
    test_preds["xgboost"] += xgb.predict_proba(x_te)[:, 1] / 3
    score = roc_auc_score(y[va], oof["xgboost"][va])
    fold_scores["xgboost"].append(score)
    print("xgboost AUC:", round(score, 5), "rounds:", xgb.best_iteration + 1)

# blend the predictions and check whether they improve the AUC
blends = {
    "blend_cb6_xgb": {"catboost_6": 0.7, "xgboost": 0.3},
    "blend_cb6_cb5": {"catboost_6": 0.5, "catboost_5": 0.5},
    "blend_all": {"catboost_6": 0.5, "catboost_5": 0.25, "xgboost": 0.25}
}
for name, weights in blends.items():
    oof[name] = sum(weight * oof[part] for part, weight in weights.items())
    test_preds[name] = sum(weight * test_preds[part] for part, weight in weights.items())

scores = pd.DataFrame([
    {"model": name, "oof_auc": roc_auc_score(y, preds)}
    for name, preds in oof.items()
]).sort_values("oof_auc", ascending=False).reset_index(drop=True)

print("\nMODEL COMPARISON - SAME 3 FOLDS")
print(scores.to_string(index=False))
print("\nV4 single-split AUC was 0.74499 (not directly comparable).")

# save the files now, but don't submit them yet
for name, preds in test_preds.items():
    submission = pd.DataFrame({
        "encounter_id": test["encounter_id"].to_numpy(),
        "readmit_30d": preds
    })
    assert len(submission) == len(test)
    assert submission["encounter_id"].is_unique
    assert np.isfinite(preds).all()
    assert submission["readmit_30d"].between(0, 1).all()
    submission.to_csv(f"/kaggle/working/{name}.csv", index=False)

print("\nAll 6 candidate CSV files saved in /kaggle/working/")


## 7. V8: Longitudinal, diagnosis and medication enhancements

**Why:** Extend V4 with short- and long-term utilization, diagnosis-code patterns, medication classes and the last 24 hours of vitals/labs. V8 removes the three potentially time-contaminated V4 lab-unit features **from its new CatBoost/LightGBM feature matrix, but not from the historical V5 component that it blends in**. It trains CatBoost and LightGBM on three stratified folds.

**Limitation to disclose:** the original V8 feature builder selected common diagnosis and medication categories from the combined unlabeled train/test population. This does not use the target labels, but it is **transductive vocabulary selection**, not strictly train-only preprocessing. For a prospective clinical deployment, vocabulary should be fitted only on training data inside each validation fold.

### 7.1 Build V8 and evaluate CatBoost/LightGBM

Retains the original model settings and blend calculations. The original result was 0.748035 OOF AUC for the selected V8 blend.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 35.

In [ ]:

import gc
import re
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# use the existing notebook data
X0 = X_train_v4_ready.reset_index(drop=True).copy()
T0 = X_test_v4_ready.reset_index(drop=True).copy()
y = train["readmit_30d"].to_numpy(dtype=int)

assert X0.columns.equals(T0.columns)
assert len(X0) == len(y)
assert len(T0) == len(test)

ids = pd.Index(
    pd.concat([
        train["encounter_id"],
        test["encounter_id"]
    ], ignore_index=True).astype(str)
)

A = pd.concat([X0, T0], ignore_index=True)
print("V8 STARTED:", A.shape, flush=True)

# find the Kaggle CSV files automatically
def read_csv(name, cols=None):
    paths = list(Path("/kaggle/input").rglob(name))
    if not paths:
        raise FileNotFoundError("Missing Kaggle input file: " + name)
    return pd.read_csv(paths[0], usecols=cols)

# add new features in the correct row order
def add(table):
    global A
    table = table.copy()
    table.index = table.index.astype(str)
    table = table[~table.index.duplicated()]
    table = table.loc[:, ~table.columns.isin(A.columns)]
    A = pd.concat([
        A,
        table.reindex(ids).reset_index(drop=True)
    ], axis=1)

# admission and discharge dates
enc = master[
    ["encounter_id", "patient_id", "admit_dt", "discharge_dt"]
].drop_duplicates("encounter_id").copy()

enc["encounter_id"] = enc["encounter_id"].astype(str)
enc["patient_id"] = enc["patient_id"].astype(str)

for c in ["admit_dt", "discharge_dt"]:
    enc[c] = pd.to_datetime(
        enc[c], errors="coerce", format="mixed"
    )

dates = enc.set_index("encounter_id").reindex(ids)

for name in ["admit_dt", "discharge_dt"]:
    dt = dates[name]
    A[name + "_hour_v8"] = dt.dt.hour.to_numpy()
    A[name + "_dow_v8"] = dt.dt.dayofweek.to_numpy()
    A[name + "_month_v8"] = dt.dt.month.to_numpy()

# patient history before admission
h = read_csv(
    "prior_utilization.csv",
    ["patient_id", "visit_dt", "visit_type"]
)

h["patient_id"] = h["patient_id"].astype(str)

h = h.merge(
    enc[["patient_id", "encounter_id", "admit_dt"]],
    on="patient_id",
    how="inner"
)

h["visit_dt"] = pd.to_datetime(
    h["visit_dt"], errors="coerce", format="mixed"
)

h["days_ago"] = (
    h["admit_dt"] - h["visit_dt"]
).dt.total_seconds() / 86400

h = h[h["days_ago"].ge(0)].copy()

h["visit_type"] = (
    h["visit_type"].fillna("").astype(str)
    .str.upper().str.strip()
)

h["visit_type"] = h["visit_type"].replace({
    "ER": "ED",
    "EMERGENCY": "ED",
    "INPATIENT": "IP"
})

hf = h.groupby("encounter_id")["days_ago"].agg(
    ["min", "mean", "std", "count"]
)

hf.columns = ["v8_history_" + c for c in hf.columns]

for days in [7, 30, 90, 180, 365, 730]:
    mask = h["days_ago"].le(days)

    hf[f"v8_visits_{days}d"] = (
        h[mask].groupby("encounter_id").size()
    )

    for typ in ["ED", "IP"]:
        hf[f"v8_{typ}_{days}d"] = (
            h[mask & h["visit_type"].eq(typ)]
            .groupby("encounter_id").size()
        )

add(hf)
del h, hf

print("HISTORY READY", flush=True)

# common diagnosis codes and groups
d = read_csv(
    "diagnoses.csv",
    ["encounter_id", "icd10_code"]
)

d["encounter_id"] = d["encounter_id"].astype(str)

d["icd10_code"] = (
    d["icd10_code"].fillna("UNK")
    .astype(str).str.upper().str.strip()
)

d["group"] = (
    d["icd10_code"]
    .str.replace(".", "", regex=False).str[:3]
)

for key, prefix, n in [
    ("icd10_code", "v8_dx", 45),
    ("group", "v8_group", 55)
]:
    top = d[key].value_counts().head(n).index
    sub = d[d[key].isin(top)]

    t = pd.crosstab(
        sub["encounter_id"], sub[key]
    )

    t.columns = [
        prefix + "_" + str(i)
        for i in range(len(t.columns))
    ]

    add(t)

del d, t

# medication classes
med = read_csv(
    "medications.csv",
    ["encounter_id", "drug_class", "continued_at_discharge"]
)

med["encounter_id"] = med["encounter_id"].astype(str)

med["drug_class"] = (
    med["drug_class"].fillna("UNK")
    .astype(str).str.upper().str.strip()
)

top = med["drug_class"].value_counts().head(35).index

continued = (
    med["continued_at_discharge"]
    .astype(str).str.lower()
    .isin(["true", "1", "1.0", "yes"])
)

for name, sub in [
    ("v8_med", med[med["drug_class"].isin(top)]),
    (
        "v8_med_discharge",
        med[med["drug_class"].isin(top) & continued]
    )
]:
    t = pd.crosstab(
        sub["encounter_id"], sub["drug_class"]
    )

    t.columns = [
        name + "_" + str(i)
        for i in range(len(t.columns))
    ]

    add(t)

del med, sub, t

print("DIAGNOSES AND MEDICATIONS READY", flush=True)

# last 24 hours of vital signs
v = read_csv(
    "vitals.csv",
    ["encounter_id", "charttime", "vital_type", "value"]
)

v["encounter_id"] = v["encounter_id"].astype(str)

v = v.merge(
    enc[["encounter_id", "admit_dt", "discharge_dt"]],
    on="encounter_id",
    how="inner"
)

v["charttime"] = pd.to_datetime(
    v["charttime"], errors="coerce", format="mixed"
)

v["hours_before"] = (
    v["discharge_dt"] - v["charttime"]
).dt.total_seconds() / 3600

v = v[
    v["hours_before"].between(0, 24)
    & (v["charttime"] >= v["admit_dt"])
].copy()

v["vital_type"] = (
    v["vital_type"].astype(str).str.upper().str.strip()
)

v["value"] = pd.to_numeric(
    v["value"], errors="coerce"
)

# convert Fahrenheit temperatures
temp = (
    v["vital_type"].eq("TEMP")
    & v["value"].between(45, 115)
)

v.loc[temp, "value"] = (
    v.loc[temp, "value"] - 32
) * 5 / 9

ranges = {
    "HR": (20, 250),
    "SBP": (40, 300),
    "DBP": (20, 200),
    "RR": (4, 70),
    "SPO2": (40, 100),
    "TEMP": (30, 43)
}

v = v[v["vital_type"].isin(ranges)].copy()

for typ, (lo, hi) in ranges.items():
    bad = (
        v["vital_type"].eq(typ)
        & ~v["value"].between(lo, hi)
    )
    v.loc[bad, "value"] = np.nan

v = v.sort_values("charttime")

vt = v.groupby(
    ["encounter_id", "vital_type"]
)["value"].agg(
    ["mean", "std", "min", "max", "last"]
).unstack()

vt.columns = [
    "v8_vital_" + str(typ) + "_" + str(stat)
    for stat, typ in vt.columns
]

add(vt)
del v, vt

print("LAST-DAY VITALS READY", flush=True)

# laboratory abnormalities in the final 24 hours
lab = read_csv(
    "lab_results.csv",
    [
        "encounter_id",
        "collected_dt",
        "lab_name",
        "abnormal_flag"
    ]
)

lab["encounter_id"] = lab["encounter_id"].astype(str)

lab = lab.merge(
    enc[["encounter_id", "admit_dt", "discharge_dt"]],
    on="encounter_id",
    how="inner"
)

lab["collected_dt"] = pd.to_datetime(
    lab["collected_dt"], errors="coerce", format="mixed"
)

lab["hours_before"] = (
    lab["discharge_dt"] - lab["collected_dt"]
).dt.total_seconds() / 3600

lab = lab[
    lab["hours_before"].between(0, 24)
    & (lab["collected_dt"] >= lab["admit_dt"])
].copy()

lab["flag"] = (
    lab["abnormal_flag"].fillna("")
    .astype(str).str.upper().str.strip()
)

lab["abnormal"] = lab["flag"].isin(["H", "L"]).astype(int)

lf = lab.groupby("encounter_id").agg(
    v8_lab_count=("lab_name", "size"),
    v8_lab_unique=("lab_name", "nunique"),
    v8_lab_abnormal=("abnormal", "sum")
)

lf["v8_lab_abnormal_fraction"] = (
    lf["v8_lab_abnormal"]
    / lf["v8_lab_count"].replace(0, np.nan)
)

add(lf)
del lab, lf

# prepare the model data
A = A.drop(
    columns=[
        "labs_missing_units",
        "labs_total_selected",
        "labs_missing_units_fraction"
    ],
    errors="ignore"
)

A = A.replace([np.inf, -np.inf], np.nan)
A["v8_missing_count"] = A.isna().sum(axis=1)

A.columns = [
    re.sub("[^a-zA-Z0-9_]", "_", str(c))
    for c in A.columns
]

assert A.columns.is_unique, "Duplicate columns found"

cats = [
    c for c in A
    if not pd.api.types.is_numeric_dtype(A[c])
]

for c in cats:
    A[c] = A[c].fillna("Missing").astype(str)

CX = A.iloc[:len(X0)].copy()
CT = A.iloc[len(X0):].copy()

L = A.copy()

for c in cats:
    L[c] = L[c].astype("category")

LX = L.iloc[:len(X0)].copy()
LT = L.iloc[len(X0):].copy()

del A, L

print("TRAINING:", CX.shape, flush=True)

# use the same validation folds as V5
folds = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

co = np.zeros(len(y))
lo = np.zeros(len(y))
ct = np.zeros(len(T0))
lt = np.zeros(len(T0))

for fold, (tr, va) in enumerate(folds.split(CX, y), 1):

    print("FOLD", fold, "CATBOOST", flush=True)

    cb = CatBoostClassifier(
        iterations=1100,
        depth=6,
        learning_rate=0.035,
        l2_leaf_reg=7,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=100 + fold,
        verbose=False,
        allow_writing_files=False
    )

    cb.fit(
        CX.iloc[tr],
        y[tr],
        cat_features=cats,
        eval_set=(CX.iloc[va], y[va]),
        early_stopping_rounds=100
    )

    co[va] = cb.predict_proba(CX.iloc[va])[:, 1]
    ct += cb.predict_proba(CT)[:, 1] / 3

    del cb

    print("FOLD", fold, "LIGHTGBM", flush=True)

    lg = LGBMClassifier(
        n_estimators=1400,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=85,
        colsample_bytree=0.82,
        reg_lambda=8,
        random_state=200 + fold,
        verbosity=-1
    )

    lg.fit(
        LX.iloc[tr],
        y[tr],
        eval_set=[(LX.iloc[va], y[va])],
        eval_metric="auc",
        callbacks=[
            early_stopping(90, verbose=False)
        ]
    )

    lo[va] = lg.predict_proba(LX.iloc[va])[:, 1]
    lt += lg.predict_proba(LT)[:, 1] / 3

    del lg

    print(
        "FOLD AUC:",
        round(roc_auc_score(y[va], co[va]), 5),
        round(roc_auc_score(y[va], lo[va]), 5),
        flush=True
    )

    gc.collect()

# compare against the previous best model
old_oof = np.asarray(oof["blend_cb6_xgb"])
old_test = np.asarray(test_preds["blend_cb6_xgb"])

new_oof = (co + lo) / 2
new_test = (ct + lt) / 2

choices = {
    "v8_catboost": (co, ct),
    "v8_lightgbm": (lo, lt),
    "v8_ensemble": (new_oof, new_test),
    "v8_old30_new70": (
        0.3 * old_oof + 0.7 * new_oof,
        0.3 * old_test + 0.7 * new_test
    ),
    "v8_old50_new50": (
        0.5 * old_oof + 0.5 * new_oof,
        0.5 * old_test + 0.5 * new_test
    ),
    "v8_old70_new30": (
        0.7 * old_oof + 0.3 * new_oof,
        0.7 * old_test + 0.3 * new_test
    ),
    "existing_v5": (old_oof, old_test)
}

scores = sorted(
    [
        (roc_auc_score(y, val[0]), key)
        for key, val in choices.items()
    ],
    reverse=True
)

print("\nMODEL COMPARISON")

for score, name in scores:
    print(name, round(score, 6))

score, best = scores[0]

if best == "existing_v5":
    print(
        "NO OFFLINE IMPROVEMENT. "
        "Keep your existing V5 submission."
    )

else:
    pred = choices[best][1]

    out = pd.DataFrame({
        "encounter_id": test["encounter_id"].values,
        "readmit_30d": pred
    })

    assert len(out) == len(test)
    assert out["encounter_id"].is_unique
    assert out["readmit_30d"].between(0, 1).all()
    assert out["readmit_30d"].notna().all()

    out.to_csv(
        "/kaggle/working/SUBMIT_THIS.csv",
        index=False
    )

    print(
        "SAVED: /kaggle/working/SUBMIT_THIS.csv",
        best
    )


## 8. V9: Boosting, bagging and course-algorithm checks

**Why:** Add clinical interactions and compare XGBoost against Random Forest (bagging). The original notebook also benchmarks entropy Decision Trees, PCA + Perceptron, and Truncated SVD + kNN on a holdout. These course models are important comparisons even when they do not win.

**Limitation:** V9's one-hot vocabulary is created before the three-fold split, using both unlabeled training and test features. Because no target labels are used, this is not target leakage, but it does not meet the strictest fold-only encoder-fitting standard. Report it honestly as an experimental limitation.

### 8.1 Train V9, compare classroom models and bootstrap improvement

Use the same folds as V8, compare OOF ROC-AUC and test whether an ensemble gain is consistent across folds.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 37.

In [ ]:

# V9 - run this entire cell after V8 has completed
import base64
import gc
import numpy as np
import pandas as pd

from IPython.display import HTML, display
from scipy.sparse import csr_matrix

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import Perceptron
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD

from xgboost import XGBClassifier

# check the previous V8 results
required = [
    "CX", "CT", "co", "lo", "ct", "lt",
    "train", "test", "oof", "test_preds"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing V8 variables: " + ", ".join(missing)
        + ". Run the complete V8 cell first."
    )

n = len(train)
y9 = train["readmit_30d"].to_numpy(dtype=int)

assert len(CX) == n
assert len(CT) == len(test)
assert CX.columns.equals(CT.columns)

# recreate the winning V8 blend
v8_oof = (
    0.30 * np.asarray(oof["blend_cb6_xgb"])
    + 0.35 * np.asarray(co)
    + 0.35 * np.asarray(lo)
)

v8_test = (
    0.30 * np.asarray(test_preds["blend_cb6_xgb"])
    + 0.35 * np.asarray(ct)
    + 0.35 * np.asarray(lt)
)

assert len(v8_oof) == n
assert len(v8_test) == len(test)

base_auc = roc_auc_score(y9, v8_oof)

print(
    "V9 STARTED | V8 OOF AUC:",
    round(base_auc, 6),
    flush=True
)

# add clinical interaction features
B = pd.concat([CX, CT], ignore_index=True).copy()

def num(name):
    if name in B:
        return pd.to_numeric(B[name], errors="coerce")
    return None

def ratio(new, a, b, offset=1.0):
    aa = num(a)
    bb = num(b)

    if aa is not None and bb is not None:
        B[new] = aa / (bb + offset)

ratio(
    "v9_shock_index",
    "vital_hr_last",
    "vital_sbp_last",
    0.01
)

ratio(
    "v9_bun_creatinine",
    "lab_bun_last",
    "lab_creatinine_last",
    0.1
)

ratio(
    "v9_recent_visit_share",
    "v8_visits_30d",
    "v8_visits_365d"
)

ratio(
    "v9_recent_ip_share",
    "v8_IP_90d",
    "v8_visits_90d"
)

ratio(
    "v9_diagnoses_per_day",
    "diagnosis_count",
    "stay_days"
)

ratio(
    "v9_abnormal_labs_per_day",
    "v8_lab_abnormal",
    "stay_days"
)

# changes in clinical measurements
changes = [
    (
        "v9_heart_rate_change",
        "v8_vital_HR_last",
        "vital_hr_first"
    ),
    (
        "v9_systolic_change",
        "v8_vital_SBP_last",
        "vital_sbp_first"
    ),
    (
        "v9_oxygen_change",
        "v8_vital_SPO2_last",
        "vital_spo2_first"
    ),
    (
        "v9_bun_change",
        "lab_bun_last",
        "lab_bun_first"
    ),
    (
        "v9_creatinine_change",
        "lab_creatinine_last",
        "lab_creatinine_first"
    )
]

for name, a, b in changes:
    aa = num(a)
    bb = num(b)

    if aa is not None and bb is not None:
        B[name] = aa - bb

# prepare categorical data for tree models
categorical = B.select_dtypes(
    exclude=[np.number]
).columns.tolist()

for c in categorical:
    B[c] = B[c].fillna("Missing").astype(str)

    if B[c].nunique() > 150:
        B[c] = B[c].str[:3]

B = pd.get_dummies(
    B,
    columns=categorical,
    dtype=np.uint8
)

B = B.replace([np.inf, -np.inf], np.nan)
B = B.loc[:, ~B.columns.duplicated()]
B = B.astype(np.float32)

X9 = B.iloc[:n].reset_index(drop=True)
T9 = B.iloc[n:].reset_index(drop=True)

del B

print("V9 FEATURES:", X9.shape, flush=True)

# fill missing values for Random Forest
medians = X9.median(axis=0).fillna(0)

XR = X9.fillna(medians).fillna(0)
TR = T9.fillna(medians).fillna(0)

# same folds as V8
folds = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

xgb_oof = np.zeros(n)
rf_oof = np.zeros(n)

xgb_test = np.zeros(len(test))
rf_test = np.zeros(len(test))

# train boosting and bagging models
for k, (tr, va) in enumerate(folds.split(X9, y9), 1):

    print(
        "FOLD", k, "/ 3: XGBOOST (boosting)",
        flush=True
    )

    xgb = XGBClassifier(
        n_estimators=1500,
        max_depth=4,
        learning_rate=0.025,
        min_child_weight=15,
        subsample=0.85,
        colsample_bytree=0.82,
        reg_lambda=12,
        reg_alpha=0.5,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=100 + k,
        n_jobs=4,
        early_stopping_rounds=110
    )

    xgb.fit(
        X9.iloc[tr],
        y9[tr],
        eval_set=[(X9.iloc[va], y9[va])],
        verbose=False
    )

    xgb_oof[va] = xgb.predict_proba(
        X9.iloc[va]
    )[:, 1]

    xgb_test += xgb.predict_proba(
        T9
    )[:, 1] / 3

    print(
        "XGB AUC:",
        round(roc_auc_score(y9[va], xgb_oof[va]), 6),
        flush=True
    )

    del xgb
    gc.collect()

    print(
        "FOLD", k, "/ 3: RANDOM FOREST (bagging)",
        flush=True
    )

    rf = RandomForestClassifier(
        n_estimators=220,
        max_depth=17,
        min_samples_leaf=18,
        max_features=0.55,
        max_samples=0.82,
        bootstrap=True,
        random_state=300 + k,
        n_jobs=4
    )

    rf.fit(XR.iloc[tr], y9[tr])

    rf_oof[va] = rf.predict_proba(
        XR.iloc[va]
    )[:, 1]

    rf_test += rf.predict_proba(
        TR
    )[:, 1] / 3

    print(
        "RF AUC:",
        round(roc_auc_score(y9[va], rf_oof[va]), 6),
        flush=True
    )

    del rf
    gc.collect()

# benchmark classroom algorithms on one holdout
print(
    "\nCLASS ALGORITHM CHECK (one holdout only)",
    flush=True
)

tr, va = next(folds.split(X9, y9))

rng = np.random.default_rng(42)

tr_s = rng.choice(
    tr,
    size=min(7000, len(tr)),
    replace=False
)

va_s = rng.choice(
    va,
    size=min(4000, len(va)),
    replace=False
)

# Decision Tree with entropy
dt = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=6,
    min_samples_leaf=100,
    random_state=42
)

dt.fit(XR.iloc[tr_s], y9[tr_s])

print(
    "Decision Tree:",
    round(
        roc_auc_score(
            y9[va_s],
            dt.predict_proba(XR.iloc[va_s])[:, 1]
        ),
        5
    ),
    flush=True
)

del dt

# PCA and Perceptron
numeric_cols = CX.select_dtypes(
    include=[np.number]
).columns.tolist()

if len(numeric_cols) >= 3:

    Q = CX[numeric_cols].replace(
        [np.inf, -np.inf], np.nan
    )

    Q = Q.fillna(
        Q.iloc[tr_s].median()
    ).fillna(0)

    scale = StandardScaler()

    qtr = scale.fit_transform(
        Q.iloc[tr_s].to_numpy(dtype=np.float32)
    )

    qva = scale.transform(
        Q.iloc[va_s].to_numpy(dtype=np.float32)
    )

    pca = PCA(
        n_components=min(
            20, qtr.shape[1], qtr.shape[0] - 1
        ),
        random_state=42
    )

    ztr = pca.fit_transform(qtr)
    zva = pca.transform(qva)

    perceptron = Perceptron(
        alpha=0.0005,
        penalty="l2",
        max_iter=1500,
        random_state=42,
        tol=1e-3
    )

    perceptron.fit(ztr, y9[tr_s])

    print(
        "PCA + Perceptron:",
        round(
            roc_auc_score(
                y9[va_s],
                perceptron.decision_function(zva)
            ),
            5
        ),
        flush=True
    )

    del Q, qtr, qva, ztr, zva, pca, perceptron

# SVD and kNN
svd = TruncatedSVD(
    n_components=min(16, XR.shape[1] - 1),
    random_state=42
)

s_tr = svd.fit_transform(
    csr_matrix(XR.iloc[tr_s].to_numpy())
)

s_va = svd.transform(
    csr_matrix(XR.iloc[va_s].to_numpy())
)

s_scale = StandardScaler()

s_tr = s_scale.fit_transform(s_tr)
s_va = s_scale.transform(s_va)

knn = KNeighborsClassifier(
    n_neighbors=75,
    weights="distance",
    n_jobs=4
)

knn.fit(s_tr, y9[tr_s])

print(
    "SVD + kNN:",
    round(
        roc_auc_score(
            y9[va_s],
            knn.predict_proba(s_va)[:, 1]
        ),
        5
    ),
    flush=True
)

del svd, knn, s_tr, s_va, s_scale

# compare real three-fold OOF candidates
candidates = {
    "v8_original": (
        v8_oof,
        v8_test
    ),
    "v9_xgboost": (
        xgb_oof,
        xgb_test
    ),
    "v9_random_forest": (
        rf_oof,
        rf_test
    ),
    "v9_v8_80_xgb_20": (
        0.80 * v8_oof + 0.20 * xgb_oof,
        0.80 * v8_test + 0.20 * xgb_test
    ),
    "v9_v8_60_xgb_40": (
        0.60 * v8_oof + 0.40 * xgb_oof,
        0.60 * v8_test + 0.40 * xgb_test
    ),
    "v9_v8_40_xgb_60": (
        0.40 * v8_oof + 0.60 * xgb_oof,
        0.40 * v8_test + 0.60 * xgb_test
    ),
    "v9_v8_90_rf_10": (
        0.90 * v8_oof + 0.10 * rf_oof,
        0.90 * v8_test + 0.10 * rf_test
    ),
    "v9_v8_70_rf_30": (
        0.70 * v8_oof + 0.30 * rf_oof,
        0.70 * v8_test + 0.30 * rf_test
    ),
    "v9_v8_60_xgb_30_rf_10": (
        0.60 * v8_oof
        + 0.30 * xgb_oof
        + 0.10 * rf_oof,

        0.60 * v8_test
        + 0.30 * xgb_test
        + 0.10 * rf_test
    )
}

ranked = sorted(
    [
        (roc_auc_score(y9, p[0]), name)
        for name, p in candidates.items()
    ],
    reverse=True
)

print("\nV9 THREE-FOLD OOF COMPARISON", flush=True)

for score, name in ranked:
    print(name, round(score, 6), flush=True)

# require an improvement across at least two folds
selected = None

for candidate_auc, candidate_name in ranked:

    if candidate_name == "v8_original":
        continue

    if candidate_auc - base_auc < 0.0003:
        continue

    candidate_gains = []

    for _, va in folds.split(X9, y9):

        gain = (
            roc_auc_score(
                y9[va],
                candidates[candidate_name][0][va]
            )
            - roc_auc_score(
                y9[va],
                v8_oof[va]
            )
        )

        candidate_gains.append(gain)

    if sum(g > 0 for g in candidate_gains) >= 2:
        selected = (
            candidate_auc,
            candidate_name,
            candidate_gains
        )
        break

if selected is None:

    print(
        "\nNO CONSISTENT OFFLINE IMPROVEMENT."
        " Keep V8 (public AUC 0.74192).",
        flush=True
    )

else:

    best_auc, best_name, gains = selected

    print(
        "SELECTED:", best_name,
        "OOF AUC:", round(best_auc, 6),
        flush=True
    )

    print(
        "FOLD IMPROVEMENTS:",
        [round(g, 6) for g in gains],
        flush=True
    )

    # bootstrap the OOF improvement
    rng = np.random.default_rng(2026)
    diffs = []

    for _ in range(100):

        idx = rng.integers(0, n, n)

        if np.unique(y9[idx]).size < 2:
            continue

        new_auc = roc_auc_score(
            y9[idx],
            candidates[best_name][0][idx]
        )

        old_auc = roc_auc_score(
            y9[idx],
            v8_oof[idx]
        )

        diffs.append(new_auc - old_auc)

    print(
        "BOOTSTRAP AUC GAIN 5%-95%:",
        np.round(
            np.percentile(diffs, [5, 95]),
            6
        ),
        flush=True
    )

    # create the final submission
    pred = np.asarray(candidates[best_name][1])

    out = pd.DataFrame({
        "encounter_id": test["encounter_id"].to_numpy(),
        "readmit_30d": pred
    })

    assert len(out) == len(test)
    assert out["encounter_id"].is_unique
    assert np.isfinite(pred).all()
    assert out["readmit_30d"].between(0, 1).all()

    path = "/kaggle/working/SUBMIT_V9.csv"

    out.to_csv(path, index=False)

    print(
        "\nSAVED:", path,
        "| MODEL:", best_name,
        flush=True
    )

    # create a direct download link
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("ascii")

    display(HTML(
        '<a download="SUBMIT_V9.csv" '
        'href="data:text/csv;base64,'
        + b64
        + '">CLICK HERE TO DOWNLOAD V9 CSV</a>'
    ))


## 9. V10: Comprehensive multi-table aggregation and generalization tests

**Why:** Build an additional feature table from raw linked tables, recording coverage and features added by source. This is an independent feature-engineering experiment. We also examine whether a model trained on **earlier admissions** generalizes to **later admissions**, and whether it generalizes to **previously unseen hospitals**.

**Important:** These are more demanding distribution-shift tests than random cross-validation. They estimate different aspects of generalization; they do not directly predict the private Kaggle score. Some event tables lack usable event timestamps, so encounter-level membership is assumed to imply information available by discharge; that assumption must be reviewed in the data dictionary. **V10 temporal/hospital holdout results evaluate the V10 stress-test models, not the exact final V11 blend.**

### 9.1 Build V10 comprehensive feature table and source coverage audit

Creates one row per encounter, time-filters timestamped events, avoids label-like columns and writes `V10_DATA_AUDIT.csv`.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 38.

In [ ]:
# v10 dataset combination
import os, re, gc
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/kaggle/input")
index_paths = list(ROOT.rglob("index_encounters.csv"))
DATA_DIR = index_paths[0].parent if index_paths else ROOT
FILES = {p.name.lower(): p for p in DATA_DIR.rglob("*.csv")}

ALIASES = {
    "diagnoses.csv": "diagnosis.csv",
    "lab_results.csv": "lab_result.csv",
    "procedures.csv": "procedure.csv",
    "medications.csv": "medication.csv",
    "unit_transfers.csv": "unit_transfer.csv"
}

def read(name):
    path = FILES.get(name.lower()) or FILES.get(
        ALIASES.get(name.lower(), "")
    )
    if path is None:
        print("MISSING:", name, flush=True)
        return None

    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()
    print("LOADED:", name, df.shape, flush=True)
    return df

def pick(df, *options):
    if df is None:
        return None
    lookup = {c.lower(): c for c in df.columns}
    return next(
        (lookup[c.lower()] for c in options if c.lower() in lookup),
        None
    )

def dates(x):
    try:
        return pd.to_datetime(x, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(x, errors="coerce")

def clean_id(df, key):
    df = df.dropna(subset=[key]).copy()
    df[key] = df[key].astype(str).str.strip()
    return df

def safe(col):
    c = col.lower()
    bad = [
        "readmit", "outcome", "label", "target",
        "future_", "next_", "post_discharge",
        "mortality", "date_of_death", "death_date",
        "30d_", "days_until", "days_to_readmission"
    ]
    return not any(s in c for s in bad)

tr0 = read("train.csv")
te0 = read("test.csv")

assert tr0 is not None and te0 is not None
assert {"encounter_id", "readmit_30d"}.issubset(tr0.columns)
assert "encounter_id" in te0
assert tr0.encounter_id.is_unique
assert te0.encounter_id.is_unique
assert tr0.encounter_id.notna().all()
assert te0.encounter_id.notna().all()

assert set(tr0.encounter_id.astype(str)).isdisjoint(
    set(te0.encounter_id.astype(str))
)

V10_Y = tr0.readmit_30d.astype(int).to_numpy()
V10_N = len(tr0)
V10_TEST_IDS = te0.encounter_id.to_numpy()

A = pd.concat(
    [
        tr0.drop(columns=["readmit_30d"]),
        te0.drop(columns=["readmit_30d"], errors="ignore")
    ],
    ignore_index=True,
    sort=False
)

A["encounter_id"] = A.encounter_id.astype(str)
IDS = pd.Index(A.encounter_id)
assert IDS.is_unique

AUDIT = []

def add_features(features, source):
    global A

    if features is None or features.empty:
        print("NO FEATURES:", source, flush=True)
        return

    features = features.copy()
    features.index = features.index.astype(str)

    assert features.index.is_unique, (
        "Duplicate aggregate keys: " + source
    )

    features = features.loc[
        :, ~features.columns.duplicated()
    ]

    features = features.rename(
        columns=lambda c: re.sub(
            r"[^a-zA-Z0-9_]", "_", str(c)
        )
    )

    features = features.loc[
        :, ~features.columns.isin(A.columns)
    ]

    if features.empty:
        return

    matched = int(IDS.isin(features.index).sum())

    AUDIT.append((
        source, matched, len(IDS), len(features.columns)
    ))

    A = pd.concat(
        [
            A.reset_index(drop=True),
            features.reindex(IDS).reset_index(drop=True)
        ],
        axis=1
    )

    print(
        "ADDED:", source,
        len(features.columns), "features;",
        "coverage", matched, "/", len(IDS),
        flush=True
    )

def count_categories(d, col, prefix, top=30):
    if col is None or col not in d:
        return None

    s = (
        d[col].fillna("Unknown")
        .astype(str).str.strip().str.upper()
    )

    topvals = s.value_counts().head(top).index

    temp = pd.DataFrame({
        "encounter_id": d.encounter_id.values,
        "cat": s.where(
            s.isin(topvals), "OTHER"
        ).values
    })

    tab = pd.crosstab(temp.encounter_id, temp.cat)

    tab.columns = [
        prefix + "_" +
        re.sub("[^A-Z0-9]+", "_", c)[:35]
        for c in tab.columns
    ]

    tab = tab.loc[:, ~tab.columns.duplicated()]
    return tab.astype("float32")

def basic_event(d, prefix, categories=(), numeric=()):
    if d is None or "encounter_id" not in d:
        return

    g = d.groupby("encounter_id", sort=False)

    f = pd.DataFrame({
        prefix + "_count": g.size()
    })

    for c, top in categories:
        if c in d:
            f[prefix + "_" + c + "_unique"] = g[c].nunique()

    for c in numeric:
        if c in d:
            d[c] = pd.to_numeric(
                d[c], errors="coerce"
            )

            stats = g[c].agg(
                ["mean", "std", "min", "max"]
            )

            stats.columns = [
                prefix + "_" + c + "_" + s
                for s in stats.columns
            ]

            f = f.join(stats)

    add_features(f, prefix + "_summary")

    for c, top in categories:
        if c in d:
            add_features(
                count_categories(
                    d, c, prefix + "_" + c, top
                ),
                prefix + "_" + c
            )

def encounter_events(
    df, source, timestamp_candidates=()
):
    if df is None or "encounter_id" not in df:
        print(
            "SKIPPED (no encounter_id):",
            source, flush=True
        )
        return None

    df = clean_id(df, "encounter_id")
    df = df[df.encounter_id.isin(IDS)]

    tcol = pick(df, *timestamp_candidates)

    if tcol is not None:
        df = df.merge(
            V10_DATES,
            on="encounter_id",
            how="left",
            validate="many_to_one"
        )

        t = dates(df[tcol])
        start = df.admit_dt
        end = df.discharge_dt

        good = (
            t.notna()
            & start.notna()
            & end.notna()
            & t.ge(start)
            & t.le(end)
        )

        print(
            "TIME FILTER:", source,
            "kept", int(good.sum()), "/", len(df),
            flush=True
        )

        df = df.loc[good].copy()
        df["_event_dt"] = t.loc[good]

    else:
        print(
            "NO EVENT TIMESTAMP:", source,
            "(using encounter-scoped records only)",
            flush=True
        )

    return df


# 1. Index encounters

idx = read("index_encounters.csv")

if idx is None or "encounter_id" not in idx:
    raise RuntimeError(
        "index_encounters.csv with encounter_id is required"
    )

idx = clean_id(idx, "encounter_id")

assert idx.encounter_id.is_unique, (
    "index_encounters.csv must have one row per encounter"
)

idx = idx[idx.encounter_id.isin(IDS)]

admit = pick(
    idx, "admit_dt", "admission_dt",
    "admission_date", "admit_date"
)

disch = pick(
    idx, "discharge_dt",
    "discharge_date", "discharged_dt"
)

if admit is None or disch is None:
    raise RuntimeError(
        "Cannot locate admission/discharge date columns. "
        + str(idx.columns.tolist())
    )

idx["admit_dt"] = dates(idx[admit])
idx["discharge_dt"] = dates(idx[disch])

V10_DATES = idx[
    ["encounter_id", "admit_dt", "discharge_dt"]
].copy()

V10_DATES = (
    V10_DATES.set_index("encounter_id")
    .reindex(IDS)
    .reset_index()
)

V10_DATES.columns = [
    "encounter_id", "admit_dt", "discharge_dt"
]

assert V10_DATES.encounter_id.is_unique

V10_ADMIT = (
    V10_DATES.admit_dt.iloc[:V10_N]
    .reset_index(drop=True)
)

if "hospital_id" in idx:
    V10_HOSP = (
        idx.set_index("encounter_id")
        .reindex(IDS)["hospital_id"]
        .iloc[:V10_N]
        .reset_index(drop=True)
    )
else:
    V10_HOSP = pd.Series(dtype="object")

static = idx.drop(
    columns=[
        c for c in idx
        if c in [
            admit, disch, "admit_dt", "discharge_dt"
        ] and c != "encounter_id"
    ],
    errors="ignore"
)

static = static.set_index("encounter_id")
static = static[[c for c in static if safe(c)]]

add_features(static, "index_encounters")

A["v10_stay_days"] = (
    V10_DATES.discharge_dt
    - V10_DATES.admit_dt
).dt.total_seconds() / 86400

A["v10_admit_month"] = V10_DATES.admit_dt.dt.month
A["v10_admit_weekday"] = V10_DATES.admit_dt.dt.dayofweek
A["v10_admit_hour"] = V10_DATES.admit_dt.dt.hour

A["v10_discharge_weekday"] = (
    V10_DATES.discharge_dt.dt.dayofweek
)

A["v10_discharge_hour"] = (
    V10_DATES.discharge_dt.dt.hour
)


# 2. Patients and hospitals

for fname, key, pref in [
    ("patients.csv", "patient_id", "patient"),
    ("hospitals.csv", "hospital_id", "hospital")
]:
    df = read(fname)

    if df is None or key not in df or key not in A:
        print(
            "SKIPPED STATIC JOIN:",
            fname, flush=True
        )
        continue

    df = clean_id(df, key)

    if df[key].duplicated().any():
        print(
            "WARNING: duplicate keys in",
            fname, "- keeping first record",
            flush=True
        )
        df = df.drop_duplicates(key)

    use = [
        c for c in df
        if c != key and safe(c)
    ]

    if pref == "patient":
        birth = pick(
            df, "dob", "date_of_birth", "birth_date"
        )

        if birth is not None:
            birth_map = df.set_index(key)[birth]

            b = dates(
                A[key].astype(str).map(birth_map)
            )

            A["v10_age_from_dob"] = (
                V10_DATES.admit_dt - b
            ).dt.days / 365.25

            A.loc[
                ~A["v10_age_from_dob"].between(0, 115),
                "v10_age_from_dob"
            ] = np.nan

    use = [
        c for c in use
        if not re.search(
            "date|_dt$|timestamp|dob|birth",
            c, re.I
        )
    ]

    part = (
        df.set_index(key)[use]
        .add_prefix("v10_" + pref + "_")
    )

    part.index = part.index.astype(str)

    add_features(
        part.reindex(
            A[key].astype(str)
        ).set_axis(IDS),
        fname
    )

    del df, part


# 3. Diagnoses and ICD-10 hierarchy

D = read("diagnoses.csv")
D = encounter_events(D, "diagnoses")

if D is not None and len(D):
    code = pick(
        D, "icd10_code", "icd_10_code",
        "diagnosis_code", "code"
    )

    if code:
        D["code3"] = (
            D[code].fillna("UNK")
            .astype(str).str.upper()
            .str.replace(".", "", regex=False)
            .str[:3]
        )

        D["icd_chapter"] = D["code3"].str[0]

        lookup = read("icd_10_lookup.csv")

        if lookup is not None:
            lkcode = pick(
                lookup, "icd10_code",
                "icd_10_code", "code"
            )

            category = pick(
                lookup, "category", "chapter",
                "diagnosis_group", "group"
            )

            if lkcode and category:
                lk = (
                    lookup[[lkcode, category]]
                    .drop_duplicates(lkcode)
                )

                lk[lkcode] = lk[lkcode].astype(str)
                D[code] = D[code].astype(str)

                D = D.merge(
                    lk,
                    left_on=code,
                    right_on=lkcode,
                    how="left",
                    validate="many_to_one"
                )

                if category in D:
                    D["lookup_group"] = D[category]

        basic_event(
            D, "v10_dx",
            [
                (code, 35),
                ("code3", 45),
                ("icd_chapter", 22),
                ("lookup_group", 20)
            ]
        )

    else:
        basic_event(D, "v10_dx")

    del D
    gc.collect()


# 4. Procedures

D = encounter_events(
    read("procedures.csv"),
    "procedures",
    (
        "performed_dt", "procedure_dt",
        "procedure_date", "event_dt"
    )
)

if D is not None and len(D):
    code = pick(
        D, "procedure_code", "cpt_code",
        "icd10_pcs", "code",
        "procedure_type", "procedure_name",
        "category"
    )

    basic_event(
        D, "v10_proc",
        [(code, 35)] if code else []
    )

    del D
    gc.collect()


# 5. Medications

D = encounter_events(
    read("medications.csv"),
    "medications",
    (
        "administered_dt", "ordered_dt",
        "start_dt", "medication_dt", "event_dt"
    )
)

if D is not None and len(D):
    med = pick(
        D, "drug_class", "med_class",
        "medication_class", "therapeutic_class"
    )

    name = pick(
        D, "drug_name",
        "medication_name", "med_name"
    )

    continued = pick(
        D, "continued_at_discharge",
        "discharge_medication"
    )

    basic_event(
        D, "v10_med",
        [(med, 40)] if med else []
    )

    if name:
        add_features(
            count_categories(
                D, name, "v10_med_name", 25
            ),
            "medication_names"
        )

    if continued:
        f = (
            D[continued].fillna("")
            .astype(str).str.lower()
            .isin(["1", "true", "yes", "y"])
        )

        add_features(
            f.groupby(D.encounter_id)
            .sum()
            .to_frame("v10_meds_continued"),
            "meds_at_discharge"
        )

    del D
    gc.collect()


# 6. Unit transfers and ICU exposure

D = encounter_events(
    read("unit_transfers.csv"),
    "unit_transfers",
    (
        "transfer_dt", "transfer_time",
        "start_dt", "event_dt"
    )
)

if D is not None and len(D):
    unit = pick(
        D, "unit_type", "unit_name",
        "to_unit", "from_unit",
        "care_unit", "unit"
    )

    basic_event(
        D, "v10_transfer",
        [(unit, 20)] if unit else []
    )

    if unit:
        icu = (
            D[unit].fillna("")
            .astype(str)
            .str.contains(
                "ICU|INTENSIVE|CRITICAL",
                case=False, regex=True
            )
        )

        add_features(
            icu.groupby(D.encounter_id)
            .sum()
            .to_frame("v10_icu_transfer_count"),
            "icu_transfers"
        )

    del D
    gc.collect()


# 7. Discharge planning

D = encounter_events(
    read("discharge_planning.csv"),
    "discharge_planning",
    (
        "created_dt", "recorded_dt",
        "plan_dt", "event_dt"
    )
)

if D is not None and len(D):
    cat = [
        (c, 12)
        for c in D.columns
        if c != "encounter_id"
        and safe(c)
        and not re.search(
            "date|_dt|time|id|note|text|description",
            c, re.I
        )
        and 1 < D[c].nunique(dropna=True) <= 40
    ][:10]

    nums = [
        c for c in D.select_dtypes(
            include="number"
        ).columns
        if safe(c)
        and not re.search(
            "id|target|label|readmit", c, re.I
        )
    ][:8]

    basic_event(
        D, "v10_plan", cat, nums
    )

    del D
    gc.collect()


# 8. Prior utilization: only visits before admission

D = read("prior_utilization.csv")

if (
    D is not None
    and "patient_id" in D
    and "patient_id" in A
):
    tcol = pick(
        D, "visit_dt", "visit_date",
        "encounter_dt", "admit_dt"
    )

    if tcol:
        pid = A[
            ["encounter_id", "patient_id"]
        ].copy()

        pid["admit_dt"] = V10_DATES.admit_dt.values
        pid = clean_id(pid, "patient_id")

        D = clean_id(D, "patient_id")
        D["_visit"] = dates(D[tcol])

        D = D.merge(
            pid, on="patient_id", how="inner"
        )

        D["days_ago"] = (
            D.admit_dt - D._visit
        ).dt.total_seconds() / 86400

        D = D[
            D.days_ago.gt(0)
            & D.days_ago.le(3650)
        ].copy()

        print(
            "HISTORY strictly before admission:",
            len(D), flush=True
        )

        g = D.groupby("encounter_id")

        f = pd.DataFrame({
            "v10_history_total": g.size(),
            "v10_history_last_days": g.days_ago.min(),
            "v10_history_first_days": g.days_ago.max()
        })

        for w in (7, 30, 90, 180, 365, 730):
            f["v10_history_" + str(w) + "d"] = (
                D.days_ago.le(w)
                .groupby(D.encounter_id).sum()
            )

        add_features(
            f, "prior_utilization"
        )

        kind = pick(
            D, "visit_type",
            "encounter_type", "care_setting"
        )

        if kind:
            add_features(
                count_categories(
                    D, kind, "v10_prior_type", 10
                ),
                "prior_visit_type"
            )

        del pid

    else:
        print(
            "SKIPPED HISTORY: no visit date (unsafe)",
            flush=True
        )

    del D
    gc.collect()


# 9. Labs: abnormalities, test types and trends

D = encounter_events(
    read("lab_results.csv"),
    "lab_results",
    (
        "collected_dt", "collection_dt",
        "result_dt", "lab_dt", "charttime"
    )
)

if D is not None and len(D):
    name = pick(
        D, "lab_name", "test_name",
        "lab_test", "analyte"
    )

    val = pick(
        D, "value", "result_value",
        "numeric_value", "lab_value", "result"
    )

    flag = pick(
        D, "abnormal_flag", "flag",
        "abnormality"
    )

    if flag:
        f = (
            D[flag].fillna("")
            .astype(str).str.upper().str.strip()
        )

        D["_abnormal"] = f.isin([
            "H", "L", "HH", "LL", "HIGH", "LOW",
            "ABNORMAL", "ABN", "A",
            "CRITICAL", "CRIT"
        ]).astype("int8")

        D["_high"] = f.isin(
            ["H", "HIGH"]
        ).astype("int8")

        D["_low"] = f.isin(
            ["L", "LOW"]
        ).astype("int8")

    else:
        D["_abnormal"] = 0
        D["_high"] = 0
        D["_low"] = 0

    g = D.groupby("encounter_id")

    f = g[
        ["_abnormal", "_high", "_low"]
    ].sum().rename(columns={
        "_abnormal": "v10_lab_abnormal",
        "_high": "v10_lab_high",
        "_low": "v10_lab_low"
    })

    f["v10_lab_count"] = g.size()

    f["v10_lab_abnormal_rate"] = (
        f.v10_lab_abnormal / f.v10_lab_count
    )

    if name:
        f["v10_lab_unique"] = g[name].nunique()

    add_features(
        f, "lab_summary"
    )

    if name:
        add_features(
            count_categories(
                D, name, "v10_lab_type", 25
            ),
            "lab_test_counts"
        )

        if val:
            D["_num"] = pd.to_numeric(
                D[val], errors="coerce"
            )

            D[name] = (
                D[name].astype(str)
                .str.upper().str.strip()
            )

            top = (
                D[name].value_counts()
                .head(22).index
            )

            small = D[
                D[name].isin(top)
                & D._num.notna()
            ].copy()

            if len(small):
                if "_event_dt" in small:
                    small = small.sort_values(
                        "_event_dt"
                    )

                g2 = small.groupby(
                    ["encounter_id", name]
                )["_num"]

                stats = g2.agg([
                    "mean", "min", "max",
                    "first", "last"
                ])

                stats["delta"] = (
                    stats["last"] - stats["first"]
                )

                wide = stats.unstack(name)

                wide.columns = [
                    "v10_lab_" + str(lab)[:20]
                    + "_" + stat
                    for stat, lab in wide.columns
                ]

                add_features(
                    wide, "lab_numeric_trajectories"
                )

                del small, stats, wide

    del D
    gc.collect()


# 10. Vitals: full-stay and early/late trajectories

D = encounter_events(
    read("vitals.csv"),
    "vitals",
    (
        "charttime", "recorded_dt",
        "vital_dt", "event_dt"
    )
)

if D is not None and len(D):
    typ = pick(
        D, "vital_type",
        "vital_name", "measurement_type"
    )

    val = pick(
        D, "value", "vital_value",
        "numeric_value"
    )

    if typ and val:
        D[typ] = (
            D[typ].fillna("UNK")
            .astype(str).str.upper().str.strip()
        )

        D["_num"] = pd.to_numeric(
            D[val], errors="coerce"
        )

        temp = (
            D[typ].eq("TEMP")
            & D._num.between(45, 115)
        )

        D.loc[temp, "_num"] = (
            D.loc[temp, "_num"] - 32
        ) * 5 / 9

        bounds = {
            "HR": (20, 250),
            "SBP": (40, 300),
            "DBP": (20, 200),
            "RR": (4, 70),
            "SPO2": (40, 100),
            "TEMP": (30, 43)
        }

        for t, (lo, hi) in bounds.items():
            bad = (
                D[typ].eq(t)
                & ~D._num.between(lo, hi)
            )
            D.loc[bad, "_num"] = np.nan

        D = D[
            D[typ].isin(bounds)
        ].copy()

        if "_event_dt" in D:
            D = D.sort_values("_event_dt")

        g = D.groupby(
            ["encounter_id", typ]
        )["_num"]

        stats = g.agg([
            "count", "mean", "std",
            "min", "max", "first", "last"
        ])

        stats["change"] = (
            stats["last"] - stats["first"]
        )

        wide = stats.unstack(typ)

        wide.columns = [
            "v10_vital_" + str(t) + "_" + s
            for s, t in wide.columns
        ]

        add_features(
            wide, "vital_full_stay"
        )

        if "_event_dt" in D:
            d_end = D["discharge_dt"]

            D["_last24"] = (
                d_end - D["_event_dt"]
            ).dt.total_seconds().between(
                0, 86400
            )

            last = (
                D[D._last24]
                .groupby(["encounter_id", typ])["_num"]
                .mean().unstack(typ)
            )

            last.columns = [
                "v10_vital_" + str(c)
                + "_last24mean"
                for c in last.columns
            ]

            add_features(
                last, "vital_last24"
            )

            D["_first24"] = (
                D["_event_dt"] - D["admit_dt"]
            ).dt.total_seconds().between(
                0, 86400
            )

            first = (
                D[D._first24]
                .groupby(["encounter_id", typ])["_num"]
                .mean().unstack(typ)
            )

            first.columns = [
                "v10_vital_" + str(c)
                + "_first24mean"
                for c in first.columns
            ]

            add_features(
                first, "vital_first24"
            )

    else:
        basic_event(
            D, "v10_vital"
        )

    del D
    gc.collect()


# 11. Cross-table clinical interactions

pairs = [
    ("v10_dx_count", "v10_stay_days", "dx_per_day"),
    ("v10_med_count", "v10_stay_days", "meds_per_day"),
    ("v10_lab_abnormal", "v10_lab_count", "abnormal_fraction"),
    ("v10_history_30d", "v10_history_365d", "recent_visit_share"),
    ("v10_transfer_count", "v10_stay_days", "transfers_per_day"),
    ("v10_proc_count", "v10_stay_days", "procedures_per_day")
]

for a, b, name in pairs:
    if a in A and b in A:
        A["v10_cross_" + name] = (
            pd.to_numeric(A[a], errors="coerce")
            / (
                pd.to_numeric(A[b], errors="coerce")
                + 1
            )
        )

for a, b, name in [
    (
        "v10_vital_HR_last24mean",
        "v10_vital_SBP_last24mean",
        "shock_index"
    ),
    (
        "v10_vital_SPO2_last24mean",
        "v10_vital_RR_last24mean",
        "oxygen_resp_ratio"
    )
]:
    if a in A and b in A:
        A["v10_cross_" + name] = (
            pd.to_numeric(A[a], errors="coerce")
            / (
                pd.to_numeric(A[b], errors="coerce")
                + 0.1
            )
        )


# 12. Reuse V8 features when row order is verified

oldx = globals().get("cat_X", None)
oldt = globals().get("cat_T", None)

if not (
    isinstance(oldx, pd.DataFrame)
    and isinstance(oldt, pd.DataFrame)
    and len(oldx) == V10_N
    and len(oldt) == len(te0)
):
    oldx = globals().get(
        "X_train_v4_ready", None
    )
    oldt = globals().get(
        "X_test_v4_ready", None
    )

if (
    isinstance(oldx, pd.DataFrame)
    and isinstance(oldt, pd.DataFrame)
    and len(oldx) == V10_N
    and len(oldt) == len(te0)
):
    if (
        "train" in globals()
        and isinstance(train, pd.DataFrame)
        and "encounter_id" in train
    ):
        aligned = (
            train.encounter_id.astype(str)
            .reset_index(drop=True)
            .equals(
                tr0.encounter_id.astype(str)
                .reset_index(drop=True)
            )
        )
    else:
        aligned = False

    if aligned:
        old = pd.concat(
            [
                oldx.reset_index(drop=True),
                oldt.reset_index(drop=True)
            ],
            ignore_index=True
        )

        old = old.loc[
            :, ~old.columns.duplicated()
        ]

        old = old.rename(
            columns=lambda c: "v8_" + str(c)
        )

        old = old.loc[
            :, [safe(c) for c in old.columns]
        ]

        A = pd.concat(
            [
                A.reset_index(drop=True),
                old.reset_index(drop=True)
            ],
            axis=1
        )

        print(
            "REUSED PREVIOUS FEATURES:",
            len(old.columns), flush=True
        )

    else:
        print(
            "OLD FEATURES NOT REUSED:",
            "train row order not verified",
            flush=True
        )


# 13. Clean the master feature table

remove = [
    c for c in A
    if c in ["encounter_id", "patient_id"]
    or not safe(c)
    or re.search(
        r"(^|_)(admit_dt|discharge_dt|birth_date|dob|date_of_birth)(_|$)",
        c, re.I
    )
]

A = A.drop(
    columns=remove, errors="ignore"
)

A = A.loc[
    :, ~A.columns.duplicated()
]

for c in list(A.columns):
    if pd.api.types.is_datetime64_any_dtype(A[c]):
        A = A.drop(columns=c)
        continue

    if not pd.api.types.is_numeric_dtype(A[c]):
        A[c] = (
            A[c].fillna("Missing")
            .astype(str)
        )

        if (
            A[c].nunique(dropna=False) > 1000
            or A[c].nunique(dropna=False) > 0.4 * len(A)
        ):
            A = A.drop(columns=c)
        else:
            A[c] = A[c].astype("category")

    else:
        A[c] = (
            pd.to_numeric(A[c], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .astype("float32")
        )

A["v10_missing_count"] = (
    A.isna().sum(axis=1).astype("float32")
)

A = A.loc[
    :, A.notna().any(axis=0)
]

A.columns = [
    re.sub("[^a-zA-Z0-9_]", "_", str(c))
    for c in A.columns
]

A = A.loc[
    :, ~A.columns.duplicated()
]

V10_X = (
    A.iloc[:V10_N]
    .reset_index(drop=True)
)

V10_T = (
    A.iloc[V10_N:]
    .reset_index(drop=True)
)

assert V10_X.columns.equals(V10_T.columns)
assert len(V10_X) == len(tr0)
assert len(V10_T) == len(te0)
assert V10_X.columns.is_unique

audit = pd.DataFrame(
    AUDIT,
    columns=[
        "source", "matched_encounters",
        "total_encounters", "added_features"
    ]
)

audit.to_csv(
    "/kaggle/working/V10_DATA_AUDIT.csv",
    index=False
)

print(
    "\nV10 MASTER READY:",
    V10_X.shape,
    "TEST:", V10_T.shape,
    flush=True
)

print(
    "FILES FOUND:", len(FILES),
    "| SOURCES ADDED:", len(AUDIT),
    flush=True
)

print(audit.to_string(index=False))

del A
gc.collect()


### 9.2 V10 cross-validation, ablation and distribution-shift tests

Compare CatBoost and LightGBM, test the value of extra features, and evaluate chronological and hospital-group holdouts. A small additive collector records stress-test results for the report.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 39.

In [ ]:

import base64, gc
import numpy as np
import pandas as pd

from IPython.display import display, HTML
from sklearn.model_selection import (
    StratifiedKFold, GroupShuffleSplit
)
from sklearn.metrics import (
    roc_auc_score, average_precision_score
)
from catboost import CatBoostClassifier
from lightgbm import (
    LGBMClassifier, early_stopping, log_evaluation
)

assert (
    "V10_X" in globals()
    and "V10_T" in globals()
), "Run V10 Cell 1 first."

X = V10_X.copy()
T = V10_T.copy()
y = V10_Y

cats = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for c in cats:
    combined = pd.concat(
        [X[c], T[c]],
        ignore_index=True
    ).astype("object").fillna("Missing").astype(str)

    categories = pd.Categorical(combined).categories

    X[c] = pd.Categorical(
        X[c].astype("object")
        .fillna("Missing").astype(str),
        categories=categories
    )

    T[c] = pd.Categorical(
        T[c].astype("object")
        .fillna("Missing").astype(str),
        categories=categories
    )

# CatBoost uses strings; LightGBM uses categories.
CX = X.copy()
CT = T.copy()

for c in cats:
    CX[c] = CX[c].astype(str)
    CT[c] = CT[c].astype(str)

folds = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

co = np.zeros(len(y))
lo = np.zeros(len(y))

ct = np.zeros(len(T))
lt = np.zeros(len(T))

rows = []
best_iters = []
cb_best_iters = []

# 1. Three-fold cross-validation

for k, (tr, va) in enumerate(
    folds.split(X, y), 1
):
    print(
        "V10 FOLD", k, "/ 3: CatBoost",
        flush=True
    )

    cb = CatBoostClassifier(
        iterations=1200,
        depth=6,
        learning_rate=0.035,
        l2_leaf_reg=9,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=100 + k,
        thread_count=4,
        verbose=False,
        allow_writing_files=False
    )

    cb.fit(
        CX.iloc[tr],
        y[tr],
        cat_features=cats,
        eval_set=(CX.iloc[va], y[va]),
        early_stopping_rounds=110,
        verbose=False
    )

    co[va] = cb.predict_proba(
        CX.iloc[va]
    )[:, 1]

    ct += cb.predict_proba(
        CT
    )[:, 1] / 3

    cb_best_iters.append(
        max(100, cb.get_best_iteration() + 1)
    )

    cb_train = roc_auc_score(
        y[tr],
        cb.predict_proba(CX.iloc[tr])[:, 1]
    )

    cb_val = roc_auc_score(
        y[va], co[va]
    )

    rows.append((
        k, "CatBoost",
        cb_train, cb_val,
        cb_train - cb_val
    ))

    del cb
    gc.collect()

    print(
        "V10 FOLD", k, "/ 3: LightGBM",
        flush=True
    )

    lg = LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=85,
        colsample_bytree=0.82,
        subsample=0.85,
        subsample_freq=1,
        reg_lambda=10,
        reg_alpha=1,
        random_state=200 + k,
        n_jobs=4,
        verbosity=-1
    )

    lg.fit(
        X.iloc[tr],
        y[tr],
        eval_set=[(X.iloc[va], y[va])],
        eval_metric="auc",
        callbacks=[
            early_stopping(
                110, verbose=False
            ),
            log_evaluation(0)
        ]
    )

    lo[va] = lg.predict_proba(
        X.iloc[va]
    )[:, 1]

    lt += lg.predict_proba(
        T
    )[:, 1] / 3

    lg_train = roc_auc_score(
        y[tr],
        lg.predict_proba(X.iloc[tr])[:, 1]
    )

    lg_val = roc_auc_score(
        y[va], lo[va]
    )

    rows.append((
        k, "LightGBM",
        lg_train, lg_val,
        lg_train - lg_val
    ))

    best_iters.append(lg.best_iteration_)

    print(
        "FOLD", k,
        "CatBoost", round(cb_val, 6),
        "LightGBM", round(lg_val, 6),
        flush=True
    )

    del lg
    gc.collect()


# 2. Training vs validation AUC

fold_table = pd.DataFrame(
    rows,
    columns=[
        "fold", "model", "train_auc",
        "validation_auc", "overfit_gap"
    ]
)

print("\nFOLD VALIDATION AND TRAINING GAP")

print(
    fold_table.round(5).to_string(index=False)
)


# 3. Compare models and ensembles

new_oof = 0.5 * co + 0.5 * lo
new_test = 0.5 * ct + 0.5 * lt

candidates = {
    "v10_catboost": (co, ct),
    "v10_lightgbm": (lo, lt),
    "v10_ensemble": (
        new_oof, new_test
    )
}

# Only reuse V8 predictions if rows align.
prior_oof = globals().get("v8_oof", None)
prior_test = globals().get("v8_test", None)

old_train = globals().get("train", None)
old_test = globals().get("test", None)

aligned = (
    isinstance(old_train, pd.DataFrame)
    and isinstance(old_test, pd.DataFrame)
    and "encounter_id" in old_train
    and "encounter_id" in old_test
    and old_train.encounter_id.astype(str)
        .reset_index(drop=True)
        .equals(
            tr0.encounter_id.astype(str)
            .reset_index(drop=True)
        )
    and old_test.encounter_id.astype(str)
        .reset_index(drop=True)
        .equals(
            te0.encounter_id.astype(str)
            .reset_index(drop=True)
        )
)

if (
    aligned
    and prior_oof is not None
    and prior_test is not None
):
    old_oof = np.asarray(
        prior_oof
    ).reshape(-1)

    old_pred = np.asarray(
        prior_test
    ).reshape(-1)

    if (
        len(old_oof) == len(y)
        and len(old_pred) == len(T)
    ):
        candidates["v8_reference"] = (
            old_oof, old_pred
        )

        candidates["v10_v8_30_new70"] = (
            0.3 * old_oof + 0.7 * new_oof,
            0.3 * old_pred + 0.7 * new_test
        )

        candidates["v10_v8_50_new50"] = (
            0.5 * old_oof + 0.5 * new_oof,
            0.5 * old_pred + 0.5 * new_test
        )

        print(
            "VERIFIED: V8 predictions aligned."
        )

else:
    print(
        "V8 predictions unavailable or unverified. "
        "Historical V8 OOF ROC-AUC: 0.748035."
    )

ranked = sorted(
    [
        (
            roc_auc_score(y, p[0]),
            name
        )
        for name, p in candidates.items()
    ],
    reverse=True
)

print("\nV10 THREE-FOLD ROC-AUC COMPARISON")

for auc, name in ranked:
    ap = average_precision_score(
        y, candidates[name][0]
    )

    print(
        f"{name:24s} "
        f"ROC-AUC={auc:.6f} "
        f"PR-AUC={ap:.6f}"
    )


# 4. Feature ablation: old vs comprehensive

v8_cols = [
    c for c in X
    if c.startswith("v8_")
]

if len(v8_cols) > 100:
    tr, va = next(folds.split(X, y))

    baseline = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=85,
        reg_lambda=10,
        n_jobs=4,
        verbosity=-1
    )

    baseline.fit(
        X.iloc[tr][v8_cols],
        y[tr],
        eval_set=[
            (X.iloc[va][v8_cols], y[va])
        ],
        callbacks=[
            early_stopping(
                90, verbose=False
            ),
            log_evaluation(0)
        ]
    )

    baseline_auc = roc_auc_score(
        y[va],
        baseline.predict_proba(
            X.iloc[va][v8_cols]
        )[:, 1]
    )

    full_auc = roc_auc_score(
        y[va], lo[va]
    )

    print(
        "\nABLATION (LightGBM fold 1):",
        "V8 features only:",
        round(baseline_auc, 6),
        "| V10 comprehensive:",
        round(full_auc, 6),
        "| gain:",
        round(full_auc - baseline_auc, 6)
    )

    del baseline
    gc.collect()


# 5. Generalization to later admissions and hospitals

fixed_trees = max(
    100, int(np.median(best_iters))
)

fixed_cb = int(
    np.median(cb_best_iters)
)

print(
    "\nGENERALIZATION STRESS TESTS",
    "| LightGBM trees:", fixed_trees,
    "| CatBoost trees:", fixed_cb
)

STRESS_RESULTS = []

def stress(name, tr, va):
    if (
        len(tr) < 1000
        or len(va) < 1000
        or len(np.unique(y[va])) < 2
    ):
        print(
            name, "SKIPPED: insufficient data"
        )
        return

    m = LGBMClassifier(
        n_estimators=fixed_trees,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=85,
        colsample_bytree=0.82,
        reg_lambda=10,
        reg_alpha=1,
        n_jobs=4,
        verbosity=-1,
        random_state=999
    )

    m.fit(
        X.iloc[tr], y[tr]
    )

    p_train = m.predict_proba(
        X.iloc[tr]
    )[:, 1]

    p_val = m.predict_proba(
        X.iloc[va]
    )[:, 1]

    print(
        f"{name} LightGBM: "
        f"train={roc_auc_score(y[tr], p_train):.6f} "
        f"holdout={roc_auc_score(y[va], p_val):.6f}",
        flush=True
    )

    del m
    gc.collect()

    c = CatBoostClassifier(
        iterations=fixed_cb,
        depth=6,
        learning_rate=0.035,
        l2_leaf_reg=9,
        loss_function="Logloss",
        random_seed=999,
        thread_count=4,
        verbose=False,
        allow_writing_files=False
    )

    c.fit(
        CX.iloc[tr],
        y[tr],
        cat_features=cats,
        verbose=False
    )

    q_train = c.predict_proba(
        CX.iloc[tr]
    )[:, 1]

    q_val = c.predict_proba(
        CX.iloc[va]
    )[:, 1]

    print(
        f"{name} CatBoost: "
        f"train={roc_auc_score(y[tr], q_train):.6f} "
        f"holdout={roc_auc_score(y[va], q_val):.6f}",
        flush=True
    )

    blend_train = 0.5 * (
        p_train + q_train
    )

    blend_val = 0.5 * (
        p_val + q_val
    )

    a = roc_auc_score(
        y[tr], blend_train
    )

    b = roc_auc_score(
        y[va], blend_val
    )

    print(
        f"{name} ENSEMBLE: "
        f"train={a:.6f} "
        f"holdout={b:.6f} "
        f"gap={a-b:.6f} "
        f"holdout_n={len(va)} "
        f"holdout_prevalence={y[va].mean():.3f}",
        flush=True
    )

    STRESS_RESULTS.append({
        'test': name, 'train_auc': float(a),
        'holdout_auc': float(b), 'gap': float(a-b),
        'holdout_n': int(len(va)),
        'holdout_prevalence': float(y[va].mean())
    })

    del c
    gc.collect()


# Temporal holdout: train on earlier admissions.

admit = V10_ADMIT
valid = admit.notna().to_numpy()

if (
    valid.mean() > 0.80
    and admit[valid].nunique() > 20
):
    order = np.where(valid)[0][
        np.argsort(
            admit[valid].to_numpy()
        )
    ]

    cut = int(
        len(order) * 0.8
    )

    stress(
        "TEMPORAL (earlier -> later admissions)",
        order[:cut],
        order[cut:]
    )

else:
    print(
        "TEMPORAL: skipped "
        "(dates unavailable or insufficient variation)"
    )


# Hospital holdout: train without some hospitals.

if (
    "V10_HOSP" in globals()
    and V10_HOSP.notna().sum() > 0
    and V10_HOSP.nunique() > 5
):
    grp = (
        V10_HOSP.fillna("Missing")
        .astype(str).to_numpy()
    )

    tr, va = next(
        GroupShuffleSplit(
            n_splits=1,
            test_size=0.2,
            random_state=42
        ).split(
            X, y, groups=grp
        )
    )

    stress(
        "HOSPITAL (unseen hospitals)",
        tr, va
    )

else:
    print(
        "HOSPITAL: skipped "
        "(hospital identifiers unavailable)"
    )


# 6. Select candidate and bootstrap AUC difference

new_ranked = [
    (auc, name)
    for auc, name in ranked
    if name != "v8_reference"
]

best_auc, best_name = new_ranked[0]

if "v8_reference" in candidates:
    old_ref = candidates[
        "v8_reference"
    ][0]

    chosen = candidates[
        best_name
    ][0]

    rng = np.random.default_rng(2026)
    diffs = []

    for _ in range(150):
        ix = rng.integers(
            0, len(y), len(y)
        )

        if len(np.unique(y[ix])) < 2:
            continue

        diffs.append(
            roc_auc_score(
                y[ix], chosen[ix]
            )
            -
            roc_auc_score(
                y[ix], old_ref[ix]
            )
        )

    print(
        "\n",
        best_name,
        "minus V8 bootstrap AUC gain",
        "(5th, 50th, 95th percentiles):",
        np.round(
            np.percentile(
                diffs, [5, 50, 95]
            ),
            6
        )
    )


# 7. Save submission, but do not submit automatically

output = pd.DataFrame({
    "encounter_id": V10_TEST_IDS,
    "readmit_30d": candidates[
        best_name
    ][1]
})

assert len(output) == len(te0)
assert output.encounter_id.is_unique
assert output.readmit_30d.notna().all()
assert output.readmit_30d.between(0, 1).all()

path = "/kaggle/working/SUBMIT_V10.csv"

output.to_csv(
    path, index=False
)

print(
    "\nSAVED:", path,
    "|", best_name,
    "| CV ROC-AUC", round(best_auc, 6)
)

print(
    "V8 historical offline ROC-AUC: 0.748035",
    "| V8 public ROC-AUC: 0.741920"
)

if best_auc <= 0.748035:
    print(
        "WARNING: V10 did not beat historical V8 "
        "offline. DO NOT SUBMIT yet."
    )
else:
    print(
        "V10 improved offline. Review fold consistency, "
        "stress tests, and bootstrap BEFORE submitting."
    )

with open(path, "rb") as f:
    encoded = base64.b64encode(
        f.read()
    ).decode("ascii")

display(HTML(
    '<a download="SUBMIT_V10.csv" '
    'href="data:text/csv;base64,'
    + encoded
    + '">DOWNLOAD V10 CSV</a>'
))


## 10. V11: Final selected modeling family

**Why:** Retain V9's strong feature representation, add selected V10 clinical features and restore medication-class indicators. Train XGBoost and CatBoost under the same three-fold splits and compare OOF blends with the earlier V8/V9 predictions. Report **train–validation AUC gaps** rather than interpreting high training AUC as success.

**Historical note:** V11 used 855 numeric features after its feature-selection steps on the original Kaggle data. This count is checked below; if libraries or source files change, the count and predictions may change.

### 10.1 Construct 855-feature V11 matrix, train and compare

Creates `Ztr`/`Zte`, trains both models, records per-fold overfitting and assembles the V8/V9/V11 ensemble options.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 40.

In [ ]:

from pathlib import Path
import gc, re, base64
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from IPython.display import HTML, display

# Check previous results are available
needed = [
    "X9", "T9", "V10_X", "V10_T",
    "tr0", "te0", "train", "test",
    "v8_oof", "v8_test",
    "xgb_oof", "xgb_test"
]

missing = [s for s in needed if s not in globals()]

if missing:
    raise RuntimeError(
        "Missing: " + ", ".join(missing)
        + ". Use the same notebook session after V9 and V10."
    )

# Verify that all three versions use identical encounter order
assert train.encounter_id.astype(str).reset_index(
    drop=True
).equals(
    tr0.encounter_id.astype(str).reset_index(drop=True)
)

assert test.encounter_id.astype(str).reset_index(
    drop=True
).equals(
    te0.encounter_id.astype(str).reset_index(drop=True)
)

assert len(X9) == len(V10_X) == len(tr0)
assert len(T9) == len(V10_T) == len(te0)

y11 = tr0.readmit_30d.to_numpy(dtype=int)
n11 = len(y11)

ids11 = pd.Index(
    pd.concat(
        [tr0.encounter_id, te0.encounter_id],
        ignore_index=True
    ).astype(str)
)

# Keep the complete V9 feature representation
base = pd.concat(
    [X9, T9], ignore_index=True
).copy()

base = base.loc[
    :, ~base.columns.duplicated()
].reset_index(drop=True)

base.columns = [
    "base_" + str(c) for c in base.columns
]

# Add selected clinical information from V10
extra = pd.concat(
    [V10_X, V10_T],
    ignore_index=True
).reset_index(drop=True)

prefixes = (
    "v10_history_",
    "v10_prior_type_",
    "v10_vital_",
    "v10_lab_",
    "v10_cross_",
    "v10_plan_",
    "v10_icu_",
    "v10_transfer_",
    "v10_proc_",
    "v10_patient_",
    "v10_hospital_",
    "v10_dx_icd_chapter_"
)

cols = [
    c for c in extra.columns
    if c.startswith(prefixes)
    and pd.api.types.is_numeric_dtype(extra[c])
]

extra = extra[cols].copy()

extra.columns = [
    "extra_" + str(c) for c in extra.columns
]

print(
    "V11: existing features", base.shape[1],
    "| additional clinical features", extra.shape[1],
    flush=True
)

# Restore encounter-level medication information.
# These are encounter-scoped and discharge-time attributes;
# no post-discharge outcomes are used.
medpaths = list(
    Path("/kaggle/input").rglob("medications.csv")
)

if medpaths:
    m = pd.read_csv(
        medpaths[0],
        usecols=lambda c: c in [
            "encounter_id",
            "drug_class",
            "continued_at_discharge"
        ],
        low_memory=False
    )

    m = m.dropna(subset=["encounter_id"])
    m["encounter_id"] = m.encounter_id.astype(str)

    g = m.groupby("encounter_id")

    medf = g.size().to_frame("v11_med_count")

    if "drug_class" in m:
        cls = (
            m.drug_class.fillna("Missing")
            .astype(str).str.upper().str.strip()
        )

        keep = cls.value_counts().head(25).index

        t = pd.crosstab(
            m.encounter_id,
            cls.where(cls.isin(keep), "OTHER")
        )

        t.columns = [
            "v11_med_" +
            re.sub("[^A-Z0-9]+", "_", str(c))[:32]
            for c in t.columns
        ]

        medf = medf.join(t)

    if "continued_at_discharge" in m:
        continued = (
            m.continued_at_discharge
            .astype(str).str.lower().str.strip()
            .isin(["true", "yes", "y", "1"])
        )

        medf["v11_med_continued"] = (
            continued.groupby(m.encounter_id).sum()
        )

    medf.index = medf.index.astype(str)

    medf = medf.reindex(ids11).reset_index(drop=True)
    medf = medf.add_prefix("extra_")

    extra = pd.concat(
        [extra, medf], axis=1
    )

    print(
        "Restored medication features:",
        medf.shape[1],
        flush=True
    )

    del m, medf

else:
    print("WARNING: medications.csv not found", flush=True)

# Prepare numerical training data
Z = pd.concat([base, extra], axis=1)
Z = Z.loc[:, ~Z.columns.duplicated()]

for c in Z.columns:
    Z[c] = pd.to_numeric(Z[c], errors="coerce")

Z = Z.replace([np.inf, -np.inf], np.nan)

Z = Z.loc[
    :, Z.iloc[:n11].notna().sum() >= 100
]

Z = Z.loc[
    :, Z.iloc[:n11].nunique(dropna=True) > 1
]

Z.columns = [
    re.sub("[^A-Za-z0-9_]", "_", str(c))
    for c in Z.columns
]

Z = Z.loc[
    :, ~Z.columns.duplicated()
].astype("float32")

Ztr = Z.iloc[:n11].reset_index(drop=True)
Zte = Z.iloc[n11:].reset_index(drop=True)

assert Ztr.columns.equals(Zte.columns)

print(
    "V11 TRAINING:", Ztr.shape,
    "TEST:", Zte.shape,
    flush=True
)

# Same folds as V8/V9 for a fair comparison
splits = list(
    StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    ).split(Ztr, y11)
)

o_xgb = np.zeros(n11)
t_xgb = np.zeros(len(Zte))

o_cb = np.zeros(n11)
t_cb = np.zeros(len(Zte))

train_gaps = []

for k, (tr, va) in enumerate(splits, 1):

    print(
        "V11 FOLD", k, "/ 3 XGBOOST",
        flush=True
    )

    xgb = XGBClassifier(
        n_estimators=1250,
        max_depth=3,
        learning_rate=0.035,
        min_child_weight=18,
        subsample=0.82,
        colsample_bytree=0.82,
        reg_lambda=14,
        reg_alpha=1,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        n_jobs=4,
        random_state=400 + k,
        early_stopping_rounds=90
    )

    xgb.fit(
        Ztr.iloc[tr],
        y11[tr],
        eval_set=[(Ztr.iloc[va], y11[va])],
        verbose=False
    )

    o_xgb[va] = xgb.predict_proba(
        Ztr.iloc[va]
    )[:, 1]

    t_xgb += xgb.predict_proba(
        Zte
    )[:, 1] / 3

    train_auc = roc_auc_score(
        y11[tr],
        xgb.predict_proba(Ztr.iloc[tr])[:, 1]
    )

    val_auc = roc_auc_score(
        y11[va], o_xgb[va]
    )

    train_gaps.append((
        "XGBoost", k,
        round(train_auc, 6),
        round(val_auc, 6),
        round(train_auc - val_auc, 6)
    ))

    print(
        "XGBoost validation:", round(val_auc, 6),
        "| overfit gap:",
        round(train_auc - val_auc, 6),
        flush=True
    )

    del xgb
    gc.collect()

    print(
        "V11 FOLD", k, "/ 3 CATBOOST",
        flush=True
    )

    cb = CatBoostClassifier(
        iterations=1050,
        depth=5,
        learning_rate=0.04,
        l2_leaf_reg=12,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=500 + k,
        thread_count=4,
        verbose=False,
        allow_writing_files=False
    )

    cb.fit(
        Ztr.iloc[tr],
        y11[tr],
        eval_set=(Ztr.iloc[va], y11[va]),
        early_stopping_rounds=90,
        verbose=False
    )

    o_cb[va] = cb.predict_proba(
        Ztr.iloc[va]
    )[:, 1]

    t_cb += cb.predict_proba(
        Zte
    )[:, 1] / 3

    train_auc = roc_auc_score(
        y11[tr],
        cb.predict_proba(Ztr.iloc[tr])[:, 1]
    )

    val_auc = roc_auc_score(
        y11[va], o_cb[va]
    )

    train_gaps.append((
        "CatBoost", k,
        round(train_auc, 6),
        round(val_auc, 6),
        round(train_auc - val_auc, 6)
    ))

    print(
        "CatBoost validation:", round(val_auc, 6),
        "| overfit gap:",
        round(train_auc - val_auc, 6),
        flush=True
    )

    del cb
    gc.collect()

# Recreate the known V8 and V9 baselines
v8o = np.asarray(v8_oof)
v8t = np.asarray(v8_test)

v9o = 0.8 * v8o + 0.2 * np.asarray(xgb_oof)
v9t = 0.8 * v8t + 0.2 * np.asarray(xgb_test)

v11o = 0.5 * o_xgb + 0.5 * o_cb
v11t = 0.5 * t_xgb + 0.5 * t_cb

options = {
    "V8_reference": (v8o, v8t),
    "V9_reference": (v9o, v9t),
    "V11_XGBoost": (o_xgb, t_xgb),
    "V11_CatBoost": (o_cb, t_cb),
    "V11_ensemble": (v11o, v11t),

    "V11_V9_80_new20": (
        0.8*v9o + 0.2*v11o,
        0.8*v9t + 0.2*v11t
    ),

    "V11_V9_60_new40": (
        0.6*v9o + 0.4*v11o,
        0.6*v9t + 0.4*v11t
    ),

    "V11_V8_60_new40": (
        0.6*v8o + 0.4*v11o,
        0.6*v8t + 0.4*v11t
    ),

    "V11_V9_80_cb20": (
        0.8*v9o + 0.2*o_cb,
        0.8*v9t + 0.2*t_cb
    ),

    "V11_V9_80_xgb20": (
        0.8*v9o + 0.2*o_xgb,
        0.8*v9t + 0.2*t_xgb
    )
}

ranked11 = sorted(
    [
        (roc_auc_score(y11, p[0]), name)
        for name, p in options.items()
    ],
    reverse=True
)

print("\nV11 TRAIN vs VALIDATION GAPS")

for record in train_gaps:
    print(record)

print("\nV11 THREE-FOLD AUC COMPARISON")

for auc, name in ranked11:
    gains = [
        roc_auc_score(
            y11[va], options[name][0][va]
        )
        -
        roc_auc_score(
            y11[va], v9o[va]
        )
        for _, va in splits
    ]

    print(
        name,
        round(auc, 6),
        "| gains vs V9:",
        [round(g, 6) for g in gains],
        flush=True
    )

bestscore, bestname = ranked11[0]

best_gains = [
    roc_auc_score(
        y11[va], options[bestname][0][va]
    )
    -
    roc_auc_score(
        y11[va], v9o[va]
    )
    for _, va in splits
]

v9score = roc_auc_score(y11, v9o)

# Avoid wasting submissions on tiny improvements
if (
    bestscore <= v9score + 0.0010
    or sum(g > 0 for g in best_gains) < 2
    or bestname in ["V8_reference", "V9_reference"]
):
    print(
        "\nSTOP: No convincing V11 improvement.",
        "Keep V9/V8. Do not submit V11.",
        flush=True
    )

else:
    pred = np.asarray(
        options[bestname][1]
    )

    assert len(pred) == len(te0)
    assert np.isfinite(pred).all()
    assert ((pred >= 0) & (pred <= 1)).all()

    out = pd.DataFrame({
        "encounter_id": te0.encounter_id,
        "readmit_30d": pred
    })

    assert out.encounter_id.is_unique

    path = "/kaggle/working/SUBMIT_V11.csv"

    out.to_csv(path, index=False)

    print(
        "\nV11 CANDIDATE SAVED:",
        bestname,
        "ROC-AUC:", round(bestscore, 6),
        flush=True
    )

    with open(path, "rb") as f:
        encoded = base64.b64encode(
            f.read()
        ).decode()

    display(HTML(
        '<a download="SUBMIT_V11.csv" '
        'href="data:text/csv;base64,'
        + encoded
        + '">DOWNLOAD V11 CSV</a>'
    ))


### 10.2 Generate the three historically selected final submissions

Creates `FINAL_A_V9_XGB.csv`, `FINAL_B_V8_NEW40.csv`, and `FINAL_C_V9_NEW40.csv`. These are the three submission variants selected for the final Kaggle evaluation.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 41.

In [ ]:

import numpy as np
import pandas as pd
import base64
from pathlib import Path
from sklearn.metrics import roc_auc_score
from IPython.display import HTML, display

# Three final candidates from the completed V11 run
final_choices = {
    "FINAL_A_V9_XGB.csv": "V11_V9_80_xgb20",
    "FINAL_B_V8_NEW40.csv": "V11_V8_60_new40",
    "FINAL_C_V9_NEW40.csv": "V11_V9_60_new40"
}

assert "options" in globals(), (
    "Run this in the completed V11 notebook."
)

assert "test" in globals()
assert "train" in globals()

missing = [
    name for name in final_choices.values()
    if name not in options
]

assert not missing, (
    "Missing V11 candidates: " + str(missing)
)

y_final = train["readmit_30d"].to_numpy()
test_ids_final = test["encounter_id"].to_numpy()

print("FINAL SUBMISSION CHECK")
print("Train:", len(y_final))
print("Test:", len(test_ids_final))

for filename, model_name in final_choices.items():

    oof_pred, test_pred = options[model_name]

    oof_pred = np.asarray(oof_pred).reshape(-1)
    test_pred = np.asarray(test_pred).reshape(-1)

    assert len(oof_pred) == len(y_final)
    assert len(test_pred) == len(test_ids_final)
    assert np.isfinite(oof_pred).all()
    assert np.isfinite(test_pred).all()
    assert ((test_pred >= 0) & (test_pred <= 1)).all()

    score = roc_auc_score(y_final, oof_pred)

    output = pd.DataFrame({
        "encounter_id": test_ids_final,
        "readmit_30d": test_pred
    })

    assert output["encounter_id"].is_unique

    path = Path("/kaggle/working") / filename
    output.to_csv(path, index=False)

    print(
        "\nSAVED:", filename,
        "| OOF AUC:", round(score, 6),
        "| ROWS:", len(output),
        flush=True
    )

    with open(path, "rb") as f:
        encoded = base64.b64encode(
            f.read()
        ).decode("ascii")

    display(HTML(
        '<a download="' + filename +
        '" href="data:text/csv;base64,' +
        encoded +
        '">DOWNLOAD ' + filename + '</a>'
    ))

print("\nALL THREE FILES ARE READY.")


### 10.3 Produce one default submission and verify its contract

The final selected model with the **highest public AUC** was `FINAL_B_V8_NEW40.csv`. Save the same predictions under `submission.csv` for the professor's requested format. This cell verifies row order, uniqueness, probability bounds and the absence of missing predictions.

In [ ]:

from sklearn.metrics import roc_auc_score

final_name = 'V11_V8_60_new40'
assert final_name in options
final_oof, final_test_predictions = options[final_name]
final_test_predictions = np.asarray(final_test_predictions, dtype=float).reshape(-1)
final_ids = test['encounter_id'].astype(str).reset_index(drop=True)
assert len(final_test_predictions) == len(final_ids)
assert final_ids.is_unique
assert np.isfinite(final_test_predictions).all()
assert np.all((final_test_predictions >= 0) & (final_test_predictions <= 1))
assert 'readmit_30d' not in Ztr.columns
assert Ztr.columns.equals(Zte.columns)

final_submission = pd.DataFrame({
    'encounter_id': final_ids.to_numpy(),
    'readmit_30d': final_test_predictions
})
final_submission.to_csv(OUTPUT_DIR/'submission.csv', index=False)
print('DEFAULT SUBMISSION:', OUTPUT_DIR/'submission.csv')
print('TRAIN:',len(train),'TEST:',len(test),'FINAL FEATURES:',Ztr.shape[1])
print('FINAL B OOF AUC:',round(roc_auc_score(y11,np.asarray(final_oof)),6))
print('CHECKS: row count, ID order, uniqueness, probability range and target removal passed.')


## 11. V12: Final ensemble and regularization experiments

**Why:** Test whether nonlinear/rank blending, smaller regularized LightGBM models or logistic stacking can improve V11. The recorded experiments did **not** produce a meaningful gain over the V11 baseline (0.748671 OOF ROC-AUC). This is evidence that adding complexity is not automatically better.

**Important stacking limitation:** the historical stacking procedure trained a meta-model on OOF base predictions, but the base-model training folds were not completely independent of the meta-model folds. This is **not a fully nested stacking evaluation**, so its AUC should not be presented as a strict independent generalization estimate.

### 11.1 V12 fast rank/probability ensembles

Compare V8/V9/V11 prediction combinations without retraining.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 42.

In [ ]:

import numpy as np
import pandas as pd
import base64

from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from IPython.display import display, HTML

assert "options" in globals(), "Run in completed V11 notebook."
assert "train" in globals() and "test" in globals()

y12 = train["readmit_30d"].to_numpy(dtype=int)
ids12 = test["encounter_id"].to_numpy()

def v12_get(name):
    assert name in options, "Missing V11 option: " + name
    a, b = options[name]
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    assert len(a) == len(y12)
    assert len(b) == len(ids12)
    assert np.isfinite(a).all() and np.isfinite(b).all()
    return a, b

v9o, v9t = v12_get("V9_reference")
v8o, v8t = v12_get("V8_reference")
v11o, v11t = v12_get("V11_V9_80_xgb20")

def v12_rank(x):
    return rankdata(x, method="average") / len(x)

def v12_mix(parts, ranked=False):
    a = np.zeros(len(y12))
    b = np.zeros(len(ids12))
    for weight, (o, t) in parts:
        if ranked:
            o, t = v12_rank(o), v12_rank(t)
        a += weight * o
        b += weight * t
    return a, b

V12_candidates = {
    "V11_BASELINE": (v11o, v11t),

    "V12_PROB_70V11_30V8": v12_mix([
        (.70, (v11o, v11t)),
        (.30, (v8o, v8t))
    ]),

    "V12_PROB_70V11_30V9": v12_mix([
        (.70, (v11o, v11t)),
        (.30, (v9o, v9t))
    ]),

    "V12_RANK_80V11_20V8": v12_mix([
        (.80, (v11o, v11t)),
        (.20, (v8o, v8t))
    ], ranked=True),

    "V12_RANK_70V11_30V9": v12_mix([
        (.70, (v11o, v11t)),
        (.30, (v9o, v9t))
    ], ranked=True),

    "V12_RANK_50V11_30V9_20V8": v12_mix([
        (.50, (v11o, v11t)),
        (.30, (v9o, v9t)),
        (.20, (v8o, v8t))
    ], ranked=True)
}

V12_results = sorted(
    [
        (roc_auc_score(y12, pred[0]), name)
        for name, pred in V12_candidates.items()
    ],
    reverse=True
)

print("V12 FAST ENSEMBLE COMPARISON")
for auc, name in V12_results:
    print(f"{name:32s} ROC-AUC={auc:.6f}")

v12_fast_auc, v12_fast_name = V12_results[0]
v12_fast_oof, v12_fast_test = V12_candidates[v12_fast_name]

def v12_save(filename, pred):
    pred = np.asarray(pred).reshape(-1)
    assert len(pred) == len(ids12)
    assert np.isfinite(pred).all()
    assert ((pred >= 0) & (pred <= 1)).all()

    out = pd.DataFrame({
        "encounter_id": ids12,
        "readmit_30d": pred
    })
    assert out.encounter_id.is_unique

    path = "/kaggle/working/" + filename
    out.to_csv(path, index=False)

    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode()

    display(HTML(
        '<a download="' + filename +
        '" href="data:text/csv;base64,' + encoded +
        '">DOWNLOAD ' + filename + '</a>'
    ))

    print("SAVED:", path)

v12_save("V12_FAST_BLEND.csv", v12_fast_test)

print("BEST FAST V12:", v12_fast_name, round(v12_fast_auc, 6))
print("TARGET: 0.750000")

if v12_fast_auc >= 0.7500:
    print("TARGET REACHED OFFLINE — verify fold consistency.")
else:
    print("Below target. Continue to V12 Cell 2.")


### 11.2 V12 small and medium regularized LightGBM models

Train two additional LightGBM configurations, compare fold AUC and overfitting gaps, and test weighted ensembles.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 43.

In [ ]:

import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

assert "Ztr" in globals() and "Zte" in globals(), (
    "Run in completed V11 notebook."
)
assert "V12_candidates" in globals(), "Run V12 Cell 1 first."

X12 = pd.DataFrame(Ztr).reset_index(drop=True).copy()
T12 = pd.DataFrame(Zte).reset_index(drop=True).copy()

assert len(X12) == len(y12)
assert len(T12) == len(ids12)
assert X12.shape[1] == T12.shape[1]

for c in X12.columns:
    X12[c] = pd.to_numeric(X12[c], errors="coerce")
    T12[c] = pd.to_numeric(T12[c], errors="coerce")

X12 = X12.replace([np.inf, -np.inf], np.nan)
T12 = T12.replace([np.inf, -np.inf], np.nan)

keep12 = (
    (X12.notna().sum() >= 100)
    & (X12.nunique(dropna=True) > 1)
)

X12 = X12.loc[:, keep12].astype("float32")
T12 = T12.loc[:, keep12].astype("float32")

# LightGBM requires unique feature names
X12.columns = [f"v12f_{i}" for i in range(X12.shape[1])]
T12.columns = X12.columns

print("V12 TRAIN:", X12.shape, "TEST:", T12.shape, flush=True)

folds12 = list(
    StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    ).split(X12, y12)
)

v12_oof = {}
v12_test = {}
v12_report = []

configs = {
    "V12_SMALL": dict(
        num_leaves=9,
        min_child_samples=110,
        reg_lambda=18,
        reg_alpha=1.0,
        max_depth=5
    ),
    "V12_MEDIUM": dict(
        num_leaves=17,
        min_child_samples=85,
        reg_lambda=14,
        reg_alpha=0.8,
        max_depth=-1
    )
}

for name, params in configs.items():

    v12_oof[name] = np.zeros(len(y12))
    v12_test[name] = np.zeros(len(ids12))

    for k, (tr, va) in enumerate(folds12, 1):

        print(name, "FOLD", k, "/3", flush=True)

        model = LGBMClassifier(
            n_estimators=850,
            learning_rate=0.035,
            objective="binary",
            colsample_bytree=0.85,
            subsample=0.85,
            subsample_freq=1,
            max_bin=127,
            n_jobs=4,
            random_state=1200 + k,
            verbosity=-1,
            **params
        )

        model.fit(
            X12.iloc[tr],
            y12[tr],
            eval_set=[(X12.iloc[va], y12[va])],
            eval_metric="auc",
            callbacks=[
                early_stopping(70, verbose=False),
                log_evaluation(0)
            ]
        )

        vp = model.predict_proba(X12.iloc[va])[:, 1]
        tp = model.predict_proba(T12)[:, 1]

        v12_oof[name][va] = vp
        v12_test[name] += tp / 3

        train_auc = roc_auc_score(
            y12[tr],
            model.predict_proba(X12.iloc[tr])[:, 1]
        )

        val_auc = roc_auc_score(y12[va], vp)

        v12_report.append((
            name, k, train_auc, val_auc,
            train_auc - val_auc
        ))

        print(
            "VALIDATION AUC:", round(val_auc, 6),
            "OVERFIT GAP:", round(train_auc - val_auc, 6),
            flush=True
        )

        del model
        gc.collect()

# New LightGBM ensemble
new_oof = (
    0.5 * v12_oof["V12_SMALL"]
    + 0.5 * v12_oof["V12_MEDIUM"]
)

new_test = (
    0.5 * v12_test["V12_SMALL"]
    + 0.5 * v12_test["V12_MEDIUM"]
)

# Compare fixed ensemble weights against V11
final12 = dict(V12_candidates)

final12.update({
    "V12_SMALL_ONLY": (
        v12_oof["V12_SMALL"],
        v12_test["V12_SMALL"]
    ),

    "V12_MEDIUM_ONLY": (
        v12_oof["V12_MEDIUM"],
        v12_test["V12_MEDIUM"]
    ),

    "V12_LGB_ENSEMBLE": (new_oof, new_test),

    "V12_V11_85_LGB15": v12_mix([
        (.85, (v11o, v11t)),
        (.15, (new_oof, new_test))
    ]),

    "V12_V11_70_LGB30": v12_mix([
        (.70, (v11o, v11t)),
        (.30, (new_oof, new_test))
    ]),

    "V12_V9_60_V11_25_LGB15": v12_mix([
        (.60, (v9o, v9t)),
        (.25, (v11o, v11t)),
        (.15, (new_oof, new_test))
    ]),

    "V12_RANK_V11_80_LGB20": v12_mix([
        (.80, (v11o, v11t)),
        (.20, (new_oof, new_test))
    ], ranked=True)
})

ranked12 = sorted(
    [
        (roc_auc_score(y12, p[0]), name)
        for name, p in final12.items()
    ],
    reverse=True
)

print("\nFINAL V12 THREE-FOLD ROC-AUC COMPARISON")

for auc, name in ranked12:
    print(f"{name:34s} {auc:.6f}")

best_auc12, best_name12 = ranked12[0]
best_oof12, best_test12 = final12[best_name12]

print("\nFOLD GENERALIZATION CHECK")

for k, (tr, va) in enumerate(folds12, 1):
    new_auc = roc_auc_score(y12[va], best_oof12[va])
    old_auc = roc_auc_score(y12[va], v11o[va])

    print(
        f"Fold {k}: V12={new_auc:.6f}",
        f"V11={old_auc:.6f}",
        f"GAIN={new_auc-old_auc:+.6f}"
    )

print("\nV12 TRAINING GAPS")

print(
    pd.DataFrame(
        v12_report,
        columns=[
            "model", "fold",
            "train_auc", "validation_auc",
            "overfit_gap"
        ]
    ).round(5).to_string(index=False)
)

v12_save("V12_FINAL.csv", best_test12)

print("\nV12 WINNER:", best_name12)
print("V12 OFFLINE ROC-AUC:", round(best_auc12, 6))
print("V11 BEST OFFLINE ROC-AUC: 0.748671")

if best_auc12 >= 0.7500:
    print("TARGET 0.7500 REACHED OFFLINE.")
elif best_auc12 > 0.748671:
    print("IMPROVED OVER V11, BUT BELOW 0.7500.")
else:
    print("NO IMPROVEMENT. KEEP EXISTING V11/V9.")


### 11.3 V12 logistic-regression stacking

Use OOF model probabilities as meta-features, with a documented caveat about fully nested validation.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 44.

In [ ]:

import numpy as np
import pandas as pd
import base64

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from IPython.display import HTML, display

assert "options" in globals()
assert "v12_oof" in globals()
assert "v12_test" in globals()
assert "y12" in globals()
assert "ids12" in globals()

# Collect different models already trained
sources = {}

for name in [
    "V9_reference",
    "V8_reference",
    "V11_XGBoost",
    "V11_CatBoost",
    "V11_ensemble"
]:
    if name in options:
        a, b = options[name]
        sources[name] = (
            np.asarray(a).reshape(-1),
            np.asarray(b).reshape(-1)
        )

for name in ["V12_SMALL", "V12_MEDIUM"]:
    if name in v12_oof and name in v12_test:
        sources[name] = (
            np.asarray(v12_oof[name]).reshape(-1),
            np.asarray(v12_test[name]).reshape(-1)
        )

assert len(sources) >= 3, (
    "Not enough existing models for stacking."
)

# Use the best validated V11 model as baseline
base_oof, base_test = options[
    "V11_V9_80_xgb20"
]

base_oof = np.asarray(base_oof).reshape(-1)
base_test = np.asarray(base_test).reshape(-1)

print("STACKING MODELS:", list(sources))

def logit12(x):
    x = np.clip(np.asarray(x), 1e-5, 1-1e-5)
    return np.log(x / (1-x))

# Base-model probability scores become meta-features
stack_X = np.column_stack([
    logit12(p[0]) for p in sources.values()
])

stack_T = np.column_stack([
    logit12(p[1]) for p in sources.values()
])

assert stack_X.shape[0] == len(y12)
assert stack_T.shape[0] == len(ids12)
assert np.isfinite(stack_X).all()
assert np.isfinite(stack_T).all()

folds = list(
    StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    ).split(stack_X, y12)
)

stack_oof = np.zeros(len(y12))
stack_test = np.zeros(len(ids12))

for k, (tr, va) in enumerate(folds, 1):

    scaler = StandardScaler()

    Xtr = scaler.fit_transform(stack_X[tr])
    Xva = scaler.transform(stack_X[va])
    Xt = scaler.transform(stack_T)

    # Strong regularization limits meta-model overfitting
    meta = LogisticRegression(
        C=0.05,
        max_iter=300,
        solver="lbfgs",
        random_state=42
    )

    meta.fit(Xtr, y12[tr])

    stack_oof[va] = meta.predict_proba(Xva)[:, 1]
    stack_test += meta.predict_proba(Xt)[:, 1] / 3

    print(
        "STACK FOLD", k,
        "AUC:",
        round(
            roc_auc_score(y12[va], stack_oof[va]),
            6
        ),
        flush=True
    )

# Evaluate different weights
candidates = {
    "V11_BASELINE": (
        base_oof,
        base_test
    ),
    "V12_STACK_ONLY": (
        stack_oof,
        stack_test
    ),
    "V12_STACK_25": (
        .75*base_oof + .25*stack_oof,
        .75*base_test + .25*stack_test
    ),
    "V12_STACK_50": (
        .50*base_oof + .50*stack_oof,
        .50*base_test + .50*stack_test
    ),
    "V12_STACK_75": (
        .25*base_oof + .75*stack_oof,
        .25*base_test + .75*stack_test
    )
}

results = sorted(
    [
        (roc_auc_score(y12, p[0]), name)
        for name, p in candidates.items()
    ],
    reverse=True
)

print("\nFINAL V12 STACKING COMPARISON")

for auc, name in results:
    print(f"{name:24s} {auc:.6f}")

# Find the best NEW candidate
challengers = [
    (auc, name)
    for auc, name in results
    if name != "V11_BASELINE"
]

challenger_auc, challenger_name = challengers[0]
challenger_oof, challenger_test = candidates[
    challenger_name
]

print("\nFOLD-BY-FOLD COMPARISON")

gains = []

for k, (tr, va) in enumerate(folds, 1):
    old_auc = roc_auc_score(y12[va], base_oof[va])
    new_auc = roc_auc_score(y12[va], challenger_oof[va])
    gain = new_auc - old_auc
    gains.append(gain)

    print(
        f"Fold {k}: "
        f"V11={old_auc:.6f} "
        f"V12={new_auc:.6f} "
        f"gain={gain:+.6f}"
    )

print("\nBEST NEW CHALLENGER:", challenger_name)
print("CHALLENGER AUC:", round(challenger_auc, 6))
print("V11 BASELINE:", round(
    roc_auc_score(y12, base_oof), 6
))

# Save challenger, without overwriting V11/V12 files
filename = "V12_STACK_CHALLENGER.csv"

out = pd.DataFrame({
    "encounter_id": ids12,
    "readmit_30d": challenger_test
})

assert len(out) == len(ids12)
assert out.encounter_id.is_unique
assert np.isfinite(out.readmit_30d).all()
assert out.readmit_30d.between(0, 1).all()

path = "/kaggle/working/" + filename
out.to_csv(path, index=False)

with open(path, "rb") as f:
    encoded = base64.b64encode(f.read()).decode()

display(HTML(
    '<a download="' + filename +
    '" href="data:text/csv;base64,' + encoded +
    '">DOWNLOAD ' + filename + '</a>'
))

if (
    challenger_auc > 0.748671
    and sum(g > 0 for g in gains) >= 2
):
    print("\nPOTENTIAL IMPROVEMENT — review before submitting.")
else:
    print("\nNO CONVINCING IMPROVEMENT — retain V11/V9.")

if challenger_auc >= 0.7600:
    print("0.7600 TARGET REACHED OFFLINE.")
else:
    print("0.7600 TARGET NOT REACHED.")


### 11.4 V12 nonlinear log-odds and geometric ensembles

Evaluate additional prediction-level blends. Tiny gains should not be treated as statistically meaningful.

**Source:** `hospital-readmission-master-v2.ipynb`, original code cell 45.

In [ ]:

import numpy as np
import pandas as pd
import base64
from scipy.special import expit, logit
from sklearn.metrics import roc_auc_score
from IPython.display import HTML, display

assert "options" in globals()
assert "y12" in globals()
assert "ids12" in globals()

names = {
    "base": "V11_V9_80_xgb20",
    "v9": "V9_reference",
    "v8": "V8_reference",
    "xgb": "V11_XGBoost",
    "cat": "V11_CatBoost"
}

assert all(n in options for n in names.values()), (
    "Missing predictions in V11 notebook."
)

P = {}
for short, name in names.items():
    a, b = options[name]
    P[short] = (
        np.asarray(a, dtype=float).ravel(),
        np.asarray(b, dtype=float).ravel()
    )

def nonlinear_blend(weights, method="logit"):
    train_p = np.zeros(len(y12))
    test_p = np.zeros(len(ids12))

    for key, weight in weights.items():
        a, b = P[key]
        a = np.clip(a, 1e-6, 1 - 1e-6)
        b = np.clip(b, 1e-6, 1 - 1e-6)

        if method == "logit":
            train_p += weight * logit(a)
            test_p += weight * logit(b)
        else:
            train_p += weight * np.log(a)
            test_p += weight * np.log(b)

    if method == "logit":
        return expit(train_p), expit(test_p)

    return np.exp(train_p), np.exp(test_p)

candidates = {
    "V12_LOGIT_BASE_CAT": nonlinear_blend(
        {"base": .75, "cat": .25}
    ),
    "V12_LOGIT_BASE_XGB": nonlinear_blend(
        {"base": .75, "xgb": .25}
    ),
    "V12_LOGIT_BASE_V9": nonlinear_blend(
        {"base": .75, "v9": .25}
    ),
    "V12_LOGIT_BASE_V8": nonlinear_blend(
        {"base": .75, "v8": .25}
    ),
    "V12_LOGIT_THREE_MODELS": nonlinear_blend(
        {"base": .60, "cat": .20, "xgb": .20}
    ),
    "V12_GEOMETRIC_BASE_CAT": nonlinear_blend(
        {"base": .80, "cat": .20},
        method="geometric"
    ),
    "V12_GEOMETRIC_BASE_XGB": nonlinear_blend(
        {"base": .80, "xgb": .20},
        method="geometric"
    )
}

baseline_auc = roc_auc_score(
    y12, P["base"][0]
)

results = sorted(
    [
        (roc_auc_score(y12, pair[0]), name)
        for name, pair in candidates.items()
    ],
    reverse=True
)

print("\nV12 LAST-CHANCE RESULTS")
print("BASELINE:", round(baseline_auc, 6))

for auc, name in results:
    print(
        name,
        "AUC:", round(auc, 6),
        "GAIN:", round(auc - baseline_auc, 6)
    )

best_auc, best_name = results[0]

if best_auc > baseline_auc:
    print("\nNEW OFFLINE WINNER:", best_name)

    predictions = candidates[best_name][1]

    output = pd.DataFrame({
        "encounter_id": ids12,
        "readmit_30d": predictions
    })

    assert output.encounter_id.is_unique
    assert np.isfinite(predictions).all()
    assert np.all(
        (predictions >= 0) & (predictions <= 1)
    )

    filename = "V12_LAST_CHALLENGER.csv"
    path = "/kaggle/working/" + filename
    output.to_csv(path, index=False)

    with open(path, "rb") as f:
        encoded = base64.b64encode(
            f.read()
        ).decode()

    display(HTML(
        '<a download="' + filename +
        '" href="data:text/csv;base64,' +
        encoded +
        '">DOWNLOAD ' + filename + '</a>'
    ))

else:
    print(
        "\nNO IMPROVEMENT. "
        "DO NOT SPEND A SUBMISSION ON V12."
    )

print(
    "\nTARGET 0.7600:",
    "REACHED" if best_auc >= .7600
    else "NOT REACHED"
)


## 12. Report-ready exports and final checks

The report requires the final feature count and **every column with its source and transformation**. The feature manifest below lists every actual V11 column. Because some original feature names were generated automatically or from one-hot categories, its source descriptions are **programmatic interpretations**, not an independently verified exact lineage. Review `needs_manual_review=True` rows against the original feature-engineering cells before submitting the report.

The raw data-quality audit records per-table missingness and duplicated full rows (within chunks). It is a **screening audit**, not proof that every unit or timestamp is correct. Additional focused checks cover invalid lengths of stay and missing feature values.

In [ ]:

import gc
from pathlib import Path
import pandas as pd
import numpy as np

# Source lineage: follow the nested V11 -> V9 -> V8/V4 and V10 prefixes.
def describe_feature(feature):
    name = str(feature).lower()
    if 'prior' in name or 'history' in name or 'visits_' in name:
        src='prior_utilization.csv + index_encounters.csv'; how='time-filtered visit counts, recency or history statistics'
    elif 'lab_' in name or 'bun_' in name or 'creatinine_' in name or 'glucose_' in name:
        src='lab_results.csv + index_encounters.csv'; how='unit-checked lab counts, abnormalities, summary or trends'
    elif 'vital_' in name or 'shock_index' in name:
        src='vitals.csv + index_encounters.csv'; how='physiology-checked vital summary, latest value or clinical ratio'
    elif 'med_' in name or 'drug_' in name:
        src='medications.csv'; how='medication count, drug class or continued-at-discharge indicator'
    elif 'dx_' in name or 'diagnos' in name or 'icd_' in name:
        src='diagnoses.csv + icd10_lookup.csv'; how='diagnosis count, code/category frequency or comorbidity indicator'
    elif 'transfer' in name or 'icu_' in name or 'unit_' in name:
        src='unit_transfers.csv'; how='unit transfer, ICU exposure or duration summary'
    elif 'proc_' in name or 'procedure' in name:
        src='procedures.csv'; how='procedure count or category summary'
    elif 'plan_' in name or 'discharge_' in name or 'home_health' in name:
        src='discharge_planning.csv + index_encounters.csv'; how='discharge planning information or discharge-time feature'
    elif 'hospital_' in name or 'ownership' in name or 'service_line' in name:
        src='hospitals.csv + index_encounters.csv'; how='hospital/encounter characteristic or one-hot indicator'
    elif 'patient_' in name or 'age' in name or 'sex_' in name or 'insurance_' in name:
        src='patients.csv + index_encounters.csv'; how='demographic or encounter-derived characteristic'
    elif 'cross_' in name or 'ratio_' in name or 'change_' in name:
        src='multiple clinical tables'; how='cross-table interaction or change/ratio'
    else:
        src='index_encounters.csv / patients.csv / derived features'; how='encounter-level attribute or encoded indicator'
    return src, how

rows=[]
for i, feature in enumerate(Ztr.columns):
    source, method=describe_feature(feature)
    rows.append({
        'feature_number':i+1, 'final_feature':feature,
        'source_table_inferred':source,
        'transformation_inferred':method,
        'train_missing_fraction':float(Ztr[feature].isna().mean()),
        'train_unique_values':int(Ztr[feature].nunique(dropna=True)),
        'needs_manual_review':bool(source.startswith('index_encounters.csv /') or feature.startswith('base_'))
    })
feature_manifest=pd.DataFrame(rows)
feature_manifest.to_csv(OUTPUT_DIR/'FINAL_FEATURE_COLUMNS.csv',index=False)

# Output-ready model comparison (OOF scores are recomputed, not copied from screenshots).
model_summary=pd.DataFrame([
    {'model':name,'oof_roc_auc':roc_auc_score(y11,np.asarray(pred[0]))}
    for name,pred in options.items()
]).sort_values('oof_roc_auc',ascending=False)
model_summary.to_csv(OUTPUT_DIR/'FINAL_MODEL_COMPARISON.csv',index=False)

if 'STRESS_RESULTS' in globals() and STRESS_RESULTS:
    pd.DataFrame(STRESS_RESULTS).to_csv(
        OUTPUT_DIR/'GENERALIZATION_STRESS_TESTS.csv',index=False)

quality=pd.DataFrame({
    'feature':Ztr.columns,
    'missing_count':Ztr.isna().sum().to_numpy(),
    'missing_fraction':Ztr.isna().mean().to_numpy()
}).sort_values('missing_fraction',ascending=False)
quality.to_csv(OUTPUT_DIR/'FINAL_FEATURE_MISSINGNESS.csv',index=False)

assert Ztr.columns.is_unique and Ztr.columns.equals(Zte.columns)
assert len(feature_manifest)==Ztr.shape[1]
assert final_submission.encounter_id.astype(str).tolist()==test.encounter_id.astype(str).tolist()
assert set(final_submission.columns)=={'encounter_id','readmit_30d'}

print('FEATURE MANIFEST:',len(feature_manifest),'rows')
print('FEATURE COUNT (historical original: 855):',Ztr.shape[1])
print('TOP MODEL COMPARISON:')
print(model_summary.head(10).round(6).to_string(index=False))
print('OUTPUT FILES:')
for name in ['submission.csv','FINAL_A_V9_XGB.csv','FINAL_B_V8_NEW40.csv',
             'FINAL_C_V9_NEW40.csv','FINAL_FEATURE_COLUMNS.csv',
             'FINAL_MODEL_COMPARISON.csv','FINAL_FEATURE_MISSINGNESS.csv',
             'V10_DATA_AUDIT.csv']:
    p=OUTPUT_DIR/name
    print(name, 'OK' if p.exists() else 'MISSING')


### 12.1 Optional full raw-data-quality audit

This final audit rereads the provided CSVs in bounded chunks, so it does not require all raw event tables to be in memory simultaneously. It records missing cells, table size, and duplicate rows **within each chunk** (cross-chunk duplicates require a separate exact audit). It does not fit any encoder or use the test labels.

In [ ]:

import gc
raw_quality=[]
for name in required_files:
    p=file_locations[name]
    nrows=0; ncols=None; missing=0; dup_within=0
    for chunk in pd.read_csv(p, chunksize=150_000, low_memory=False):
        nrows+=len(chunk)
        ncols=len(chunk.columns)
        missing+=int(chunk.isna().sum().sum())
        dup_within+=int(chunk.duplicated().sum())
        del chunk
    raw_quality.append({
        'table':name,'rows':nrows,'columns':ncols,
        'missing_cells':missing,'duplicates_within_chunks_lower_bound':dup_within,
        'file_bytes':p.stat().st_size
    })
    print(name, nrows, 'rows', flush=True)
raw_quality_df=pd.DataFrame(raw_quality)
raw_quality_df.to_csv(OUTPUT_DIR/'RAW_DATA_QUALITY_AUDIT.csv',index=False)
print('RAW QUALITY AUDIT SAVED:',OUTPUT_DIR/'RAW_DATA_QUALITY_AUDIT.csv')


### 12.2 Visualize model comparisons and generalization

These figures are generated from the actual in-session OOF results and stress-test records, not from manually entered leaderboard scores. Private Kaggle scores must be reported separately because they cannot be calculated from public train labels.

In [ ]:

import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(9,4.8))
shown=model_summary.head(9).sort_values('oof_roc_auc')
ax.barh(shown['model'],shown['oof_roc_auc'])
ax.set_xlabel('Three-fold out-of-fold ROC-AUC')
ax.set_title('V11 candidate model and ensemble comparison')
ax.set_xlim(max(0.68,float(shown.oof_roc_auc.min())-0.012),
             min(0.81,float(shown.oof_roc_auc.max())+0.008))
ax.grid(axis='x',alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR/'MODEL_COMPARISON.png',dpi=160)
plt.show()

if 'STRESS_RESULTS' in globals() and STRESS_RESULTS:
    st=pd.DataFrame(STRESS_RESULTS)
    fig,ax=plt.subplots(figsize=(8,4))
    x=np.arange(len(st))
    ax.bar(x-0.17,st.train_auc,width=0.34,label='Training')
    ax.bar(x+0.17,st.holdout_auc,width=0.34,label='Holdout')
    ax.set_xticks(x,st.test,rotation=12,ha='right')
    ax.set_ylabel('ROC-AUC')
    ax.set_title('Temporal and unseen-hospital generalization checks')
    ax.legend(); ax.grid(axis='y',alpha=0.2)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR/'GENERALIZATION_STRESS.png',dpi=160)
    plt.show()


## 13. Historical results and interpretation (not recomputed in this Markdown)

The following results were **observed in the uploaded notebooks and conversation**. They are historical facts, not guarantees about a new Kaggle run or another hospital population.

| Experiment | Recorded three-fold OOF ROC-AUC | Interpretation |
|---|---:|---|
| V5 CatBoost/XGBoost blend | 0.746144 | Early strong baseline; category handling + boosting |
| V8 blend | 0.748035 | Better longitudinal and clinical features |
| V9 best blend | 0.748383 | XGBoost added a small complementary signal |
| V10 comprehensive best | 0.746482 | More features did **not** improve generalization |
| V11 best ensemble | 0.748671 | Best substantial historical OOF result |
| V12 geometric variation | 0.748678 | +0.000007 vs V11; too small to claim a real gain |
| NOVA best | 0.745043 | Separate comprehensive dataset and classroom-algorithm experiment |
| NOVA-BOOST best | 0.745283 | Added clinical features gave only a marginal improvement |

**Final Kaggle results:** 11 submissions; public ROC-AUC **0.74263** for the best public submission (`FINAL_B`); final private ROC-AUC **0.75356**; final private rank **205**. The private leaderboard used approximately 70% of the test encounters. Public and private scores are different partitions, so their difference is not a formal confidence interval or proof of future deployment performance.

**Model-selection caveat:** choosing the best among many OOF blends on the same folds can produce optimistic estimates. Bootstrap intervals computed after this selection do not fully correct selection bias. A stronger future design would use nested cross-validation or a genuinely untouched temporal/hospital holdout for final model selection.

# Appendix A — Complete original master notebook source (archived, not re-executed)

Every source code cell below is preserved from `hospital-readmission-master-v2.ipynb` in its original order, including superseded variants, diagnostics and abandoned attempts. They are placed under `if False:` so **Restart & Run All** executes only the curated dependency-correct path above. The original code remains visible and can be copied into a separate experiment notebook if a particular historical version needs rerunning.

**Version gaps:** no explicitly labeled V0, V7 or V13 implementation appears in the two supplied notebooks. It would be misleading to fabricate those versions as if they were completed experiments.

### Archive M00 — Kaggle starter environment / file listing

**Original source cell 0; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution
    # This Python 3 environment comes with many helpful analytics libraries installed
    # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
    # For example, here's several helpful packages to load

    import numpy as np # linear algebra
    import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

    # Input data files are available in the read-only "../input/" directory
    # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

    import os
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

    # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
    # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

    # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
    # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

    import kagglehub
    # kagglehub.dataset_download('<owner>/<dataset-slug>')


### Archive M01 — Raw-file audit

**Original source cell 1; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    from pathlib import Path
    import pandas as pd
    import csv

    # Find the competition's CSV files
    files = {}
    for path in Path("/kaggle/input").rglob("*.csv"):
        files[path.name.lower()] = path

    expected = [
        "train.csv", "test.csv", "sample_submission.csv",
        "index_encounters.csv", "patients.csv",
        "hospitals.csv", "diagnoses.csv",
        "procedures.csv", "lab_results.csv",
        "vitals.csv", "medications.csv",
        "unit_transfers.csv", "prior_utilization.csv",
        "discharge_planning.csv", "icd10_lookup.csv",
        "data_dictionary.csv"
    ]

    print("DATASET AUDIT")
    print("=" * 55)

    for name in expected:
        if name not in files:
            print("MISSING:", name)
            continue

        path = files[name]

        # Count CSV records without loading the full file
        with open(path, newline="", encoding="utf-8-sig") as f:
            reader = csv.reader(f)
            header = next(reader)
            rows = sum(1 for row in reader)

        print(f"{name}: {rows:,} rows")
        print("  Columns:", header)

    print("\nTRAIN / TEST CHECK")
    print("=" * 55)

    train = pd.read_csv(files["train.csv"])
    test = pd.read_csv(files["test.csv"])
    enc = pd.read_csv(files["index_encounters.csv"])
    patients = pd.read_csv(files["patients.csv"])

    print("Train:", train.shape)
    print("Test:", test.shape)
    print("Encounters:", enc.shape)
    print("Patients:", patients.shape)

    print("\nReadmission rate:")
    print(train["readmit_30d"].value_counts(normalize=True))

    print("\nDuplicate encounter IDs:")
    print("Train:", train.encounter_id.duplicated().sum())
    print("Test:", test.encounter_id.duplicated().sum())
    print("Index:", enc.encounter_id.duplicated().sum())

    print("\nTrain/test ID overlap:",
          len(set(train.encounter_id) & set(test.encounter_id)))

    print("Train IDs missing from index:",
          (~train.encounter_id.isin(enc.encounter_id)).sum())

    print("Test IDs missing from index:",
          (~test.encounter_id.isin(enc.encounter_id)).sum())

    print("\nAUDIT COMPLETE")


### Archive M02 — V1 encounter master

**Original source cell 2; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import pandas as pd
    import numpy as np
    from pathlib import Path

    # Find the competition data files
    files = {
        p.name.lower(): p
        for p in Path("/kaggle/input").rglob("*.csv")
    }

    def load(name):
        return pd.read_csv(files[name])

    # 1. Load the main tables
    train = load("train.csv")
    test = load("test.csv")
    enc = load("index_encounters.csv")
    patients = load("patients.csv")
    hospitals = load("hospitals.csv")

    # 2. Start with all encounters
    master = enc.copy()

    # 3. Add patient and hospital information
    master = master.merge(
        patients, on="patient_id", how="left",
        validate="one_to_one"
    )

    master = master.merge(
        hospitals, on="hospital_id", how="left",
        validate="many_to_one"
    )

    # 4. Convert dates into real date values
    master["admit_dt"] = pd.to_datetime(
        master["admit_dt"], errors="coerce"
    )

    master["discharge_dt"] = pd.to_datetime(
        master["discharge_dt"], errors="coerce"
    )

    # 5. Calculate age and length of stay
    master["age"] = (
        master["admit_dt"].dt.year -
        master["birth_year"]
    )

    master["stay_days"] = (
        master["discharge_dt"] -
        master["admit_dt"]
    ).dt.total_seconds() / 86400

    # Keep missing values for impossible dates
    master.loc[
        (master["stay_days"] < 0) |
        (master["stay_days"] > 365),
        "stay_days"
    ] = np.nan

    master.loc[
        (master["age"] < 18) |
        (master["age"] > 110),
        "age"
    ] = np.nan

    # 6. Add historical visit counts
    prior = load("prior_utilization.csv")
    prior["visit_dt"] = pd.to_datetime(
        prior["visit_dt"], errors="coerce"
    )

    prior = prior.merge(
        master[["patient_id", "admit_dt"]],
        on="patient_id",
        how="inner",
        validate="many_to_one"
    )

    # Only count visits before current admission
    prior = prior[
        prior["visit_dt"] < prior["admit_dt"]
    ].copy()

    prior["days_before_admit"] = (
        prior["admit_dt"] - prior["visit_dt"]
    ).dt.total_seconds() / 86400

    prior["within_90d"] = (
        prior["days_before_admit"] <= 90
    ).astype(int)

    visit_counts = prior.groupby("patient_id").agg(
        previous_visits=("visit_dt", "count"),
        visits_90d=("within_90d", "sum"),
        days_since_last_visit=("days_before_admit", "min"),
        hospitals_visited=("hospital_id", "nunique")
    ).reset_index()

    master = master.merge(
        visit_counts, on="patient_id",
        how="left", validate="one_to_one"
    )

    master["previous_visits"] = (
        master["previous_visits"].fillna(0)
    )

    master["visits_90d"] = (
        master["visits_90d"].fillna(0)
    )

    master["hospitals_visited"] = (
        master["hospitals_visited"].fillna(0)
    )

    # 7. Count diagnoses per encounter
    diag = load("diagnoses.csv")

    diag_counts = diag.groupby("encounter_id").agg(
        diagnosis_count=("icd10_code", "count"),
        unique_diagnoses=("icd10_code", "nunique")
    ).reset_index()

    master = master.merge(
        diag_counts, on="encounter_id",
        how="left", validate="one_to_one"
    )

    for col in ["diagnosis_count", "unique_diagnoses"]:
        master[col] = master[col].fillna(0)

    # 8. Add discharge-planning information
    discharge = load("discharge_planning.csv")

    master = master.merge(
        discharge, on="encounter_id",
        how="left", validate="one_to_one"
    )

    # 9. Create simple date features
    master["admit_month"] = master["admit_dt"].dt.month
    master["discharge_weekday"] = (
        master["discharge_dt"].dt.dayofweek
    )

    # 10. Remove identifiers and raw date columns
    drop_cols = [
        "encounter_id", "patient_id",
        "attending_id", "admit_dt",
        "discharge_dt", "birth_year"
    ]

    features = master.drop(columns=drop_cols)

    # 11. Standardize categorical text
    cat_cols = features.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()

    for col in cat_cols:
        features[col] = (
            features[col]
            .fillna("Missing")
            .astype(str)
            .str.strip()
            .str.lower()
        )

    # 12. Separate training and test data
    train_index = master["encounter_id"].isin(
        train["encounter_id"]
    )

    test_index = master["encounter_id"].isin(
        test["encounter_id"]
    )

    X_train = features.loc[train_index].copy()
    X_test = features.loc[test_index].copy()

    # Put rows in the same order as train/test
    train_order = master.loc[
        train_index, "encounter_id"
    ]

    test_order = master.loc[
        test_index, "encounter_id"
    ]

    X_train.index = train_order
    X_test.index = test_order

    X_train = X_train.loc[
        train["encounter_id"]
    ].reset_index(drop=True)

    X_test = X_test.loc[
        test["encounter_id"]
    ].reset_index(drop=True)

    y_train = train["readmit_30d"]

    # 13. Final checks
    assert len(X_train) == len(train)
    assert len(X_test) == len(test)
    assert X_train.columns.equals(X_test.columns)

    print("MASTER TABLE CREATED SUCCESSFULLY")
    print("Training shape:", X_train.shape)
    print("Test shape:", X_test.shape)
    print("Categorical features:", len(cat_cols))
    print("Total features:", X_train.shape[1])
    print("Missing numeric values:",
          X_train.select_dtypes(include="number")
          .isna().sum().sum())


### Archive M03 — V1 initial CatBoost holdout

**Original source cell 3; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score
    from catboost import CatBoostClassifier

    # Split the training data into two groups
    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.20,
        random_state=42,
        stratify=y_train
    )

    print("Training patients:", len(X_fit))
    print("Validation patients:", len(X_val))

    # Create the CatBoost model
    model = CatBoostClassifier(
        iterations=700,
        depth=6,
        learning_rate=0.045,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        thread_count=-1,
        allow_writing_files=False
    )

    # Train the model
    model.fit(
        X_fit,
        y_fit,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        early_stopping_rounds=80
    )

    # Predict probabilities for validation patients
    val_predictions = model.predict_proba(X_val)[:, 1]

    # Calculate ROC-AUC
    auc = roc_auc_score(y_val, val_predictions)

    print("\n==========================")
    print("VALIDATION ROC-AUC:", round(auc, 5))
    print("BEST ITERATION:", model.best_iteration_)
    print("==========================")


### Archive M04 — V1 full-data prediction

**Original source cell 4; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # Train the final model on ALL labeled patients
    best_rounds = max(100, model.best_iteration_ + 1)

    final_model = CatBoostClassifier(
        iterations=best_rounds,
        depth=6,
        learning_rate=0.045,
        loss_function="Logloss",
        random_seed=42,
        verbose=100,
        thread_count=-1,
        allow_writing_files=False
    )

    final_model.fit(
        X_train,
        y_train,
        cat_features=cat_cols
    )

    # Predict probabilities for Kaggle test patients
    test_predictions = final_model.predict_proba(X_test)[:, 1]

    # Create the submission file
    submission = pd.DataFrame({
        "encounter_id": test["encounter_id"],
        "readmit_30d": test_predictions
    })

    # Check the output
    assert len(submission) == len(test)
    assert submission["encounter_id"].equals(test["encounter_id"])
    assert submission["readmit_30d"].between(0, 1).all()
    assert submission["readmit_30d"].notna().all()

    # Save the file
    submission.to_csv(
        "/kaggle/working/submission_v1.csv",
        index=False
    )

    print("SUBMISSION FILE CREATED!")
    print("Rows:", len(submission))
    print(submission.head())
    print("\nFile: /kaggle/working/submission_v1.csv")


### Archive M05 — Baseline history inspection

**Original source cell 5; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # STEP 1: Save original baseline results

    v1_validation_auc = 0.72946
    v1_public_auc = 0.71646

    # the types of previous hospital visits
    prior_check = load("prior_utilization.csv")

    print("PREVIOUS VISIT TYPES:")
    print(prior_check["visit_type"].value_counts(dropna=False))

    # diagnosis categories
    lookup_check = load("icd10_lookup.csv")

    print("\nDIAGNOSIS CHAPTERS:")
    print(lookup_check["chapter"].value_counts(dropna=False))

    # Confirm original model features are available
    print("\nV1 FEATURES:", X_train.shape[1])
    print("V1 TRAINING ROWS:", len(X_train))
    print("V1 TEST ROWS:", len(X_test))


### Archive M06 — V2 engineered clinical features

**Original source cell 6; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # additional V2 features

    import pandas as pd
    import numpy as np

    extra = master[["encounter_id", "patient_id"]].copy()

    # A. PREVIOUS HOSPITAL VISITS

    prior2 = load("prior_utilization.csv")

    prior2["visit_dt"] = pd.to_datetime(
        prior2["visit_dt"], errors="coerce"
    )

    prior2 = prior2.merge(
        master[["patient_id", "admit_dt"]],
        on="patient_id",
        how="inner",
        validate="many_to_one"
    )

    # Only visits BEFORE the index admission
    prior2 = prior2[
        prior2["visit_dt"] < prior2["admit_dt"]
    ].copy()

    prior2["days_ago"] = (
        prior2["admit_dt"] - prior2["visit_dt"]
    ).dt.total_seconds() / 86400

    visit_type = (
        prior2["visit_type"]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.strip()
    )

    prior2["is_ed"] = visit_type.isin(
        ["ED", "ER", "EMERGENCY"]
    ).astype(int)

    prior2["is_ip"] = visit_type.isin(
        ["IP", "INPATIENT"]
    ).astype(int)

    prior2["is_op"] = visit_type.isin(
        ["OP", "OUTPATIENT"]
    ).astype(int)

    for days in [30, 90, 365]:
        recent = prior2["days_ago"] <= days

        prior2[f"visits_{days}d_v2"] = recent.astype(int)
        prior2[f"ed_{days}d"] = (
            recent & (prior2["is_ed"] == 1)
        ).astype(int)

        prior2[f"ip_{days}d"] = (
            recent & (prior2["is_ip"] == 1)
        ).astype(int)

    visit_features = prior2.groupby("patient_id").agg({
        "is_ed": "sum",
        "is_ip": "sum",
        "is_op": "sum",
        **{
            f"{prefix}_{days}d": "sum"
            for days in [30, 90, 365]
            for prefix in ["visits", "ed", "ip"]
            if prefix != "visits"
        },
        **{
            f"visits_{days}d_v2": "sum"
            for days in [30, 90, 365]
        }
    }).reset_index()

    visit_features = visit_features.rename(columns={
        "is_ed": "total_ed_visits",
        "is_ip": "total_ip_visits",
        "is_op": "total_op_visits"
    })

    extra = extra.merge(
        visit_features,
        on="patient_id",
        how="left",
        validate="one_to_one"
    )

    # B. DIAGNOSIS FEATURES


    diag2 = load("diagnoses.csv")
    lookup = load("icd10_lookup.csv")

    # Match diagnosis codes to lookup descriptions
    diag2["icd10_code"] = (
        diag2["icd10_code"].astype("string")
        .str.strip().str.upper()
    )

    lookup["icd10_code"] = (
        lookup["icd10_code"].astype("string")
        .str.strip().str.upper()
    )

    lookup = lookup.drop_duplicates("icd10_code")

    diag2 = diag2.merge(
        lookup[["icd10_code", "chapter", "chronic_flag"]],
        on="icd10_code",
        how="left",
        validate="many_to_one"
    )

    diag2["is_chronic"] = (
        diag2["chronic_flag"]
        .astype("string")
        .str.upper()
        .isin(["1", "Y", "YES", "TRUE"])
        .astype(int)
    )

    diag2["primary_num"] = pd.to_numeric(
        diag2["is_primary"], errors="coerce"
    ).fillna(0)

    diag2["seq_num"] = pd.to_numeric(
        diag2["seq"], errors="coerce"
    )

    diag_features = diag2.groupby("encounter_id").agg(
        chronic_diagnoses=("is_chronic", "sum"),
        diagnosis_chapters=("chapter", "nunique"),
        primary_diagnosis_count=("primary_num", "sum")
    ).reset_index()

    # the primary diagnosis
    primary_diag = diag2.sort_values(
        ["encounter_id", "primary_num", "seq_num"],
        ascending=[True, False, True]
    ).drop_duplicates("encounter_id")

    primary_diag = primary_diag[
        ["encounter_id", "icd10_code", "chapter"]
    ].rename(columns={
        "icd10_code": "primary_icd_code",
        "chapter": "primary_icd_chapter"
    })

    diag_features = diag_features.merge(
        primary_diag,
        on="encounter_id",
        how="left",
        validate="one_to_one"
    )

    extra = extra.merge(
        diag_features,
        on="encounter_id",
        how="left",
        validate="one_to_one"
    )

    # C. MEDICATION FEATURES

    med = load("medications.csv")

    med["drug_class"] = (
        med["drug_class"]
        .fillna("Unknown")
        .astype(str)
        .str.upper()
        .str.strip()
    )

    med["continued_num"] = pd.to_numeric(
        med["continued_at_discharge"],
        errors="coerce"
    )

    med_features = med.groupby("encounter_id").agg(
        medication_count=("drug_name", "count"),
        medication_classes=("drug_class", "nunique"),
        continued_medications=("continued_num", "sum")
    ).reset_index()

    extra = extra.merge(
        med_features,
        on="encounter_id",
        how="left",
        validate="one_to_one"
    )


    # D. PROCEDURE FEATURES

    proc = load("procedures.csv")

    proc_features = proc.groupby("encounter_id").agg(
        procedure_count=("code", "count"),
        unique_procedures=("code", "nunique")
    ).reset_index()

    extra = extra.merge(
        proc_features,
        on="encounter_id",
        how="left",
        validate="one_to_one"
    )


    # E. HOSPITAL UNIT TRANSFERS


    transfers = load("unit_transfers.csv")

    transfers["in_dt"] = pd.to_datetime(
        transfers["in_dt"], errors="coerce"
    )

    transfers["out_dt"] = pd.to_datetime(
        transfers["out_dt"], errors="coerce"
    )

    transfers["unit"] = (
        transfers["unit"]
        .fillna("Unknown")
        .astype(str)
        .str.upper()
        .str.strip()
    )

    transfers["unit_hours"] = (
        transfers["out_dt"] - transfers["in_dt"]
    ).dt.total_seconds() / 3600

    # Remove impossible durations
    transfers.loc[
        (transfers["unit_hours"] < 0) |
        (transfers["unit_hours"] > 8760),
        "unit_hours"
    ] = np.nan

    transfers["icu_flag"] = (
        transfers["unit"].str.contains(
            r"\bICU\b", regex=True
        )
    ).astype(int)

    transfers["icu_hours"] = np.where(
        transfers["icu_flag"] == 1,
        transfers["unit_hours"],
        0
    )

    transfer_features = transfers.groupby("encounter_id").agg(
        unit_records=("unit", "count"),
        unique_units=("unit", "nunique"),
        icu_exposure=("icu_flag", "max"),
        total_icu_hours=("icu_hours", "sum"),
        total_unit_hours=("unit_hours", "sum")
    ).reset_index()

    extra = extra.merge(
        transfer_features,
        on="encounter_id",
        how="left",
        validate="one_to_one"
    )


    # F. COMBINE WITH ORIGINAL V1 FEATURES


    # Keep original features in their original row order
    v2_features = features.reset_index(drop=True).copy()

    extra_features = extra.drop(
        columns=["encounter_id", "patient_id"]
    ).reset_index(drop=True)

    v2_features = pd.concat(
        [v2_features, extra_features],
        axis=1
    )

    # Count-based missing values mean zero records
    count_columns = [
        c for c in extra_features.columns
        if c not in [
            "primary_icd_code",
            "primary_icd_chapter"
        ]
    ]

    v2_features[count_columns] = (
        v2_features[count_columns].fillna(0)
    )

    # Standardize categorical columns
    cat_cols_v2 = v2_features.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()

    for col in cat_cols_v2:
        v2_features[col] = (
            v2_features[col]
            .fillna("Missing")
            .astype(str)
            .str.strip()
            .str.lower()
        )

    # Use encounter IDs to guarantee correct row alignment
    v2_features.index = master["encounter_id"]

    X_train_v2 = v2_features.loc[
        train["encounter_id"]
    ].reset_index(drop=True)

    X_test_v2 = v2_features.loc[
        test["encounter_id"]
    ].reset_index(drop=True)

    # Final checks
    assert len(X_train_v2) == 76975
    assert len(X_test_v2) == 32812
    assert X_train_v2.columns.is_unique
    assert X_train_v2.columns.equals(X_test_v2.columns)

    print("V2 FEATURE ENGINEERING COMPLETE")
    print("V1 features:", X_train.shape[1])
    print("V2 features:", X_train_v2.shape[1])
    print("New features:",
          X_train_v2.shape[1] - X_train.shape[1])
    print("Training rows:", len(X_train_v2))
    print("Test rows:", len(X_test_v2))


### Archive M07 — V2 holdout model

**Original source cell 7; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    #Train V2

    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score
    from catboost import CatBoostClassifier

    # Same split as V1
    X_fit_v2, X_val_v2, y_fit_v2, y_val_v2 = train_test_split(
        X_train_v2,
        y_train,
        test_size=0.20,
        random_state=42,
        stratify=y_train
    )

    # Create the improved model
    model_v2 = CatBoostClassifier(
        iterations=1000,
        depth=6,
        learning_rate=0.04,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        thread_count=-1,
        allow_writing_files=False
    )

    # Train
    model_v2.fit(
        X_fit_v2,
        y_fit_v2,
        cat_features=cat_cols_v2,
        eval_set=(X_val_v2, y_val_v2),
        early_stopping_rounds=100
    )

    # Validate
    pred_v2 = model_v2.predict_proba(X_val_v2)[:, 1]
    auc_v2 = roc_auc_score(y_val_v2, pred_v2)

    # Compare against V1
    print("\n==============================")
    print("V1 VALIDATION AUC:", v1_validation_auc)
    print("V2 VALIDATION AUC:", round(auc_v2, 5))
    print("IMPROVEMENT:", round(auc_v2 - v1_validation_auc, 5))
    print("BEST ITERATION:", model_v2.best_iteration_)
    print("==============================")

    if auc_v2 > v1_validation_auc:
        print("V2 BEATS V1 ON THIS VALIDATION SPLIT")
    else:
        print("V2 DID NOT BEAT V1 - REVIEW FEATURES")


### Archive M08 — V2 full-data prediction

**Original source cell 8; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # Train V2 on all labeled encounters

    best_rounds_v2 = max(
        100, model_v2.best_iteration_ + 1
    )

    final_v2 = CatBoostClassifier(
        iterations=best_rounds_v2,
        depth=6,
        learning_rate=0.04,
        loss_function="Logloss",
        random_seed=42,
        verbose=100,
        thread_count=-1,
        allow_writing_files=False
    )

    final_v2.fit(
        X_train_v2,
        y_train,
        cat_features=cat_cols_v2
    )

    # Predict test probabilities
    pred_test_v2 = final_v2.predict_proba(X_test_v2)[:, 1]

    # Preserve Kaggle's required encounter order
    submission_v2 = pd.DataFrame({
        "encounter_id": test["encounter_id"],
        "readmit_30d": pred_test_v2
    })

    # Validate submission format
    assert len(submission_v2) == len(test)
    assert submission_v2["encounter_id"].equals(test["encounter_id"])
    assert submission_v2["readmit_30d"].between(0, 1).all()
    assert submission_v2["readmit_30d"].notna().all()

    submission_v2.to_csv(
        "/kaggle/working/submission_v2.csv",
        index=False
    )

    print("V2 SUBMISSION FILE READY!")
    print("Rows:", len(submission_v2))
    print(submission_v2.head())


### Archive M09 — V3 laboratory summaries

**Original source cell 9; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # V3 LABORATORY FEATURES

    import pandas as pd
    import numpy as np

    labs = pd.read_csv(
        files["lab_results.csv"],
        usecols=[
            "encounter_id", "lab_name",
            "collected_dt", "value",
            "unit", "abnormal_flag"
        ]
    )

    print("Original lab rows:", len(labs))

    # Convert dates and numeric measurements
    labs["collected_dt"] = pd.to_datetime(
        labs["collected_dt"], errors="coerce"
    )
    labs["value"] = pd.to_numeric(
        labs["value"], errors="coerce"
    )

    # Attach admission/discharge timestamps
    labs = labs.merge(
        master[[
            "encounter_id", "admit_dt", "discharge_dt"
        ]],
        on="encounter_id",
        how="inner",
        validate="many_to_one"
    )

    # Only use measurements 
    labs = labs[
        (labs["collected_dt"] >= labs["admit_dt"]) &
        (labs["collected_dt"] <= labs["discharge_dt"])
    ].copy()

    # Standardize test names and units
    labs["lab_key"] = (
        labs["lab_name"].astype("string")
        .str.strip().str.lower()
    )

    labs["unit_clean"] = (
        labs["unit"].astype("string")
        .str.strip().str.lower()
        .str.replace("µ", "u", regex=False)
        .str.replace("μ", "u", regex=False)
    )

    # Convert creatinine from umol/L to mg/dL
    mask = (
        (labs["lab_key"] == "creatinine") &
        (labs["unit_clean"] == "umol/l")
    )
    labs.loc[mask, "value"] /= 88.4

    # Convert hemoglobin from g/L to g/dL
    mask = (
        (labs["lab_key"] == "hemoglobin") &
        (labs["unit_clean"] == "g/l")
    )
    labs.loc[mask, "value"] /= 10

    # Convert BUN from mmol/L to mg/dL
    mask = (
        (labs["lab_key"] == "bun") &
        (labs["unit_clean"] == "mmol/l")
    )
    labs.loc[mask, "value"] *= 2.801

    # Convert glucose from mmol/L to mg/dL
    mask = (
        (labs["lab_key"] == "glucose") &
        (labs["unit_clean"] == "mmol/l")
    )
    labs.loc[mask, "value"] *= 18.018

    # Select the common tests
    selected_labs = [
        "creatinine", "bun", "hemoglobin",
        "wbc", "potassium", "sodium", "glucose"
    ]

    # Overall number of tests and abnormal flags
    labs["flag_clean"] = (
        labs["abnormal_flag"].fillna("")
        .astype(str).str.strip().str.upper()
    )

    labs["abnormal_num"] = (
        labs["flag_clean"].isin(["H", "L"])
    ).astype(int)

    labs["high_num"] = (
        labs["flag_clean"] == "H"
    ).astype(int)

    labs["low_num"] = (
        labs["flag_clean"] == "L"
    ).astype(int)

    lab_overall = labs.groupby("encounter_id").agg(
        lab_total_count=("value", "count"),
        lab_abnormal_count=("abnormal_num", "sum"),
        lab_high_count=("high_num", "sum"),
        lab_low_count=("low_num", "sum")
    )

    lab_overall["lab_abnormal_fraction"] = (
        lab_overall["lab_abnormal_count"] /
        lab_overall["lab_total_count"].replace(0, np.nan)
    )

    # Summarize individual laboratory tests
    labs_selected = labs[
        labs["lab_key"].isin(selected_labs)
    ].copy()

    # Avoid mixing incompatible units
    allowed_units = {
        "creatinine": ["mg/dl", "umol/l"],
        "bun": ["mg/dl", "mmol/l"],
        "hemoglobin": ["g/dl", "g/l"],
        "wbc": ["10^3/ul", "10^9/l"],
        "potassium": ["mmol/l", "meq/l"],
        "sodium": ["mmol/l", "meq/l"],
        "glucose": ["mg/dl", "mmol/l"]
    }

    valid_units = pd.Series(False, index=labs_selected.index)

    for test_name, units in allowed_units.items():
        valid_units |= (
            (labs_selected["lab_key"] == test_name) &
            (labs_selected["unit_clean"].isin(units))
        )

    print("Selected test rows:", len(labs_selected))
    print("Recognized unit rows:", int(valid_units.sum()))
    print("Units not recognized:")
    print(
        labs_selected.loc[~valid_units]
        .groupby(["lab_key", "unit_clean"])
        .size().head(20)
    )

    labs_selected = labs_selected[valid_units].copy()

    labs_selected = labs_selected.sort_values(
        ["encounter_id", "lab_key", "collected_dt"]
    )

    lab_stats = (
        labs_selected
        .groupby(["encounter_id", "lab_key"])["value"]
        .agg(["count", "mean", "min", "max",
              "std", "first", "last"])
        .unstack("lab_key")
    )

    lab_stats.columns = [
        f"lab_{test}_{stat}"
        for stat, test in lab_stats.columns
    ]

    # Calculate first-to-last changes
    for test_name in selected_labs:
        first_col = f"lab_{test_name}_first"
        last_col = f"lab_{test_name}_last"

        if first_col in lab_stats and last_col in lab_stats:
            lab_stats[f"lab_{test_name}_change"] = (
                lab_stats[last_col] - lab_stats[first_col]
            )

    lab_features = lab_overall.join(
        lab_stats, how="left"
    )

    print("\nLAB FEATURES CREATED")
    print("Feature count:", lab_features.shape[1])
    print("Encounters covered:", len(lab_features))


### Archive M10 — V3 vital summaries

**Original source cell 10; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # V3 VITAL SIGN FEATURES

    vitals = pd.read_csv(
        files["vitals.csv"],
        usecols=[
            "encounter_id", "charttime",
            "vital_type", "value"
        ]
    )

    print("Original vital rows:", len(vitals))

    # Convert dates and numeric values
    vitals["charttime"] = pd.to_datetime(
        vitals["charttime"], errors="coerce"
    )

    vitals["value"] = pd.to_numeric(
        vitals["value"], errors="coerce"
    )

    vitals["vital_type"] = (
        vitals["vital_type"].astype("string")
        .str.strip().str.upper()
    )

    # Attach admission/discharge dates
    vitals = vitals.merge(
        master[[
            "encounter_id", "admit_dt", "discharge_dt"
        ]],
        on="encounter_id",
        how="inner",
        validate="many_to_one"
    )

    # Keep observations during hospitalization
    vitals = vitals[
        (vitals["charttime"] >= vitals["admit_dt"]) &
        (vitals["charttime"] <= vitals["discharge_dt"])
    ].copy()

    # Convert Fahrenheit temp to Celsius
    temp_mask = (
        (vitals["vital_type"] == "TEMP") &
        (vitals["value"] > 45) &
        (vitals["value"] <= 115)
    )

    vitals.loc[temp_mask, "value"] = (
        (vitals.loc[temp_mask, "value"] - 32) * 5 / 9
    )

    # Reject clearly impossible meas
    valid_ranges = {
        "HR": (20, 250),
        "SBP": (40, 300),
        "DBP": (20, 200),
        "RR": (4, 70),
        "SPO2": (40, 100),
        "TEMP": (30, 43)
    }

    for vital_name, (minimum, maximum) in valid_ranges.items():
        invalid = (
            (vitals["vital_type"] == vital_name) &
            (~vitals["value"].between(minimum, maximum))
        )
        vitals.loc[invalid, "value"] = np.nan

    # Keep the six relevant vital types
    vitals = vitals[
        vitals["vital_type"].isin(valid_ranges)
    ].copy()

    # Sort measurements in chronological order
    vitals = vitals.sort_values(
        ["encounter_id", "vital_type", "charttime"]
    )

    # Calculate summary statistics
    vital_stats = (
        vitals
        .groupby(["encounter_id", "vital_type"])["value"]
        .agg(["count", "mean", "min", "max",
              "std", "first", "last"])
        .unstack("vital_type")
    )

    vital_stats.columns = [
        f"vital_{vital.lower()}_{stat}"
        for stat, vital in vital_stats.columns
    ]

    # Calculate changes between first and last readings
    for vital_name in valid_ranges:
        name = vital_name.lower()

        first_col = f"vital_{name}_first"
        last_col = f"vital_{name}_last"

        if first_col in vital_stats and last_col in vital_stats:
            vital_stats[f"vital_{name}_change"] = (
                vital_stats[last_col] -
                vital_stats[first_col]
            )

    # Additional feature: final pulse pressure
    if (
        "vital_sbp_last" in vital_stats and
        "vital_dbp_last" in vital_stats
    ):
        vital_stats["pulse_pressure_last"] = (
            vital_stats["vital_sbp_last"] -
            vital_stats["vital_dbp_last"]
        )

    vital_features = vital_stats.copy()

    print("\nVITAL FEATURES CREATED")
    print("Feature count:", vital_features.shape[1])
    print("Encounters covered:", len(vital_features))


### Archive M11 — V3 merge

**Original source cell 11; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # V3  COMBINE ALL FEATURES

    # Start with V2
    v3_features = v2_features.copy()

    # Attach laboratory features by encounter ID
    v3_features = v3_features.join(
        lab_features, how="left", validate="one_to_one"
    )

    # Attach vital-sign features by encounter ID
    v3_features = v3_features.join(
        vital_features, how="left", validate="one_to_one"
    )

    # Keep only columns that contain usable training data
    train_ids = train["encounter_id"]
    test_ids = test["encounter_id"]

    X_train_v3 = v3_features.reindex(
        train_ids
    ).reset_index(drop=True)

    X_test_v3 = v3_features.reindex(
        test_ids
    ).reset_index(drop=True)

    # Drop features entirely missing in training
    empty_cols = X_train_v3.columns[
        X_train_v3.isna().all()
    ].tolist()

    X_train_v3 = X_train_v3.drop(columns=empty_cols)
    X_test_v3 = X_test_v3.drop(columns=empty_cols)

    cat_cols_v3 = [
        c for c in cat_cols_v2
        if c in X_train_v3.columns
    ]

    # Ensure row and column consistency
    assert len(X_train_v3) == len(train)
    assert len(X_test_v3) == len(test)
    assert X_train_v3.columns.equals(X_test_v3.columns)
    assert X_train_v3.columns.is_unique

    print("V3 DATASET READY")
    print("V2 features:", X_train_v2.shape[1])
    print("V3 features:", X_train_v3.shape[1])
    print("New features:",
          X_train_v3.shape[1] - X_train_v2.shape[1])
    print("Training rows:", len(X_train_v3))
    print("Test rows:", len(X_test_v3))
    print("All-missing columns removed:", len(empty_cols))


### Archive M12 — V3 holdout training

**Original source cell 12; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # V3 TRAIN AND EVALUATE

    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score
    from catboost import CatBoostClassifier

    # Use the same validation split as before
    X_fit_v3, X_val_v3, y_fit_v3, y_val_v3 = train_test_split(
        X_train_v3,
        y_train,
        test_size=0.20,
        random_state=42,
        stratify=y_train
    )

    # Train the V3 model
    model_v3 = CatBoostClassifier(
        iterations=1200,
        depth=6,
        learning_rate=0.04,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        thread_count=-1,
        allow_writing_files=False
    )

    model_v3.fit(
        X_fit_v3,
        y_fit_v3,
        cat_features=cat_cols_v3,
        eval_set=(X_val_v3, y_val_v3),
        early_stopping_rounds=100
    )

    # Predict validation probabilities
    pred_v3 = model_v3.predict_proba(X_val_v3)[:, 1]

    auc_v3 = roc_auc_score(y_val_v3, pred_v3)

    print("\n================================")
    print("V1 VALIDATION:", 0.72946)
    print("V2 VALIDATION:", round(auc_v2, 5))
    print("V3 VALIDATION:", round(auc_v3, 5))
    print("V3 IMPROVEMENT OVER V2:",
          round(auc_v3 - auc_v2, 5))
    print("BEST ITERATION:", model_v3.best_iteration_)
    print("================================")

    if auc_v3 > auc_v2:
        print("V3 BEATS V2 ON THIS SPLIT")
    else:
        print("V3 DID NOT BEAT V2 ON THIS SPLIT")

    # Show the 15 most influential features
    importance = pd.DataFrame({
        "feature": X_train_v3.columns,
        "importance": model_v3.feature_importances_
    })

    print("\nTOP 15 FEATURES:")
    print(
        importance.sort_values(
            "importance", ascending=False
        ).head(15).to_string(index=False)
    )


### Archive M13 — V3 cross-validation

**Original source cell 13; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # V3  CROSS-VALIDATION

    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from catboost import CatBoostClassifier
    import numpy as np

    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    results = {}

    models_to_compare = [
        ("V2", X_train_v2, cat_cols_v2,
         model_v2.best_iteration_ + 1),
        ("V3", X_train_v3, cat_cols_v3,
         model_v3.best_iteration_ + 1)
    ]

    for name, X, categories, rounds in models_to_compare:

        scores = []

        for fold, (tr_idx, val_idx) in enumerate(
            skf.split(X, y_train), start=1
        ):

            cv_model = CatBoostClassifier(
                iterations=max(100, rounds),
                depth=6,
                learning_rate=0.04,
                loss_function="Logloss",
                random_seed=42,
                verbose=False,
                thread_count=-1,
                allow_writing_files=False
            )

            cv_model.fit(
                X.iloc[tr_idx],
                y_train.iloc[tr_idx],
                cat_features=categories
            )

            predictions = cv_model.predict_proba(
                X.iloc[val_idx]
            )[:, 1]

            score = roc_auc_score(
                y_train.iloc[val_idx],
                predictions
            )

            scores.append(score)

            print(
                name, "Fold", fold,
                "AUC:", round(score, 5)
            )

        results[name] = scores

        print(
            name, "MEAN AUC:",
            round(np.mean(scores), 5)
        )

    print("\n==============================")
    print("V2 CV MEAN:", round(np.mean(results["V2"]), 5))
    print("V3 CV MEAN:", round(np.mean(results["V3"]), 5))
    print("MEAN IMPROVEMENT:",
          round(
              np.mean(results["V3"]) -
              np.mean(results["V2"]), 5
          ))
    print("==============================")


### Archive M14 — V3 missing lab-unit check

**Original source cell 14; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # CHECK MISSING LAB UNITS

    print("Missing units by laboratory test:")

    print(
        labs_selected.groupby("lab_key")["unit_clean"]
        .apply(lambda x: x.isna().sum())
        .sort_values(ascending=False)
    )

    print("\nAvailable units:")
    print(
        labs_selected.groupby(
            ["lab_key", "unit_clean"],
            dropna=False
        ).size().to_string()
    )


### Archive M15 — V3 full training

**Original source cell 15; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # FINAL V3 MODEL - ALL TRAINING DATA

    from catboost import CatBoostClassifier

    final_v3 = CatBoostClassifier(
        iterations=model_v3.best_iteration_ + 1,
        depth=6,
        learning_rate=0.04,
        loss_function="Logloss",
        random_seed=42,
        verbose=100,
        thread_count=-1,
        allow_writing_files=False
    )

    final_v3.fit(
        X_train_v3,
        y_train,
        cat_features=cat_cols_v3
    )

    print("FINAL V3 MODEL TRAINED!")
    print("Training rows:", len(X_train_v3))
    print("Features:", X_train_v3.shape[1])


### Archive M16 — V3 submission

**Original source cell 16; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # CREATE V3 KAGGLE SUBMISSION

    import pandas as pd
    import numpy as np

    predictions_v3 = final_v3.predict_proba(
        X_test_v3
    )[:, 1]

    submission_v3 = pd.DataFrame({
        "encounter_id": test["encounter_id"].values,
        "readmit_30d": predictions_v3
    })

    # Safety checks
    assert len(submission_v3) == 32812
    assert submission_v3["encounter_id"].is_unique
    assert submission_v3["readmit_30d"].notna().all()
    assert np.isfinite(predictions_v3).all()
    assert submission_v3["readmit_30d"].between(0, 1).all()

    submission_v3.to_csv(
        "/kaggle/working/submission_v3.csv",
        index=False
    )

    print("V3 SUBMISSION READY!")
    print(submission_v3.head())
    print("Rows:", len(submission_v3))


### Archive M17 — V4 raw lab audit

**Original source cell 17; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # V4 - LAB AUDIT BEFORE FILTer

    raw_labs = pd.read_csv(files["lab_results.csv"])

    raw_labs["lab_key"] = (
        raw_labs["lab_name"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    raw_labs["unit_clean"] = (
        raw_labs["unit"]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace("µ", "u", regex=False)
        .str.replace("μ", "u", regex=False)
    )

    selected_raw = raw_labs[
        raw_labs["lab_key"].isin(selected_labs)
    ].copy()

    print("Original selected lab rows:", len(selected_raw))

    print("\nUnit distribution before filtering:")
    print(
        selected_raw.groupby(
            ["lab_key", "unit_clean"],
            dropna=False
        ).size().to_string()
    )

    print("\nMissing lab values:")
    print(selected_raw["value"].isna().sum())


### Archive M18 — V4 prior visits

**Original source cell 18; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # patient's previous hospital visits

    history = pd.read_csv(files["prior_utilization.csv"])

    # dates are actual dates, not text
    history["visit_dt"] = pd.to_datetime(
        history["visit_dt"],
        errors="coerce",
        format="mixed"
    )

    #  admission date 
    admissions = master[
        ["encounter_id", "patient_id", "admit_dt"]
    ].copy()

    admissions["admit_dt"] = pd.to_datetime(
        admissions["admit_dt"],
        errors="coerce",
        format="mixed"
    )

    history = history.merge(
        admissions,
        on="patient_id",
        how="inner",
        validate="many_to_one"
    )

    history["days_before"] = (
        history["admit_dt"] - history["visit_dt"]
    ).dt.total_seconds() / 86400

    history = history[
        history["days_before"].ge(0)
    ].copy()

    # Clean up the visit names
    history["visit_type"] = (
        history["visit_type"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Make a separate table for new features
    visit_features = pd.DataFrame(
        index=admissions["encounter_id"]
    )

    visit_features.index.name = "encounter_id"

    # Count visits in different time windows
    for days in [30, 90, 365]:
        recent = history[
            history["days_before"] <= days
        ]

        counts = recent.groupby("encounter_id").size()

        visit_features[f"visits_{days}d"] = counts

    # Count emergency, inpatient and outpatient visits
    for visit_type in ["ED", "IP", "OP"]:
        selected = history[
            history["visit_type"] == visit_type
        ]

        counts = selected.groupby("encounter_id").size()

        visit_features[f"visits_{visit_type.lower()}"] = counts

    # time since most recent visit 
    visit_features["days_since_last_visit"] = (
        history.groupby("encounter_id")["days_before"].min()
    )

    #  different hospitals per patient visitation
    visit_features["different_hospitals_visited"] = (
        history.groupby("encounter_id")["hospital_id"].nunique()
    )


    count_columns = [
        c for c in visit_features.columns
        if c != "days_since_last_visit"
    ]

    visit_features[count_columns] = (
        visit_features[count_columns].fillna(0)
    )

    print("PREVIOUS VISIT FEATURES READY")
    print("New features:", visit_features.shape[1])
    print("Patients with recorded previous visits:",
          visit_features["days_since_last_visit"].notna().sum())


### Archive M19 — V4 feature table

**Original source cell 19; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import pandas as pd

    # V4 - rebuild the dataset

    needed = [
        "v2_features", "lab_features", "vital_features",
        "visit_features", "selected_raw", "train", "test"
    ]

    missing = [name for name in needed if name not in globals()]

    if missing:
        raise RuntimeError("Missing source tables: " + str(missing))


    # make sure all tables use encounter_id as the index
    def fix_index(df):
        df = df.copy()

        if "encounter_id" in df.columns:
            df = df.set_index("encounter_id")

        df.index = df.index.astype(str)
        df.index.name = "encounter_id"

        if not df.index.is_unique:
            raise ValueError("Duplicate encounter ids found")

        if not df.columns.is_unique:
            raise ValueError("Duplicate columns found")

        return df


    # recreate the missing lab unit features
    if "unit_clean" not in selected_raw.columns:
        raise ValueError("unit_clean is missing from selected_raw")

    if "encounter_id" in selected_raw.columns:
        lab_units = selected_raw[
            ["encounter_id", "unit_clean"]
        ].copy()

    elif selected_raw.index.name == "encounter_id":
        lab_units = selected_raw[
            ["unit_clean"]
        ].reset_index()

    else:
        raise ValueError("encounter_id is missing from selected_raw")


    lab_units["unit_missing"] = (
        lab_units["unit_clean"].isna().astype(int)
    )

    missing_lab_features = lab_units.groupby(
        "encounter_id"
    ).agg(
        labs_missing_units=("unit_missing", "sum"),
        labs_total_selected=("unit_missing", "size")
    )

    missing_lab_features["labs_missing_units_fraction"] = (
        missing_lab_features["labs_missing_units"] /
        missing_lab_features["labs_total_selected"]
    )


    # rebuild all the v4 features
    v4_features = fix_index(v2_features)

    for table in [
        lab_features,
        vital_features,
        visit_features,
        missing_lab_features
    ]:
        extra = fix_index(table)

        # only add columns we don't already have
        new_cols = extra.columns.difference(
            v4_features.columns, sort=False
        )

        v4_features = v4_features.join(
            extra[new_cols],
            how="left",
            validate="one_to_one"
        )


    # make sure the target isn't part of the features
    v4_features = v4_features.drop(
        columns=["readmit_30d"],
        errors="ignore"
    )

    if v4_features.shape[1] != 178:
        raise ValueError(
            f"Expected 178 features, got {v4_features.shape[1]}"
        )


    # separate the training and test rows
    train_ids = train["encounter_id"].astype(str)
    test_ids = test["encounter_id"].astype(str)

    all_ids = pd.Index(
        pd.concat([train_ids, test_ids], ignore_index=True)
    )

    missing_ids = all_ids.difference(v4_features.index)

    if len(missing_ids) > 0:
        raise ValueError(
            f"{len(missing_ids)} encounters are missing"
        )

    X_train_v4 = v4_features.loc[
        train_ids
    ].reset_index(drop=True)

    X_test_v4 = v4_features.loc[
        test_ids
    ].reset_index(drop=True)


    # categorical columns from v4
    known_cats = [
        "hospital_id", "admission_type", "admission_source",
        "service_line", "discharge_disposition",
        "attending_specialty", "sex", "race_ethnicity",
        "language", "marital_status", "insurance_type",
        "urban_rural", "ownership", "home_health_referral",
        "education_done", "primary_icd_code",
        "primary_icd_chapter"
    ]

    cat_cols_v4 = [
        col for col in known_cats
        if col in X_train_v4.columns
    ]

    for col in X_train_v4.columns:
        if (
            col not in cat_cols_v4
            and not pd.api.types.is_numeric_dtype(X_train_v4[col])
        ):
            cat_cols_v4.append(col)


    # prepare the categorical values for training
    X_train_v4_ready = X_train_v4.copy()
    X_test_v4_ready = X_test_v4.copy()

    for col in cat_cols_v4:
        for df in [X_train_v4_ready, X_test_v4_ready]:
            df[col] = (
                df[col]
                .astype(object)
                .where(df[col].notna(), "Missing")
                .astype(str)
            )


    # check the final dataset
    print("V4 DATASET READY")
    print("Training shape:", X_train_v4_ready.shape)
    print("Test shape:", X_test_v4_ready.shape)
    print("Categorical columns:", len(cat_cols_v4))
    print("Duplicate columns:", v4_features.columns.duplicated().sum())
    print("Missing lab features:", list(missing_lab_features.columns))


### Archive M20 — V4 feature audit

**Original source cell 20; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # check the v4 dataset before training

    print("Training shape:", X_train_v4.shape)
    print("Test shape:", X_test_v4.shape)

    print("Categorical columns:", len(cat_cols_v4))
    print("Missing values:", X_train_v4.isna().sum().sum())

    print("Infinite values:",
          np.isinf(X_train_v4.select_dtypes(include="number")).sum().sum())

    print("Target available:", "readmit_30d" in train.columns)
    print("Same columns:", X_train_v4.columns.equals(X_test_v4.columns))

    print("\nCategorical columns:")
    print(cat_cols_v4)


### Archive M21 — V4 CatBoost holdout

**Original source cell 21; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    from catboost import CatBoostClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score

    # use the same target from the training file
    y_v4 = train["readmit_30d"].astype(int)

    # split the data into training and validation
    train_idx, valid_idx = train_test_split(
        np.arange(len(y_v4)),
        test_size=0.20,
        random_state=42,
        stratify=y_v4
    )

    # train the v4 model
    model_v4 = CatBoostClassifier(
        iterations=1500,
        depth=6,
        learning_rate=0.035,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        l2_leaf_reg=5,
        verbose=100,
        thread_count=-1
    )

    model_v4.fit(
        X_train_v4_ready.iloc[train_idx],
        y_v4.iloc[train_idx],
        cat_features=cat_cols_v4,
        eval_set=(
            X_train_v4_ready.iloc[valid_idx],
            y_v4.iloc[valid_idx]
        ),
        early_stopping_rounds=100
    )

    # calculate the validation score
    pred_v4 = model_v4.predict_proba(
        X_train_v4_ready.iloc[valid_idx]
    )[:, 1]

    auc_v4 = roc_auc_score(
        y_v4.iloc[valid_idx],
        pred_v4
    )

    print("\nV4 VALIDATION ROC-AUC:", round(auc_v4, 5))
    print("BEST ITERATION:", model_v4.best_iteration_)


### Archive M22 — V4 submission

**Original source cell 22; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution
    # V4 - save test predictions

    best_rounds_v4 = max(100, model_v4.best_iteration_ + 1)

    final_v4 = CatBoostClassifier(
        iterations=best_rounds_v4,
        depth=6,
        learning_rate=0.035,
        loss_function="Logloss",
        random_seed=42,
        l2_leaf_reg=5,
        verbose=100,
        thread_count=-1,
        allow_writing_files=False
    )

    final_v4.fit(
        X_train_v4_ready,
        y_v4,
        cat_features=cat_cols_v4
    )

    pred_test_v4 = final_v4.predict_proba(X_test_v4_ready)[:, 1]

    submission_v4 = pd.DataFrame({
        "encounter_id": test["encounter_id"].values,
        "readmit_30d": pred_test_v4
    })

    # check the submission before saving
    assert len(submission_v4) == len(test)
    assert submission_v4["encounter_id"].equals(test["encounter_id"])
    assert submission_v4["readmit_30d"].between(0, 1).all()
    assert submission_v4["readmit_30d"].notna().all()

    submission_v4.to_csv(
        "/kaggle/working/submission_v4.csv",
        index=False
    )

    print("V4 SUBMISSION FILE READY")
    print("Rows:", len(submission_v4))
    print("File: /kaggle/working/submission_v4.csv")
    print(submission_v4.head())


### Archive M23 — V5 cross-validated models

**Original source cell 23; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution
    # V5 - compare models before using more submissions

    import numpy as np
    import pandas as pd
    from catboost import CatBoostClassifier
    from xgboost import XGBClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OrdinalEncoder

    # use the V4 dataset already made
    X = X_train_v4_ready.reset_index(drop=True).copy()
    T = X_test_v4_ready.reset_index(drop=True).copy()
    y = train["readmit_30d"].to_numpy(dtype=int)
    cat_columns = list(cat_cols_v4)
    num_columns = [col for col in X.columns if col not in cat_columns]

    assert X.shape == (76975, 178), X.shape
    assert T.shape == (32812, 178), T.shape
    assert X.columns.equals(T.columns)
    assert len(y) == len(X)
    assert not X.columns.duplicated().any()
    assert set(np.unique(y)) == {0, 1}

    # keep the same folds for every model
    folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    model_names = ["catboost_6", "catboost_5", "xgboost"]
    oof = {name: np.zeros(len(X)) for name in model_names}
    test_preds = {name: np.zeros(len(T)) for name in model_names}
    fold_scores = {name: [] for name in model_names}

    for fold, (tr, va) in enumerate(folds.split(X, y), start=1):
        print("\nFOLD", fold, "OF 3")

        for name, depth, rate, reg in [
            ("catboost_6", 6, 0.035, 5),
            ("catboost_5", 5, 0.05, 7)
        ]:
            model = CatBoostClassifier(
                iterations=900,
                depth=depth,
                learning_rate=rate,
                l2_leaf_reg=reg,
                loss_function="Logloss",
                eval_metric="AUC",
                random_seed=42 + fold,
                thread_count=-1,
                verbose=False,
                allow_writing_files=False
            )
            model.fit(
                X.iloc[tr], y[tr],
                cat_features=cat_columns,
                eval_set=(X.iloc[va], y[va]),
                early_stopping_rounds=90,
                verbose=False
            )
            oof[name][va] = model.predict_proba(X.iloc[va])[:, 1]
            test_preds[name] += model.predict_proba(T)[:, 1] / 3
            score = roc_auc_score(y[va], oof[name][va])
            fold_scores[name].append(score)
            print(name, "AUC:", round(score, 5), "rounds:", model.best_iteration_ + 1)

        # fit the XGBoost encoder only on this fold's training rows
        prep = ColumnTransformer([
            ("num", SimpleImputer(strategy="median", keep_empty_features=True), num_columns),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_columns)
        ])
        x_tr = prep.fit_transform(X.iloc[tr])
        x_va = prep.transform(X.iloc[va])
        x_te = prep.transform(T)

        xgb = XGBClassifier(
            n_estimators=1100,
            max_depth=4,
            learning_rate=0.035,
            min_child_weight=8,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=8,
            objective="binary:logistic",
            tree_method="hist",
            eval_metric="auc",
            early_stopping_rounds=90,
            random_state=42 + fold,
            n_jobs=-1
        )
        xgb.fit(x_tr, y[tr], eval_set=[(x_va, y[va])], verbose=False)
        oof["xgboost"][va] = xgb.predict_proba(x_va)[:, 1]
        test_preds["xgboost"] += xgb.predict_proba(x_te)[:, 1] / 3
        score = roc_auc_score(y[va], oof["xgboost"][va])
        fold_scores["xgboost"].append(score)
        print("xgboost AUC:", round(score, 5), "rounds:", xgb.best_iteration + 1)

    # blend the predictions and check whether they improve the AUC
    blends = {
        "blend_cb6_xgb": {"catboost_6": 0.7, "xgboost": 0.3},
        "blend_cb6_cb5": {"catboost_6": 0.5, "catboost_5": 0.5},
        "blend_all": {"catboost_6": 0.5, "catboost_5": 0.25, "xgboost": 0.25}
    }
    for name, weights in blends.items():
        oof[name] = sum(weight * oof[part] for part, weight in weights.items())
        test_preds[name] = sum(weight * test_preds[part] for part, weight in weights.items())

    scores = pd.DataFrame([
        {"model": name, "oof_auc": roc_auc_score(y, preds)}
        for name, preds in oof.items()
    ]).sort_values("oof_auc", ascending=False).reset_index(drop=True)

    print("\nMODEL COMPARISON - SAME 3 FOLDS")
    print(scores.to_string(index=False))
    print("\nV4 single-split AUC was 0.74499 (not directly comparable).")

    # save the files now, but don't submit them yet
    for name, preds in test_preds.items():
        submission = pd.DataFrame({
            "encounter_id": test["encounter_id"].to_numpy(),
            "readmit_30d": preds
        })
        assert len(submission) == len(test)
        assert submission["encounter_id"].is_unique
        assert np.isfinite(preds).all()
        assert submission["readmit_30d"].between(0, 1).all()
        submission.to_csv(f"/kaggle/working/{name}.csv", index=False)

    print("\nAll 6 candidate CSV files saved in /kaggle/working/")


### Archive M24 — V5 extra blends

**Original source cell 24; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution
    # V5 - check a few more blends without training again

    import numpy as np
    import pandas as pd
    from scipy.stats import rankdata
    from sklearn.metrics import roc_auc_score

    needed = ["oof", "test_preds", "train", "test"]
    missing = [name for name in needed if name not in globals()]
    if missing:
        raise RuntimeError("Run the V5 comparison cell first. Missing: " + str(missing))

    for key in ["catboost_6", "catboost_5", "xgboost"]:
        if key not in oof or key not in test_preds:
            raise ValueError("Missing V5 predictions for " + key)

    y_check = train["readmit_30d"].to_numpy(dtype=int)
    cb_oof, xgb_oof = oof["catboost_6"], oof["xgboost"]
    cb_test, xgb_test = test_preds["catboost_6"], test_preds["xgboost"]

    candidates = {}
    for weight in [0.5, 0.6, 0.7, 0.8]:
        name = "v5_cb6_xgb_" + str(int(round(weight * 100))) + "_" + str(100 - int(round(weight * 100)))
        candidates[name] = (
            weight * cb_oof + (1 - weight) * xgb_oof,
            weight * cb_test + (1 - weight) * xgb_test
        )

    # ranks can sometimes work better than averaging raw probabilities
    n_oof = len(cb_oof)
    n_test = len(cb_test)
    for weight in [0.5, 0.7]:
        name = "v5_rank_blend_" + str(int(round(weight * 100)))
        candidates[name] = (
            weight * rankdata(cb_oof) / n_oof + (1 - weight) * rankdata(xgb_oof) / n_oof,
            weight * rankdata(cb_test) / n_test + (1 - weight) * rankdata(xgb_test) / n_test
        )

    results = []
    for name, (pred_oof, pred_test) in candidates.items():
        auc = roc_auc_score(y_check, pred_oof)
        results.append({"model": name, "oof_auc": auc})
        out = pd.DataFrame({
            "encounter_id": test["encounter_id"].to_numpy(),
            "readmit_30d": pred_test
        })
        assert len(out) == len(test)
        assert out["encounter_id"].is_unique
        assert out["readmit_30d"].between(0, 1).all()
        assert np.isfinite(out["readmit_30d"]).all()
        out.to_csv("/kaggle/working/" + name + ".csv", index=False)

    print("V5 EXTRA BLENDS - SAME OOF FOLDS")
    print(pd.DataFrame(results).sort_values("oof_auc", ascending=False).to_string(index=False))
    print("Original V5 best OOF:", round(roc_auc_score(y_check, oof["blend_cb6_xgb"]), 6))
    print("Extra blend files saved. Do not submit all of them.")


### Archive M25 — V6 experimental features (initial attempt)

**Original source cell 25; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution
    # V6 - extra features and two new models

    import numpy as np
    import pandas as pd
    from catboost import CatBoostClassifier
    from xgboost import XGBClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder

    needed = ["X_train_v4_ready", "X_test_v4_ready", "cat_cols_v4", "master", "train", "test", "oof", "test_preds"]
    missing = [name for name in needed if name not in globals()]
    if missing:
        raise RuntimeError("Run the V4 and V5 cells first. Missing: " + str(missing))

    # use the V4 features and keep the same train/test row order
    X6 = X_train_v4_ready.reset_index(drop=True).copy()
    T6 = X_test_v4_ready.reset_index(drop=True).copy()
    y6 = train["readmit_30d"].to_numpy(dtype=int)
    assert X6.shape == (76975, 178), X6.shape
    assert T6.shape == (32812, 178), T6.shape
    assert X6.columns.equals(T6.columns)

    both = pd.concat([X6, T6], ignore_index=True)
    original_columns = set(both.columns)

    # add calendar information from the index admission and discharge
    ids = pd.concat([train["encounter_id"], test["encounter_id"]], ignore_index=True).astype(str)
    enc_dates = master.copy()
    enc_dates["encounter_id"] = enc_dates["encounter_id"].astype(str)
    assert enc_dates["encounter_id"].is_unique
    enc_dates = enc_dates.set_index("encounter_id").reindex(ids)
    for date_name in ["admit_dt", "discharge_dt"]:
        dates = pd.to_datetime(enc_dates[date_name], errors="coerce")
        both[date_name + "_hour"] = dates.dt.hour.to_numpy(dtype=float)
        both[date_name + "_month"] = dates.dt.month.to_numpy(dtype=float)
        both[date_name + "_year"] = dates.dt.year.to_numpy(dtype=float)
        both[date_name + "_weekday"] = dates.dt.dayofweek.to_numpy(dtype=float)
        both[date_name + "_weekend"] = (dates.dt.dayofweek >= 5).where(dates.notna()).to_numpy(dtype=float)

    # make some simple combinations of the existing medical measurements
    def numeric(col):
        return pd.to_numeric(both[col], errors="coerce") if col in both else None

    def ratio(name, top, bottom, add=0.0):
        a, b = numeric(top), numeric(bottom)
        if a is not None and b is not None:
            both[name] = (a / (b + add).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

    def difference(name, left, right):
        a, b = numeric(left), numeric(right)
        if a is not None and b is not None:
            both[name] = a - b

    ratio("bun_creatinine_ratio", "lab_bun_last", "lab_creatinine_last")
    ratio("shock_index_last", "vital_hr_last", "vital_sbp_last")
    ratio("lab_tests_per_stay", "lab_total_count", "stay_days", add=1)
    ratio("medications_per_stay", "medication_count", "stay_days", add=1)
    ratio("diagnoses_per_stay", "diagnosis_count", "stay_days", add=1)
    ratio("prior_ed_fraction", "total_ed_visits", "previous_visits", add=1)
    ratio("prior_ip_fraction", "total_ip_visits", "previous_visits", add=1)
    ratio("recent_visit_fraction", "visits_30d", "visits_365d", add=1)
    ratio("icu_fraction_stay", "total_icu_hours", "total_unit_hours", add=1)
    difference("lab_creatinine_vs_first", "lab_creatinine_last", "lab_creatinine_first")
    difference("lab_hemoglobin_vs_first", "lab_hemoglobin_last", "lab_hemoglobin_first")

    # count how many numeric measurements are missing for each encounter
    num_before = both.select_dtypes(include=[np.number]).columns
    both["numeric_missing_count"] = both[num_before].isna().sum(axis=1)

    # flag a few common high-risk measurements
    for feature, source, limit, direction in [
        ("low_hemoglobin_last", "lab_hemoglobin_last", 10, "low"),
        ("high_creatinine_last", "lab_creatinine_last", 1.5, "high"),
        ("low_sodium_last", "lab_sodium_last", 135, "low"),
        ("high_bun_last", "lab_bun_last", 25, "high"),
        ("low_spo2_last", "vital_spo2_last", 92, "low")
    ]:
        val = numeric(source)
        if val is not None:
            both[feature] = np.where(val.notna(), (val < limit if direction == "low" else val > limit).astype(int), np.nan)

    # these V4 missing-unit counts were built from unfiltered raw labs
    # leave them out here so post-discharge records cannot affect this experiment
    possibly_late = ["labs_missing_units", "labs_total_selected", "labs_missing_units_fraction"]
    both = both.drop(columns=possibly_late, errors="ignore")

    added = sorted(set(both.columns) - original_columns)
    X6 = both.iloc[:len(train)].reset_index(drop=True).copy()
    T6 = both.iloc[len(train):].reset_index(drop=True).copy()
    assert len(T6) == len(test)
    assert X6.columns.equals(T6.columns)
    assert X6.columns.is_unique


### Archive M26 — V6 variable diagnostics

**Original source cell 26; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    print("Checking notebook variables")

    for name in ["X_train_v4_ready", "X_test_v4_ready",
                 "oof", "test_preds", "results6"]:
        print(name, "FOUND" if name in globals() else "MISSING")

    if "results6" in globals():
        print(pd.DataFrame(results6).sort_values(
            "oof_auc", ascending=False
        ).to_string(index=False))


### Archive M27 — V6 retry

**Original source cell 27; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution
    # V6 - extra features and two new models

    print("V6 STARTED - checking data", flush=True)

    import numpy as np
    import pandas as pd
    from catboost import CatBoostClassifier
    from xgboost import XGBClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder

    needed = ["X_train_v4_ready", "X_test_v4_ready", "cat_cols_v4", "master", "train", "test", "oof", "test_preds"]
    missing = [name for name in needed if name not in globals()]
    print("Missing inputs:", missing if missing else "none", flush=True)
    if missing:
        raise RuntimeError("Run the V4 and V5 cells first. Missing: " + str(missing))

    # use the V4 features and keep the same train/test row order
    X6 = X_train_v4_ready.reset_index(drop=True).copy()
    T6 = X_test_v4_ready.reset_index(drop=True).copy()
    y6 = train["readmit_30d"].to_numpy(dtype=int)
    assert X6.shape == (76975, 178), X6.shape
    assert T6.shape == (32812, 178), T6.shape
    assert X6.columns.equals(T6.columns)

    print("V4 training and test tables found:", X6.shape, T6.shape, flush=True)
    both = pd.concat([X6, T6], ignore_index=True)
    original_columns = set(both.columns)

    print("Building V6 features...", flush=True)
    # add calendar information from the index admission and discharge
    ids = pd.concat([train["encounter_id"], test["encounter_id"]], ignore_index=True).astype(str)
    enc_dates = master.copy()
    enc_dates["encounter_id"] = enc_dates["encounter_id"].astype(str)
    assert enc_dates["encounter_id"].is_unique
    enc_dates = enc_dates.set_index("encounter_id").reindex(ids)
    for date_name in ["admit_dt", "discharge_dt"]:
        dates = pd.to_datetime(enc_dates[date_name], errors="coerce")
        both[date_name + "_hour"] = dates.dt.hour.to_numpy(dtype=float)
        both[date_name + "_month"] = dates.dt.month.to_numpy(dtype=float)
        both[date_name + "_year"] = dates.dt.year.to_numpy(dtype=float)
        both[date_name + "_weekday"] = dates.dt.dayofweek.to_numpy(dtype=float)
        both[date_name + "_weekend"] = (dates.dt.dayofweek >= 5).where(dates.notna()).to_numpy(dtype=float)

    # make some simple combinations of the existing medical measurements
    def numeric(col):
        return pd.to_numeric(both[col], errors="coerce") if col in both else None

    def ratio(name, top, bottom, add=0.0):
        a, b = numeric(top), numeric(bottom)
        if a is not None and b is not None:
            both[name] = (a / (b + add).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

    def difference(name, left, right):
        a, b = numeric(left), numeric(right)
        if a is not None and b is not None:
            both[name] = a - b

    ratio("bun_creatinine_ratio", "lab_bun_last", "lab_creatinine_last")
    ratio("shock_index_last", "vital_hr_last", "vital_sbp_last")
    ratio("lab_tests_per_stay", "lab_total_count", "stay_days", add=1)
    ratio("medications_per_stay", "medication_count", "stay_days", add=1)
    ratio("diagnoses_per_stay", "diagnosis_count", "stay_days", add=1)
    ratio("prior_ed_fraction", "total_ed_visits", "previous_visits", add=1)
    ratio("prior_ip_fraction", "total_ip_visits", "previous_visits", add=1)
    ratio("recent_visit_fraction", "visits_30d", "visits_365d", add=1)
    ratio("icu_fraction_stay", "total_icu_hours", "total_unit_hours", add=1)
    difference("lab_creatinine_vs_first", "lab_creatinine_last", "lab_creatinine_first")
    difference("lab_hemoglobin_vs_first", "lab_hemoglobin_last", "lab_hemoglobin_first")

    # count how many numeric measurements are missing for each encounter
    num_before = both.select_dtypes(include=[np.number]).columns
    both["numeric_missing_count"] = both[num_before].isna().sum(axis=1)

    # flag a few common high-risk measurements
    for feature, source, limit, direction in [
        ("low_hemoglobin_last", "lab_hemoglobin_last", 10, "low"),
        ("high_creatinine_last", "lab_creatinine_last", 1.5, "high"),
        ("low_sodium_last", "lab_sodium_last", 135, "low"),
        ("high_bun_last", "lab_bun_last", 25, "high"),
        ("low_spo2_last", "vital_spo2_last", 92, "low")
    ]:
        val = numeric(source)
        if val is not None:
            both[feature] = np.where(val.notna(), (val < limit if direction == "low" else val > limit).astype(int), np.nan)

    # these V4 missing-unit counts were built from unfiltered raw labs
    # leave them out here so post-discharge records cannot affect this experiment
    possibly_late = ["labs_missing_units", "labs_total_selected", "labs_missing_units_fraction"]
    both = both.drop(columns=possibly_late, errors="ignore")

    added = sorted(set(both.columns) - original_columns)
    X6 = both.iloc[:len(train)].reset_index(drop=True).copy()


### Archive M28 — V6 output checks

**Original source cell 28; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import os

    filename = "/kaggle/working/v5_cb6_xgb_50_50.csv"

    print("File exists:", os.path.exists(filename))
    print("CSV files available:")

    for file in os.listdir("/kaggle/working"):
        if file.endswith(".csv"):
            print(file)


### Archive M29 — V6 file inspection

**Original source cell 29; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import os

    file = "/kaggle/working/v5_cb6_xgb_50_50.csv"

    print("File found:", os.path.isfile(file))

    if os.path.isfile(file):
        print("File size:", os.path.getsize(file), "bytes")
        print(pd.read_csv(file).head())


### Archive M30 — V6 file inspection

**Original source cell 30; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution
    import os
    import pandas as pd

    # check if the submission file is saved
    path = "/kaggle/working/submission_v4.csv"

    print("File found:", os.path.exists(path))

    if os.path.exists(path):
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print(df.head())


### Archive M31 — V6 download

**Original source cell 31; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import os
    import base64
    from IPython.display import HTML, display

    # get the V5 submission file
    path = "/kaggle/working/v5_cb6_xgb_50_50.csv"

    if os.path.isfile(path):
        with open(path, "rb") as f:
            data = base64.b64encode(f.read()).decode()

        display(HTML(
            f'<a download="v5_cb6_xgb_50_50.csv" '
            f'href="data:text/csv;base64,{data}">'
            f'CLICK HERE TO DOWNLOAD V5 50/50 CSV</a>'
        ))
    else:
        print("V5 file missing. Do not submit V4 instead.")


### Archive M32 — V6 result check

**Original source cell 32; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    if "results6" in globals():
        print(
            pd.DataFrame(results6)
            .sort_values("oof_auc", ascending=False)
            .to_string(index=False)
        )
    else:
        print("V6 has not completed successfully yet")


### Archive M33 — V6 environment check

**Original source cell 33; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import os

    # check which data is still in memory
    for name in [
        "X_train_v4_ready",
        "X_test_v4_ready",
        "train",
        "test",
        "oof",
        "test_preds",
        "X6",
        "both",
        "enc_dates",
        "results6"
    ]:
        print(name, "YES" if name in globals() else "NO")

    # check whether V6 saved anything
    files = [
        f for f in os.listdir("/kaggle/working")
        if f.startswith("v6_") and f.endswith(".csv")
    ]
    print("V6 CSV files:", files)


### Archive M34 — V6 fast retraining

**Original source cell 34; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import numpy as np
    import pandas as pd
    from catboost import CatBoostClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score

    # use the data already have
    X = X_train_v4_ready.reset_index(drop=True).copy()
    T = X_test_v4_ready.reset_index(drop=True).copy()
    y = train["readmit_30d"].to_numpy(dtype=int)

    assert len(X) == len(y)
    assert len(T) == len(test)
    assert X.columns.equals(T.columns)

    print("V6 FAST STARTED", flush=True)

    # combine train and test to create the same features
    all_data = pd.concat([X, T], ignore_index=True)

    # get admission and discharge dates
    dates = master[["encounter_id", "admit_dt", "discharge_dt"]].copy()
    dates["encounter_id"] = dates["encounter_id"].astype(str)
    dates = dates.set_index("encounter_id")

    ids = pd.concat([
        train["encounter_id"],
        test["encounter_id"]
    ], ignore_index=True).astype(str)

    dates = dates.reindex(ids)

    for col in ["admit_dt", "discharge_dt"]:
        d = pd.to_datetime(dates[col], errors="coerce")
        all_data[col + "_hour"] = d.dt.hour.to_numpy(dtype=float)
        all_data[col + "_weekday"] = d.dt.dayofweek.to_numpy(dtype=float)
        all_data[col + "_month"] = d.dt.month.to_numpy(dtype=float)

    # add medical ratios
    ratios = [
        ("bun_creatinine_ratio", "lab_bun_last", "lab_creatinine_last"),
        ("shock_index", "vital_hr_last", "vital_sbp_last"),
        ("recent_visit_ratio", "visits_30d", "visits_365d"),
        ("icu_stay_ratio", "total_icu_hours", "total_unit_hours"),
        ("medications_per_day", "medication_count", "stay_days")
    ]

    for name, a, b in ratios:
        if a in all_data.columns and b in all_data.columns:
            top = pd.to_numeric(all_data[a], errors="coerce")
            bottom = pd.to_numeric(all_data[b], errors="coerce")
            all_data[name] = top / (bottom + 1)

    # add a few risk flags
    flags = [
        ("low_oxygen", "vital_spo2_last", 92, "low"),
        ("high_creatinine", "lab_creatinine_last", 1.5, "high"),
        ("low_hemoglobin", "lab_hemoglobin_last", 10, "low")
    ]

    for name, col, limit, direction in flags:
        if col in all_data.columns:
            value = pd.to_numeric(all_data[col], errors="coerce")
            condition = value < limit if direction == "low" else value > limit
            all_data[name] = np.where(value.notna(), condition.astype(int), np.nan)

    # remove three potentially unsafe lab features
    all_data = all_data.drop(columns=[
        "labs_missing_units",
        "labs_total_selected",
        "labs_missing_units_fraction"
    ], errors="ignore")

    X_new = all_data.iloc[:len(X)].reset_index(drop=True).copy()
    T_new = all_data.iloc[len(X):].reset_index(drop=True).copy()

    cats = [c for c in cat_cols_v4 if c in X_new.columns]

    for col in cats:
        X_new[col] = X_new[col].fillna("Missing").astype(str)
        T_new[col] = T_new[col].fillna("Missing").astype(str)

    print("FEATURES READY:", X_new.shape, flush=True)

    # train three folds
    folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    new_oof = np.zeros(len(X_new))
    new_test = np.zeros(len(T_new))

    for fold, (tr, va) in enumerate(folds.split(X_new, y), start=1):
        print("TRAINING FOLD", fold, "OF 3", flush=True)

        model = CatBoostClassifier(
            iterations=900,
            depth=7,
            learning_rate=0.035,
            l2_leaf_reg=8,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=52 + fold,
            thread_count=-1,
            verbose=False,
            allow_writing_files=False
        )

        model.fit(
            X_new.iloc[tr], y[tr],
            cat_features=cats,
            eval_set=(X_new.iloc[va], y[va]),
            early_stopping_rounds=80,
            verbose=False
        )

        new_oof[va] = model.predict_proba(X_new.iloc[va])[:, 1]
        new_test += model.predict_proba(T_new)[:, 1] / 3

        print(
            "FOLD", fold,
            "AUC:", round(roc_auc_score(y[va], new_oof[va]), 5),
            flush=True
        )

    # compare with our best V5 model
    old_oof = oof["blend_cb6_xgb"]
    old_test = test_preds["blend_cb6_xgb"]

    blend_oof = 0.7 * old_oof + 0.3 * new_oof
    blend_test = 0.7 * old_test + 0.3 * new_test

    print("\nFINAL COMPARISON")
    print("V5 original:", round(roc_auc_score(y, old_oof), 6))
    print("V6 new:", round(roc_auc_score(y, new_oof), 6))
    print("V6 blend:", round(roc_auc_score(y, blend_oof), 6))

    # save both new submissions
    for name, predictions in [
        ("v6_fast_cat7", new_test),
        ("v6_fast_blend", blend_test)
    ]:
        submission = pd.DataFrame({
            "encounter_id": test["encounter_id"].to_numpy(),
            "readmit_30d": predictions
        })

        assert len(submission) == 32812
        assert submission["encounter_id"].is_unique
        assert np.isfinite(predictions).all()
        assert submission["readmit_30d"].between(0, 1).all()

        submission.to_csv(
            "/kaggle/working/" + name + ".csv",
            index=False
        )

        print("SAVED:", name + ".csv", flush=True)

    print("V6 FAST COMPLETE", flush=True)


### Archive M35 — V8 longitudinal features and models

**Original source cell 35; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import gc
    import re
    from pathlib import Path
    import numpy as np
    import pandas as pd
    from catboost import CatBoostClassifier
    from lightgbm import LGBMClassifier, early_stopping
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score

    # use the existing notebook data
    X0 = X_train_v4_ready.reset_index(drop=True).copy()
    T0 = X_test_v4_ready.reset_index(drop=True).copy()
    y = train["readmit_30d"].to_numpy(dtype=int)

    assert X0.columns.equals(T0.columns)
    assert len(X0) == len(y)
    assert len(T0) == len(test)

    ids = pd.Index(
        pd.concat([
            train["encounter_id"],
            test["encounter_id"]
        ], ignore_index=True).astype(str)
    )

    A = pd.concat([X0, T0], ignore_index=True)
    print("V8 STARTED:", A.shape, flush=True)

    # find the Kaggle CSV files automatically
    def read_csv(name, cols=None):
        paths = list(Path("/kaggle/input").rglob(name))
        if not paths:
            raise FileNotFoundError("Missing Kaggle input file: " + name)
        return pd.read_csv(paths[0], usecols=cols)

    # add new features in the correct row order
    def add(table):
        global A
        table = table.copy()
        table.index = table.index.astype(str)
        table = table[~table.index.duplicated()]
        table = table.loc[:, ~table.columns.isin(A.columns)]
        A = pd.concat([
            A,
            table.reindex(ids).reset_index(drop=True)
        ], axis=1)

    # admission and discharge dates
    enc = master[
        ["encounter_id", "patient_id", "admit_dt", "discharge_dt"]
    ].drop_duplicates("encounter_id").copy()

    enc["encounter_id"] = enc["encounter_id"].astype(str)
    enc["patient_id"] = enc["patient_id"].astype(str)

    for c in ["admit_dt", "discharge_dt"]:
        enc[c] = pd.to_datetime(
            enc[c], errors="coerce", format="mixed"
        )

    dates = enc.set_index("encounter_id").reindex(ids)

    for name in ["admit_dt", "discharge_dt"]:
        dt = dates[name]
        A[name + "_hour_v8"] = dt.dt.hour.to_numpy()
        A[name + "_dow_v8"] = dt.dt.dayofweek.to_numpy()
        A[name + "_month_v8"] = dt.dt.month.to_numpy()

    # patient history before admission
    h = read_csv(
        "prior_utilization.csv",
        ["patient_id", "visit_dt", "visit_type"]
    )

    h["patient_id"] = h["patient_id"].astype(str)

    h = h.merge(
        enc[["patient_id", "encounter_id", "admit_dt"]],
        on="patient_id",
        how="inner"
    )

    h["visit_dt"] = pd.to_datetime(
        h["visit_dt"], errors="coerce", format="mixed"
    )

    h["days_ago"] = (
        h["admit_dt"] - h["visit_dt"]
    ).dt.total_seconds() / 86400

    h = h[h["days_ago"].ge(0)].copy()

    h["visit_type"] = (
        h["visit_type"].fillna("").astype(str)
        .str.upper().str.strip()
    )

    h["visit_type"] = h["visit_type"].replace({
        "ER": "ED",
        "EMERGENCY": "ED",
        "INPATIENT": "IP"
    })

    hf = h.groupby("encounter_id")["days_ago"].agg(
        ["min", "mean", "std", "count"]
    )

    hf.columns = ["v8_history_" + c for c in hf.columns]

    for days in [7, 30, 90, 180, 365, 730]:
        mask = h["days_ago"].le(days)

        hf[f"v8_visits_{days}d"] = (
            h[mask].groupby("encounter_id").size()
        )

        for typ in ["ED", "IP"]:
            hf[f"v8_{typ}_{days}d"] = (
                h[mask & h["visit_type"].eq(typ)]
                .groupby("encounter_id").size()
            )

    add(hf)
    del h, hf

    print("HISTORY READY", flush=True)

    # common diagnosis codes and groups
    d = read_csv(
        "diagnoses.csv",
        ["encounter_id", "icd10_code"]
    )

    d["encounter_id"] = d["encounter_id"].astype(str)

    d["icd10_code"] = (
        d["icd10_code"].fillna("UNK")
        .astype(str).str.upper().str.strip()
    )

    d["group"] = (
        d["icd10_code"]
        .str.replace(".", "", regex=False).str[:3]
    )

    for key, prefix, n in [
        ("icd10_code", "v8_dx", 45),
        ("group", "v8_group", 55)
    ]:
        top = d[key].value_counts().head(n).index
        sub = d[d[key].isin(top)]

        t = pd.crosstab(
            sub["encounter_id"], sub[key]
        )

        t.columns = [
            prefix + "_" + str(i)
            for i in range(len(t.columns))
        ]

        add(t)

    del d, t

    # medication classes
    med = read_csv(
        "medications.csv",
        ["encounter_id", "drug_class", "continued_at_discharge"]
    )

    med["encounter_id"] = med["encounter_id"].astype(str)

    med["drug_class"] = (
        med["drug_class"].fillna("UNK")
        .astype(str).str.upper().str.strip()
    )

    top = med["drug_class"].value_counts().head(35).index

    continued = (
        med["continued_at_discharge"]
        .astype(str).str.lower()
        .isin(["true", "1", "1.0", "yes"])
    )

    for name, sub in [
        ("v8_med", med[med["drug_class"].isin(top)]),
        (
            "v8_med_discharge",
            med[med["drug_class"].isin(top) & continued]
        )
    ]:
        t = pd.crosstab(
            sub["encounter_id"], sub["drug_class"]
        )

        t.columns = [
            name + "_" + str(i)
            for i in range(len(t.columns))
        ]

        add(t)

    del med, sub, t

    print("DIAGNOSES AND MEDICATIONS READY", flush=True)

    # last 24 hours of vital signs
    v = read_csv(
        "vitals.csv",
        ["encounter_id", "charttime", "vital_type", "value"]
    )

    v["encounter_id"] = v["encounter_id"].astype(str)

    v = v.merge(
        enc[["encounter_id", "admit_dt", "discharge_dt"]],
        on="encounter_id",
        how="inner"
    )

    v["charttime"] = pd.to_datetime(
        v["charttime"], errors="coerce", format="mixed"
    )

    v["hours_before"] = (
        v["discharge_dt"] - v["charttime"]
    ).dt.total_seconds() / 3600

    v = v[
        v["hours_before"].between(0, 24)
        & (v["charttime"] >= v["admit_dt"])
    ].copy()

    v["vital_type"] = (
        v["vital_type"].astype(str).str.upper().str.strip()
    )

    v["value"] = pd.to_numeric(
        v["value"], errors="coerce"
    )

    # convert Fahrenheit temperatures
    temp = (
        v["vital_type"].eq("TEMP")
        & v["value"].between(45, 115)
    )

    v.loc[temp, "value"] = (
        v.loc[temp, "value"] - 32
    ) * 5 / 9

    ranges = {
        "HR": (20, 250),
        "SBP": (40, 300),
        "DBP": (20, 200),
        "RR": (4, 70),
        "SPO2": (40, 100),
        "TEMP": (30, 43)
    }

    v = v[v["vital_type"].isin(ranges)].copy()

    for typ, (lo, hi) in ranges.items():
        bad = (
            v["vital_type"].eq(typ)
            & ~v["value"].between(lo, hi)
        )
        v.loc[bad, "value"] = np.nan

    v = v.sort_values("charttime")

    vt = v.groupby(
        ["encounter_id", "vital_type"]
    )["value"].agg(
        ["mean", "std", "min", "max", "last"]
    ).unstack()

    vt.columns = [
        "v8_vital_" + str(typ) + "_" + str(stat)
        for stat, typ in vt.columns
    ]

    add(vt)
    del v, vt

    print("LAST-DAY VITALS READY", flush=True)

    # laboratory abnormalities in the final 24 hours
    lab = read_csv(
        "lab_results.csv",
        [
            "encounter_id",
            "collected_dt",
            "lab_name",
            "abnormal_flag"
        ]
    )

    lab["encounter_id"] = lab["encounter_id"].astype(str)

    lab = lab.merge(
        enc[["encounter_id", "admit_dt", "discharge_dt"]],
        on="encounter_id",
        how="inner"
    )

    lab["collected_dt"] = pd.to_datetime(
        lab["collected_dt"], errors="coerce", format="mixed"
    )

    lab["hours_before"] = (
        lab["discharge_dt"] - lab["collected_dt"]
    ).dt.total_seconds() / 3600

    lab = lab[
        lab["hours_before"].between(0, 24)
        & (lab["collected_dt"] >= lab["admit_dt"])
    ].copy()

    lab["flag"] = (
        lab["abnormal_flag"].fillna("")
        .astype(str).str.upper().str.strip()
    )

    lab["abnormal"] = lab["flag"].isin(["H", "L"]).astype(int)

    lf = lab.groupby("encounter_id").agg(
        v8_lab_count=("lab_name", "size"),
        v8_lab_unique=("lab_name", "nunique"),
        v8_lab_abnormal=("abnormal", "sum")
    )

    lf["v8_lab_abnormal_fraction"] = (
        lf["v8_lab_abnormal"]
        / lf["v8_lab_count"].replace(0, np.nan)
    )

    add(lf)
    del lab, lf

    # prepare the model data
    A = A.drop(
        columns=[
            "labs_missing_units",
            "labs_total_selected",
            "labs_missing_units_fraction"
        ],
        errors="ignore"
    )

    A = A.replace([np.inf, -np.inf], np.nan)
    A["v8_missing_count"] = A.isna().sum(axis=1)

    A.columns = [
        re.sub("[^a-zA-Z0-9_]", "_", str(c))
        for c in A.columns
    ]

    assert A.columns.is_unique, "Duplicate columns found"

    cats = [
        c for c in A
        if not pd.api.types.is_numeric_dtype(A[c])
    ]

    for c in cats:
        A[c] = A[c].fillna("Missing").astype(str)

    CX = A.iloc[:len(X0)].copy()
    CT = A.iloc[len(X0):].copy()

    L = A.copy()

    for c in cats:
        L[c] = L[c].astype("category")

    LX = L.iloc[:len(X0)].copy()
    LT = L.iloc[len(X0):].copy()

    del A, L

    print("TRAINING:", CX.shape, flush=True)

    # use the same validation folds as V5
    folds = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    co = np.zeros(len(y))
    lo = np.zeros(len(y))
    ct = np.zeros(len(T0))
    lt = np.zeros(len(T0))

    for fold, (tr, va) in enumerate(folds.split(CX, y), 1):

        print("FOLD", fold, "CATBOOST", flush=True)

        cb = CatBoostClassifier(
            iterations=1100,
            depth=6,
            learning_rate=0.035,
            l2_leaf_reg=7,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=100 + fold,
            verbose=False,
            allow_writing_files=False
        )

        cb.fit(
            CX.iloc[tr],
            y[tr],
            cat_features=cats,
            eval_set=(CX.iloc[va], y[va]),
            early_stopping_rounds=100
        )

        co[va] = cb.predict_proba(CX.iloc[va])[:, 1]
        ct += cb.predict_proba(CT)[:, 1] / 3

        del cb

        print("FOLD", fold, "LIGHTGBM", flush=True)

        lg = LGBMClassifier(
            n_estimators=1400,
            learning_rate=0.03,
            num_leaves=15,
            min_child_samples=85,
            colsample_bytree=0.82,
            reg_lambda=8,
            random_state=200 + fold,
            verbosity=-1
        )

        lg.fit(
            LX.iloc[tr],
            y[tr],
            eval_set=[(LX.iloc[va], y[va])],
            eval_metric="auc",
            callbacks=[
                early_stopping(90, verbose=False)
            ]
        )

        lo[va] = lg.predict_proba(LX.iloc[va])[:, 1]
        lt += lg.predict_proba(LT)[:, 1] / 3

        del lg

        print(
            "FOLD AUC:",
            round(roc_auc_score(y[va], co[va]), 5),
            round(roc_auc_score(y[va], lo[va]), 5),
            flush=True
        )

        gc.collect()

    # compare against the previous best model
    old_oof = np.asarray(oof["blend_cb6_xgb"])
    old_test = np.asarray(test_preds["blend_cb6_xgb"])

    new_oof = (co + lo) / 2
    new_test = (ct + lt) / 2

    choices = {
        "v8_catboost": (co, ct),
        "v8_lightgbm": (lo, lt),
        "v8_ensemble": (new_oof, new_test),
        "v8_old30_new70": (
            0.3 * old_oof + 0.7 * new_oof,
            0.3 * old_test + 0.7 * new_test
        ),
        "v8_old50_new50": (
            0.5 * old_oof + 0.5 * new_oof,
            0.5 * old_test + 0.5 * new_test
        ),
        "v8_old70_new30": (
            0.7 * old_oof + 0.3 * new_oof,
            0.7 * old_test + 0.3 * new_test
        ),
        "existing_v5": (old_oof, old_test)
    }

    scores = sorted(
        [
            (roc_auc_score(y, val[0]), key)
            for key, val in choices.items()
        ],
        reverse=True
    )

    print("\nMODEL COMPARISON")

    for score, name in scores:
        print(name, round(score, 6))

    score, best = scores[0]

    if best == "existing_v5":
        print(
            "NO OFFLINE IMPROVEMENT. "
            "Keep your existing V5 submission."
        )

    else:
        pred = choices[best][1]

        out = pd.DataFrame({
            "encounter_id": test["encounter_id"].values,
            "readmit_30d": pred
        })

        assert len(out) == len(test)
        assert out["encounter_id"].is_unique
        assert out["readmit_30d"].between(0, 1).all()
        assert out["readmit_30d"].notna().all()

        out.to_csv(
            "/kaggle/working/SUBMIT_THIS.csv",
            index=False
        )

        print(
            "SAVED: /kaggle/working/SUBMIT_THIS.csv",
            best
        )


### Archive M36 — V8 output download

**Original source cell 36; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import os
    import base64
    from IPython.display import HTML, display

    # get the V8 submission file
    path = "/kaggle/working/SUBMIT_THIS.csv"

    if os.path.isfile(path):
        with open(path, "rb") as f:
            data = base64.b64encode(f.read()).decode()

        display(HTML(
            f'<a download="SUBMIT_THIS.csv" '
            f'href="data:text/csv;base64,{data}">'
            f'CLICK HERE TO DOWNLOAD V8 SUBMISSION</a>'
        ))
    else:
        print("File not found. Check that V8 completed successfully.")


### Archive M37 — V9 XGBoost, Random Forest and classroom algorithms

**Original source cell 37; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    # V9 - run this entire cell after V8 has completed
    import base64
    import gc
    import numpy as np
    import pandas as pd

    from IPython.display import HTML, display
    from scipy.sparse import csr_matrix

    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.linear_model import Perceptron
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA, TruncatedSVD

    from xgboost import XGBClassifier

    # check the previous V8 results
    required = [
        "CX", "CT", "co", "lo", "ct", "lt",
        "train", "test", "oof", "test_preds"
    ]

    missing = [name for name in required if name not in globals()]

    if missing:
        raise RuntimeError(
            "Missing V8 variables: " + ", ".join(missing)
            + ". Run the complete V8 cell first."
        )

    n = len(train)
    y9 = train["readmit_30d"].to_numpy(dtype=int)

    assert len(CX) == n
    assert len(CT) == len(test)
    assert CX.columns.equals(CT.columns)

    # recreate the winning V8 blend
    v8_oof = (
        0.30 * np.asarray(oof["blend_cb6_xgb"])
        + 0.35 * np.asarray(co)
        + 0.35 * np.asarray(lo)
    )

    v8_test = (
        0.30 * np.asarray(test_preds["blend_cb6_xgb"])
        + 0.35 * np.asarray(ct)
        + 0.35 * np.asarray(lt)
    )

    assert len(v8_oof) == n
    assert len(v8_test) == len(test)

    base_auc = roc_auc_score(y9, v8_oof)

    print(
        "V9 STARTED | V8 OOF AUC:",
        round(base_auc, 6),
        flush=True
    )

    # add clinical interaction features
    B = pd.concat([CX, CT], ignore_index=True).copy()

    def num(name):
        if name in B:
            return pd.to_numeric(B[name], errors="coerce")
        return None

    def ratio(new, a, b, offset=1.0):
        aa = num(a)
        bb = num(b)

        if aa is not None and bb is not None:
            B[new] = aa / (bb + offset)

    ratio(
        "v9_shock_index",
        "vital_hr_last",
        "vital_sbp_last",
        0.01
    )

    ratio(
        "v9_bun_creatinine",
        "lab_bun_last",
        "lab_creatinine_last",
        0.1
    )

    ratio(
        "v9_recent_visit_share",
        "v8_visits_30d",
        "v8_visits_365d"
    )

    ratio(
        "v9_recent_ip_share",
        "v8_IP_90d",
        "v8_visits_90d"
    )

    ratio(
        "v9_diagnoses_per_day",
        "diagnosis_count",
        "stay_days"
    )

    ratio(
        "v9_abnormal_labs_per_day",
        "v8_lab_abnormal",
        "stay_days"
    )

    # changes in clinical measurements
    changes = [
        (
            "v9_heart_rate_change",
            "v8_vital_HR_last",
            "vital_hr_first"
        ),
        (
            "v9_systolic_change",
            "v8_vital_SBP_last",
            "vital_sbp_first"
        ),
        (
            "v9_oxygen_change",
            "v8_vital_SPO2_last",
            "vital_spo2_first"
        ),
        (
            "v9_bun_change",
            "lab_bun_last",
            "lab_bun_first"
        ),
        (
            "v9_creatinine_change",
            "lab_creatinine_last",
            "lab_creatinine_first"
        )
    ]

    for name, a, b in changes:
        aa = num(a)
        bb = num(b)

        if aa is not None and bb is not None:
            B[name] = aa - bb

    # prepare categorical data for tree models
    categorical = B.select_dtypes(
        exclude=[np.number]
    ).columns.tolist()

    for c in categorical:
        B[c] = B[c].fillna("Missing").astype(str)

        if B[c].nunique() > 150:
            B[c] = B[c].str[:3]

    B = pd.get_dummies(
        B,
        columns=categorical,
        dtype=np.uint8
    )

    B = B.replace([np.inf, -np.inf], np.nan)
    B = B.loc[:, ~B.columns.duplicated()]
    B = B.astype(np.float32)

    X9 = B.iloc[:n].reset_index(drop=True)
    T9 = B.iloc[n:].reset_index(drop=True)

    del B

    print("V9 FEATURES:", X9.shape, flush=True)

    # fill missing values for Random Forest
    medians = X9.median(axis=0).fillna(0)

    XR = X9.fillna(medians).fillna(0)
    TR = T9.fillna(medians).fillna(0)

    # same folds as V8
    folds = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    xgb_oof = np.zeros(n)
    rf_oof = np.zeros(n)

    xgb_test = np.zeros(len(test))
    rf_test = np.zeros(len(test))

    # train boosting and bagging models
    for k, (tr, va) in enumerate(folds.split(X9, y9), 1):

        print(
            "FOLD", k, "/ 3: XGBOOST (boosting)",
            flush=True
        )

        xgb = XGBClassifier(
            n_estimators=1500,
            max_depth=4,
            learning_rate=0.025,
            min_child_weight=15,
            subsample=0.85,
            colsample_bytree=0.82,
            reg_lambda=12,
            reg_alpha=0.5,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            random_state=100 + k,
            n_jobs=4,
            early_stopping_rounds=110
        )

        xgb.fit(
            X9.iloc[tr],
            y9[tr],
            eval_set=[(X9.iloc[va], y9[va])],
            verbose=False
        )

        xgb_oof[va] = xgb.predict_proba(
            X9.iloc[va]
        )[:, 1]

        xgb_test += xgb.predict_proba(
            T9
        )[:, 1] / 3

        print(
            "XGB AUC:",
            round(roc_auc_score(y9[va], xgb_oof[va]), 6),
            flush=True
        )

        del xgb
        gc.collect()

        print(
            "FOLD", k, "/ 3: RANDOM FOREST (bagging)",
            flush=True
        )

        rf = RandomForestClassifier(
            n_estimators=220,
            max_depth=17,
            min_samples_leaf=18,
            max_features=0.55,
            max_samples=0.82,
            bootstrap=True,
            random_state=300 + k,
            n_jobs=4
        )

        rf.fit(XR.iloc[tr], y9[tr])

        rf_oof[va] = rf.predict_proba(
            XR.iloc[va]
        )[:, 1]

        rf_test += rf.predict_proba(
            TR
        )[:, 1] / 3

        print(
            "RF AUC:",
            round(roc_auc_score(y9[va], rf_oof[va]), 6),
            flush=True
        )

        del rf
        gc.collect()

    # benchmark classroom algorithms on one holdout
    print(
        "\nCLASS ALGORITHM CHECK (one holdout only)",
        flush=True
    )

    tr, va = next(folds.split(X9, y9))

    rng = np.random.default_rng(42)

    tr_s = rng.choice(
        tr,
        size=min(7000, len(tr)),
        replace=False
    )

    va_s = rng.choice(
        va,
        size=min(4000, len(va)),
        replace=False
    )

    # Decision Tree with entropy
    dt = DecisionTreeClassifier(
        criterion="entropy",
        max_depth=6,
        min_samples_leaf=100,
        random_state=42
    )

    dt.fit(XR.iloc[tr_s], y9[tr_s])

    print(
        "Decision Tree:",
        round(
            roc_auc_score(
                y9[va_s],
                dt.predict_proba(XR.iloc[va_s])[:, 1]
            ),
            5
        ),
        flush=True
    )

    del dt

    # PCA and Perceptron
    numeric_cols = CX.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    if len(numeric_cols) >= 3:

        Q = CX[numeric_cols].replace(
            [np.inf, -np.inf], np.nan
        )

        Q = Q.fillna(
            Q.iloc[tr_s].median()
        ).fillna(0)

        scale = StandardScaler()

        qtr = scale.fit_transform(
            Q.iloc[tr_s].to_numpy(dtype=np.float32)
        )

        qva = scale.transform(
            Q.iloc[va_s].to_numpy(dtype=np.float32)
        )

        pca = PCA(
            n_components=min(
                20, qtr.shape[1], qtr.shape[0] - 1
            ),
            random_state=42
        )

        ztr = pca.fit_transform(qtr)
        zva = pca.transform(qva)

        perceptron = Perceptron(
            alpha=0.0005,
            penalty="l2",
            max_iter=1500,
            random_state=42,
            tol=1e-3
        )

        perceptron.fit(ztr, y9[tr_s])

        print(
            "PCA + Perceptron:",
            round(
                roc_auc_score(
                    y9[va_s],
                    perceptron.decision_function(zva)
                ),
                5
            ),
            flush=True
        )

        del Q, qtr, qva, ztr, zva, pca, perceptron

    # SVD and kNN
    svd = TruncatedSVD(
        n_components=min(16, XR.shape[1] - 1),
        random_state=42
    )

    s_tr = svd.fit_transform(
        csr_matrix(XR.iloc[tr_s].to_numpy())
    )

    s_va = svd.transform(
        csr_matrix(XR.iloc[va_s].to_numpy())
    )

    s_scale = StandardScaler()

    s_tr = s_scale.fit_transform(s_tr)
    s_va = s_scale.transform(s_va)

    knn = KNeighborsClassifier(
        n_neighbors=75,
        weights="distance",
        n_jobs=4
    )

    knn.fit(s_tr, y9[tr_s])

    print(
        "SVD + kNN:",
        round(
            roc_auc_score(
                y9[va_s],
                knn.predict_proba(s_va)[:, 1]
            ),
            5
        ),
        flush=True
    )

    del svd, knn, s_tr, s_va, s_scale

    # compare real three-fold OOF candidates
    candidates = {
        "v8_original": (
            v8_oof,
            v8_test
        ),
        "v9_xgboost": (
            xgb_oof,
            xgb_test
        ),
        "v9_random_forest": (
            rf_oof,
            rf_test
        ),
        "v9_v8_80_xgb_20": (
            0.80 * v8_oof + 0.20 * xgb_oof,
            0.80 * v8_test + 0.20 * xgb_test
        ),
        "v9_v8_60_xgb_40": (
            0.60 * v8_oof + 0.40 * xgb_oof,
            0.60 * v8_test + 0.40 * xgb_test
        ),
        "v9_v8_40_xgb_60": (
            0.40 * v8_oof + 0.60 * xgb_oof,
            0.40 * v8_test + 0.60 * xgb_test
        ),
        "v9_v8_90_rf_10": (
            0.90 * v8_oof + 0.10 * rf_oof,
            0.90 * v8_test + 0.10 * rf_test
        ),
        "v9_v8_70_rf_30": (
            0.70 * v8_oof + 0.30 * rf_oof,
            0.70 * v8_test + 0.30 * rf_test
        ),
        "v9_v8_60_xgb_30_rf_10": (
            0.60 * v8_oof
            + 0.30 * xgb_oof
            + 0.10 * rf_oof,

            0.60 * v8_test
            + 0.30 * xgb_test
            + 0.10 * rf_test
        )
    }

    ranked = sorted(
        [
            (roc_auc_score(y9, p[0]), name)
            for name, p in candidates.items()
        ],
        reverse=True
    )

    print("\nV9 THREE-FOLD OOF COMPARISON", flush=True)

    for score, name in ranked:
        print(name, round(score, 6), flush=True)

    # require an improvement across at least two folds
    selected = None

    for candidate_auc, candidate_name in ranked:

        if candidate_name == "v8_original":
            continue

        if candidate_auc - base_auc < 0.0003:
            continue

        candidate_gains = []

        for _, va in folds.split(X9, y9):

            gain = (
                roc_auc_score(
                    y9[va],
                    candidates[candidate_name][0][va]
                )
                - roc_auc_score(
                    y9[va],
                    v8_oof[va]
                )
            )

            candidate_gains.append(gain)

        if sum(g > 0 for g in candidate_gains) >= 2:
            selected = (
                candidate_auc,
                candidate_name,
                candidate_gains
            )
            break

    if selected is None:

        print(
            "\nNO CONSISTENT OFFLINE IMPROVEMENT."
            " Keep V8 (public AUC 0.74192).",
            flush=True
        )

    else:

        best_auc, best_name, gains = selected

        print(
            "SELECTED:", best_name,
            "OOF AUC:", round(best_auc, 6),
            flush=True
        )

        print(
            "FOLD IMPROVEMENTS:",
            [round(g, 6) for g in gains],
            flush=True
        )

        # bootstrap the OOF improvement
        rng = np.random.default_rng(2026)
        diffs = []

        for _ in range(100):

            idx = rng.integers(0, n, n)

            if np.unique(y9[idx]).size < 2:
                continue

            new_auc = roc_auc_score(
                y9[idx],
                candidates[best_name][0][idx]
            )

            old_auc = roc_auc_score(
                y9[idx],
                v8_oof[idx]
            )

            diffs.append(new_auc - old_auc)

        print(
            "BOOTSTRAP AUC GAIN 5%-95%:",
            np.round(
                np.percentile(diffs, [5, 95]),
                6
            ),
            flush=True
        )

        # create the final submission
        pred = np.asarray(candidates[best_name][1])

        out = pd.DataFrame({
            "encounter_id": test["encounter_id"].to_numpy(),
            "readmit_30d": pred
        })

        assert len(out) == len(test)
        assert out["encounter_id"].is_unique
        assert np.isfinite(pred).all()
        assert out["readmit_30d"].between(0, 1).all()

        path = "/kaggle/working/SUBMIT_V9.csv"

        out.to_csv(path, index=False)

        print(
            "\nSAVED:", path,
            "| MODEL:", best_name,
            flush=True
        )

        # create a direct download link
        with open(path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("ascii")

        display(HTML(
            '<a download="SUBMIT_V9.csv" '
            'href="data:text/csv;base64,'
            + b64
            + '">CLICK HERE TO DOWNLOAD V9 CSV</a>'
        ))


### Archive M38 — V10 comprehensive data engineering

**Original source cell 38; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution
    # v10 dataset combination
    import os, re, gc
    from pathlib import Path
    import numpy as np
    import pandas as pd

    ROOT = Path("/kaggle/input")
    index_paths = list(ROOT.rglob("index_encounters.csv"))
    DATA_DIR = index_paths[0].parent if index_paths else ROOT
    FILES = {p.name.lower(): p for p in DATA_DIR.rglob("*.csv")}

    ALIASES = {
        "diagnoses.csv": "diagnosis.csv",
        "lab_results.csv": "lab_result.csv",
        "procedures.csv": "procedure.csv",
        "medications.csv": "medication.csv",
        "unit_transfers.csv": "unit_transfer.csv"
    }

    def read(name):
        path = FILES.get(name.lower()) or FILES.get(
            ALIASES.get(name.lower(), "")
        )
        if path is None:
            print("MISSING:", name, flush=True)
            return None

        df = pd.read_csv(path, low_memory=False)
        df.columns = df.columns.str.strip()
        print("LOADED:", name, df.shape, flush=True)
        return df

    def pick(df, *options):
        if df is None:
            return None
        lookup = {c.lower(): c for c in df.columns}
        return next(
            (lookup[c.lower()] for c in options if c.lower() in lookup),
            None
        )

    def dates(x):
        try:
            return pd.to_datetime(x, errors="coerce", format="mixed")
        except TypeError:
            return pd.to_datetime(x, errors="coerce")

    def clean_id(df, key):
        df = df.dropna(subset=[key]).copy()
        df[key] = df[key].astype(str).str.strip()
        return df

    def safe(col):
        c = col.lower()
        bad = [
            "readmit", "outcome", "label", "target",
            "future_", "next_", "post_discharge",
            "mortality", "date_of_death", "death_date",
            "30d_", "days_until", "days_to_readmission"
        ]
        return not any(s in c for s in bad)

    tr0 = read("train.csv")
    te0 = read("test.csv")

    assert tr0 is not None and te0 is not None
    assert {"encounter_id", "readmit_30d"}.issubset(tr0.columns)
    assert "encounter_id" in te0
    assert tr0.encounter_id.is_unique
    assert te0.encounter_id.is_unique
    assert tr0.encounter_id.notna().all()
    assert te0.encounter_id.notna().all()

    assert set(tr0.encounter_id.astype(str)).isdisjoint(
        set(te0.encounter_id.astype(str))
    )

    V10_Y = tr0.readmit_30d.astype(int).to_numpy()
    V10_N = len(tr0)
    V10_TEST_IDS = te0.encounter_id.to_numpy()

    A = pd.concat(
        [
            tr0.drop(columns=["readmit_30d"]),
            te0.drop(columns=["readmit_30d"], errors="ignore")
        ],
        ignore_index=True,
        sort=False
    )

    A["encounter_id"] = A.encounter_id.astype(str)
    IDS = pd.Index(A.encounter_id)
    assert IDS.is_unique

    AUDIT = []

    def add_features(features, source):
        global A

        if features is None or features.empty:
            print("NO FEATURES:", source, flush=True)
            return

        features = features.copy()
        features.index = features.index.astype(str)

        assert features.index.is_unique, (
            "Duplicate aggregate keys: " + source
        )

        features = features.loc[
            :, ~features.columns.duplicated()
        ]

        features = features.rename(
            columns=lambda c: re.sub(
                r"[^a-zA-Z0-9_]", "_", str(c)
            )
        )

        features = features.loc[
            :, ~features.columns.isin(A.columns)
        ]

        if features.empty:
            return

        matched = int(IDS.isin(features.index).sum())

        AUDIT.append((
            source, matched, len(IDS), len(features.columns)
        ))

        A = pd.concat(
            [
                A.reset_index(drop=True),
                features.reindex(IDS).reset_index(drop=True)
            ],
            axis=1
        )

        print(
            "ADDED:", source,
            len(features.columns), "features;",
            "coverage", matched, "/", len(IDS),
            flush=True
        )

    def count_categories(d, col, prefix, top=30):
        if col is None or col not in d:
            return None

        s = (
            d[col].fillna("Unknown")
            .astype(str).str.strip().str.upper()
        )

        topvals = s.value_counts().head(top).index

        temp = pd.DataFrame({
            "encounter_id": d.encounter_id.values,
            "cat": s.where(
                s.isin(topvals), "OTHER"
            ).values
        })

        tab = pd.crosstab(temp.encounter_id, temp.cat)

        tab.columns = [
            prefix + "_" +
            re.sub("[^A-Z0-9]+", "_", c)[:35]
            for c in tab.columns
        ]

        tab = tab.loc[:, ~tab.columns.duplicated()]
        return tab.astype("float32")

    def basic_event(d, prefix, categories=(), numeric=()):
        if d is None or "encounter_id" not in d:
            return

        g = d.groupby("encounter_id", sort=False)

        f = pd.DataFrame({
            prefix + "_count": g.size()
        })

        for c, top in categories:
            if c in d:
                f[prefix + "_" + c + "_unique"] = g[c].nunique()

        for c in numeric:
            if c in d:
                d[c] = pd.to_numeric(
                    d[c], errors="coerce"
                )

                stats = g[c].agg(
                    ["mean", "std", "min", "max"]
                )

                stats.columns = [
                    prefix + "_" + c + "_" + s
                    for s in stats.columns
                ]

                f = f.join(stats)

        add_features(f, prefix + "_summary")

        for c, top in categories:
            if c in d:
                add_features(
                    count_categories(
                        d, c, prefix + "_" + c, top
                    ),
                    prefix + "_" + c
                )

    def encounter_events(
        df, source, timestamp_candidates=()
    ):
        if df is None or "encounter_id" not in df:
            print(
                "SKIPPED (no encounter_id):",
                source, flush=True
            )
            return None

        df = clean_id(df, "encounter_id")
        df = df[df.encounter_id.isin(IDS)]

        tcol = pick(df, *timestamp_candidates)

        if tcol is not None:
            df = df.merge(
                V10_DATES,
                on="encounter_id",
                how="left",
                validate="many_to_one"
            )

            t = dates(df[tcol])
            start = df.admit_dt
            end = df.discharge_dt

            good = (
                t.notna()
                & start.notna()
                & end.notna()
                & t.ge(start)
                & t.le(end)
            )

            print(
                "TIME FILTER:", source,
                "kept", int(good.sum()), "/", len(df),
                flush=True
            )

            df = df.loc[good].copy()
            df["_event_dt"] = t.loc[good]

        else:
            print(
                "NO EVENT TIMESTAMP:", source,
                "(using encounter-scoped records only)",
                flush=True
            )

        return df


    # 1. Index encounters

    idx = read("index_encounters.csv")

    if idx is None or "encounter_id" not in idx:
        raise RuntimeError(
            "index_encounters.csv with encounter_id is required"
        )

    idx = clean_id(idx, "encounter_id")

    assert idx.encounter_id.is_unique, (
        "index_encounters.csv must have one row per encounter"
    )

    idx = idx[idx.encounter_id.isin(IDS)]

    admit = pick(
        idx, "admit_dt", "admission_dt",
        "admission_date", "admit_date"
    )

    disch = pick(
        idx, "discharge_dt",
        "discharge_date", "discharged_dt"
    )

    if admit is None or disch is None:
        raise RuntimeError(
            "Cannot locate admission/discharge date columns. "
            + str(idx.columns.tolist())
        )

    idx["admit_dt"] = dates(idx[admit])
    idx["discharge_dt"] = dates(idx[disch])

    V10_DATES = idx[
        ["encounter_id", "admit_dt", "discharge_dt"]
    ].copy()

    V10_DATES = (
        V10_DATES.set_index("encounter_id")
        .reindex(IDS)
        .reset_index()
    )

    V10_DATES.columns = [
        "encounter_id", "admit_dt", "discharge_dt"
    ]

    assert V10_DATES.encounter_id.is_unique

    V10_ADMIT = (
        V10_DATES.admit_dt.iloc[:V10_N]
        .reset_index(drop=True)
    )

    if "hospital_id" in idx:
        V10_HOSP = (
            idx.set_index("encounter_id")
            .reindex(IDS)["hospital_id"]
            .iloc[:V10_N]
            .reset_index(drop=True)
        )
    else:
        V10_HOSP = pd.Series(dtype="object")

    static = idx.drop(
        columns=[
            c for c in idx
            if c in [
                admit, disch, "admit_dt", "discharge_dt"
            ] and c != "encounter_id"
        ],
        errors="ignore"
    )

    static = static.set_index("encounter_id")
    static = static[[c for c in static if safe(c)]]

    add_features(static, "index_encounters")

    A["v10_stay_days"] = (
        V10_DATES.discharge_dt
        - V10_DATES.admit_dt
    ).dt.total_seconds() / 86400

    A["v10_admit_month"] = V10_DATES.admit_dt.dt.month
    A["v10_admit_weekday"] = V10_DATES.admit_dt.dt.dayofweek
    A["v10_admit_hour"] = V10_DATES.admit_dt.dt.hour

    A["v10_discharge_weekday"] = (
        V10_DATES.discharge_dt.dt.dayofweek
    )

    A["v10_discharge_hour"] = (
        V10_DATES.discharge_dt.dt.hour
    )


    # 2. Patients and hospitals

    for fname, key, pref in [
        ("patients.csv", "patient_id", "patient"),
        ("hospitals.csv", "hospital_id", "hospital")
    ]:
        df = read(fname)

        if df is None or key not in df or key not in A:
            print(
                "SKIPPED STATIC JOIN:",
                fname, flush=True
            )
            continue

        df = clean_id(df, key)

        if df[key].duplicated().any():
            print(
                "WARNING: duplicate keys in",
                fname, "- keeping first record",
                flush=True
            )
            df = df.drop_duplicates(key)

        use = [
            c for c in df
            if c != key and safe(c)
        ]

        if pref == "patient":
            birth = pick(
                df, "dob", "date_of_birth", "birth_date"
            )

            if birth is not None:
                birth_map = df.set_index(key)[birth]

                b = dates(
                    A[key].astype(str).map(birth_map)
                )

                A["v10_age_from_dob"] = (
                    V10_DATES.admit_dt - b
                ).dt.days / 365.25

                A.loc[
                    ~A["v10_age_from_dob"].between(0, 115),
                    "v10_age_from_dob"
                ] = np.nan

        use = [
            c for c in use
            if not re.search(
                "date|_dt$|timestamp|dob|birth",
                c, re.I
            )
        ]

        part = (
            df.set_index(key)[use]
            .add_prefix("v10_" + pref + "_")
        )

        part.index = part.index.astype(str)

        add_features(
            part.reindex(
                A[key].astype(str)
            ).set_axis(IDS),
            fname
        )

        del df, part


    # 3. Diagnoses and ICD-10 hierarchy

    D = read("diagnoses.csv")
    D = encounter_events(D, "diagnoses")

    if D is not None and len(D):
        code = pick(
            D, "icd10_code", "icd_10_code",
            "diagnosis_code", "code"
        )

        if code:
            D["code3"] = (
                D[code].fillna("UNK")
                .astype(str).str.upper()
                .str.replace(".", "", regex=False)
                .str[:3]
            )

            D["icd_chapter"] = D["code3"].str[0]

            lookup = read("icd_10_lookup.csv")

            if lookup is not None:
                lkcode = pick(
                    lookup, "icd10_code",
                    "icd_10_code", "code"
                )

                category = pick(
                    lookup, "category", "chapter",
                    "diagnosis_group", "group"
                )

                if lkcode and category:
                    lk = (
                        lookup[[lkcode, category]]
                        .drop_duplicates(lkcode)
                    )

                    lk[lkcode] = lk[lkcode].astype(str)
                    D[code] = D[code].astype(str)

                    D = D.merge(
                        lk,
                        left_on=code,
                        right_on=lkcode,
                        how="left",
                        validate="many_to_one"
                    )

                    if category in D:
                        D["lookup_group"] = D[category]

            basic_event(
                D, "v10_dx",
                [
                    (code, 35),
                    ("code3", 45),
                    ("icd_chapter", 22),
                    ("lookup_group", 20)
                ]
            )

        else:
            basic_event(D, "v10_dx")

        del D
        gc.collect()


    # 4. Procedures

    D = encounter_events(
        read("procedures.csv"),
        "procedures",
        (
            "performed_dt", "procedure_dt",
            "procedure_date", "event_dt"
        )
    )

    if D is not None and len(D):
        code = pick(
            D, "procedure_code", "cpt_code",
            "icd10_pcs", "code",
            "procedure_type", "procedure_name",
            "category"
        )

        basic_event(
            D, "v10_proc",
            [(code, 35)] if code else []
        )

        del D
        gc.collect()


    # 5. Medications

    D = encounter_events(
        read("medications.csv"),
        "medications",
        (
            "administered_dt", "ordered_dt",
            "start_dt", "medication_dt", "event_dt"
        )
    )

    if D is not None and len(D):
        med = pick(
            D, "drug_class", "med_class",
            "medication_class", "therapeutic_class"
        )

        name = pick(
            D, "drug_name",
            "medication_name", "med_name"
        )

        continued = pick(
            D, "continued_at_discharge",
            "discharge_medication"
        )

        basic_event(
            D, "v10_med",
            [(med, 40)] if med else []
        )

        if name:
            add_features(
                count_categories(
                    D, name, "v10_med_name", 25
                ),
                "medication_names"
            )

        if continued:
            f = (
                D[continued].fillna("")
                .astype(str).str.lower()
                .isin(["1", "true", "yes", "y"])
            )

            add_features(
                f.groupby(D.encounter_id)
                .sum()
                .to_frame("v10_meds_continued"),
                "meds_at_discharge"
            )

        del D
        gc.collect()


    # 6. Unit transfers and ICU exposure

    D = encounter_events(
        read("unit_transfers.csv"),
        "unit_transfers",
        (
            "transfer_dt", "transfer_time",
            "start_dt", "event_dt"
        )
    )

    if D is not None and len(D):
        unit = pick(
            D, "unit_type", "unit_name",
            "to_unit", "from_unit",
            "care_unit", "unit"
        )

        basic_event(
            D, "v10_transfer",
            [(unit, 20)] if unit else []
        )

        if unit:
            icu = (
                D[unit].fillna("")
                .astype(str)
                .str.contains(
                    "ICU|INTENSIVE|CRITICAL",
                    case=False, regex=True
                )
            )

            add_features(
                icu.groupby(D.encounter_id)
                .sum()
                .to_frame("v10_icu_transfer_count"),
                "icu_transfers"
            )

        del D
        gc.collect()


    # 7. Discharge planning

    D = encounter_events(
        read("discharge_planning.csv"),
        "discharge_planning",
        (
            "created_dt", "recorded_dt",
            "plan_dt", "event_dt"
        )
    )

    if D is not None and len(D):
        cat = [
            (c, 12)
            for c in D.columns
            if c != "encounter_id"
            and safe(c)
            and not re.search(
                "date|_dt|time|id|note|text|description",
                c, re.I
            )
            and 1 < D[c].nunique(dropna=True) <= 40
        ][:10]

        nums = [
            c for c in D.select_dtypes(
                include="number"
            ).columns
            if safe(c)
            and not re.search(
                "id|target|label|readmit", c, re.I
            )
        ][:8]

        basic_event(
            D, "v10_plan", cat, nums
        )

        del D
        gc.collect()


    # 8. Prior utilization: only visits before admission

    D = read("prior_utilization.csv")

    if (
        D is not None
        and "patient_id" in D
        and "patient_id" in A
    ):
        tcol = pick(
            D, "visit_dt", "visit_date",
            "encounter_dt", "admit_dt"
        )

        if tcol:
            pid = A[
                ["encounter_id", "patient_id"]
            ].copy()

            pid["admit_dt"] = V10_DATES.admit_dt.values
            pid = clean_id(pid, "patient_id")

            D = clean_id(D, "patient_id")
            D["_visit"] = dates(D[tcol])

            D = D.merge(
                pid, on="patient_id", how="inner"
            )

            D["days_ago"] = (
                D.admit_dt - D._visit
            ).dt.total_seconds() / 86400

            D = D[
                D.days_ago.gt(0)
                & D.days_ago.le(3650)
            ].copy()

            print(
                "HISTORY strictly before admission:",
                len(D), flush=True
            )

            g = D.groupby("encounter_id")

            f = pd.DataFrame({
                "v10_history_total": g.size(),
                "v10_history_last_days": g.days_ago.min(),
                "v10_history_first_days": g.days_ago.max()
            })

            for w in (7, 30, 90, 180, 365, 730):
                f["v10_history_" + str(w) + "d"] = (
                    D.days_ago.le(w)
                    .groupby(D.encounter_id).sum()
                )

            add_features(
                f, "prior_utilization"
            )

            kind = pick(
                D, "visit_type",
                "encounter_type", "care_setting"
            )

            if kind:
                add_features(
                    count_categories(
                        D, kind, "v10_prior_type", 10
                    ),
                    "prior_visit_type"
                )

            del pid

        else:
            print(
                "SKIPPED HISTORY: no visit date (unsafe)",
                flush=True
            )

        del D
        gc.collect()


    # 9. Labs: abnormalities, test types and trends

    D = encounter_events(
        read("lab_results.csv"),
        "lab_results",
        (
            "collected_dt", "collection_dt",
            "result_dt", "lab_dt", "charttime"
        )
    )

    if D is not None and len(D):
        name = pick(
            D, "lab_name", "test_name",
            "lab_test", "analyte"
        )

        val = pick(
            D, "value", "result_value",
            "numeric_value", "lab_value", "result"
        )

        flag = pick(
            D, "abnormal_flag", "flag",
            "abnormality"
        )

        if flag:
            f = (
                D[flag].fillna("")
                .astype(str).str.upper().str.strip()
            )

            D["_abnormal"] = f.isin([
                "H", "L", "HH", "LL", "HIGH", "LOW",
                "ABNORMAL", "ABN", "A",
                "CRITICAL", "CRIT"
            ]).astype("int8")

            D["_high"] = f.isin(
                ["H", "HIGH"]
            ).astype("int8")

            D["_low"] = f.isin(
                ["L", "LOW"]
            ).astype("int8")

        else:
            D["_abnormal"] = 0
            D["_high"] = 0
            D["_low"] = 0

        g = D.groupby("encounter_id")

        f = g[
            ["_abnormal", "_high", "_low"]
        ].sum().rename(columns={
            "_abnormal": "v10_lab_abnormal",
            "_high": "v10_lab_high",
            "_low": "v10_lab_low"
        })

        f["v10_lab_count"] = g.size()

        f["v10_lab_abnormal_rate"] = (
            f.v10_lab_abnormal / f.v10_lab_count
        )

        if name:
            f["v10_lab_unique"] = g[name].nunique()

        add_features(
            f, "lab_summary"
        )

        if name:
            add_features(
                count_categories(
                    D, name, "v10_lab_type", 25
                ),
                "lab_test_counts"
            )

            if val:
                D["_num"] = pd.to_numeric(
                    D[val], errors="coerce"
                )

                D[name] = (
                    D[name].astype(str)
                    .str.upper().str.strip()
                )

                top = (
                    D[name].value_counts()
                    .head(22).index
                )

                small = D[
                    D[name].isin(top)
                    & D._num.notna()
                ].copy()

                if len(small):
                    if "_event_dt" in small:
                        small = small.sort_values(
                            "_event_dt"
                        )

                    g2 = small.groupby(
                        ["encounter_id", name]
                    )["_num"]

                    stats = g2.agg([
                        "mean", "min", "max",
                        "first", "last"
                    ])

                    stats["delta"] = (
                        stats["last"] - stats["first"]
                    )

                    wide = stats.unstack(name)

                    wide.columns = [
                        "v10_lab_" + str(lab)[:20]
                        + "_" + stat
                        for stat, lab in wide.columns
                    ]

                    add_features(
                        wide, "lab_numeric_trajectories"
                    )

                    del small, stats, wide

        del D
        gc.collect()


    # 10. Vitals: full-stay and early/late trajectories

    D = encounter_events(
        read("vitals.csv"),
        "vitals",
        (
            "charttime", "recorded_dt",
            "vital_dt", "event_dt"
        )
    )

    if D is not None and len(D):
        typ = pick(
            D, "vital_type",
            "vital_name", "measurement_type"
        )

        val = pick(
            D, "value", "vital_value",
            "numeric_value"
        )

        if typ and val:
            D[typ] = (
                D[typ].fillna("UNK")
                .astype(str).str.upper().str.strip()
            )

            D["_num"] = pd.to_numeric(
                D[val], errors="coerce"
            )

            temp = (
                D[typ].eq("TEMP")
                & D._num.between(45, 115)
            )

            D.loc[temp, "_num"] = (
                D.loc[temp, "_num"] - 32
            ) * 5 / 9

            bounds = {
                "HR": (20, 250),
                "SBP": (40, 300),
                "DBP": (20, 200),
                "RR": (4, 70),
                "SPO2": (40, 100),
                "TEMP": (30, 43)
            }

            for t, (lo, hi) in bounds.items():
                bad = (
                    D[typ].eq(t)
                    & ~D._num.between(lo, hi)
                )
                D.loc[bad, "_num"] = np.nan

            D = D[
                D[typ].isin(bounds)
            ].copy()

            if "_event_dt" in D:
                D = D.sort_values("_event_dt")

            g = D.groupby(
                ["encounter_id", typ]
            )["_num"]

            stats = g.agg([
                "count", "mean", "std",
                "min", "max", "first", "last"
            ])

            stats["change"] = (
                stats["last"] - stats["first"]
            )

            wide = stats.unstack(typ)

            wide.columns = [
                "v10_vital_" + str(t) + "_" + s
                for s, t in wide.columns
            ]

            add_features(
                wide, "vital_full_stay"
            )

            if "_event_dt" in D:
                d_end = D["discharge_dt"]

                D["_last24"] = (
                    d_end - D["_event_dt"]
                ).dt.total_seconds().between(
                    0, 86400
                )

                last = (
                    D[D._last24]
                    .groupby(["encounter_id", typ])["_num"]
                    .mean().unstack(typ)
                )

                last.columns = [
                    "v10_vital_" + str(c)
                    + "_last24mean"
                    for c in last.columns
                ]

                add_features(
                    last, "vital_last24"
                )

                D["_first24"] = (
                    D["_event_dt"] - D["admit_dt"]
                ).dt.total_seconds().between(
                    0, 86400
                )

                first = (
                    D[D._first24]
                    .groupby(["encounter_id", typ])["_num"]
                    .mean().unstack(typ)
                )

                first.columns = [
                    "v10_vital_" + str(c)
                    + "_first24mean"
                    for c in first.columns
                ]

                add_features(
                    first, "vital_first24"
                )

        else:
            basic_event(
                D, "v10_vital"
            )

        del D
        gc.collect()


    # 11. Cross-table clinical interactions

    pairs = [
        ("v10_dx_count", "v10_stay_days", "dx_per_day"),
        ("v10_med_count", "v10_stay_days", "meds_per_day"),
        ("v10_lab_abnormal", "v10_lab_count", "abnormal_fraction"),
        ("v10_history_30d", "v10_history_365d", "recent_visit_share"),
        ("v10_transfer_count", "v10_stay_days", "transfers_per_day"),
        ("v10_proc_count", "v10_stay_days", "procedures_per_day")
    ]

    for a, b, name in pairs:
        if a in A and b in A:
            A["v10_cross_" + name] = (
                pd.to_numeric(A[a], errors="coerce")
                / (
                    pd.to_numeric(A[b], errors="coerce")
                    + 1
                )
            )

    for a, b, name in [
        (
            "v10_vital_HR_last24mean",
            "v10_vital_SBP_last24mean",
            "shock_index"
        ),
        (
            "v10_vital_SPO2_last24mean",
            "v10_vital_RR_last24mean",
            "oxygen_resp_ratio"
        )
    ]:
        if a in A and b in A:
            A["v10_cross_" + name] = (
                pd.to_numeric(A[a], errors="coerce")
                / (
                    pd.to_numeric(A[b], errors="coerce")
                    + 0.1
                )
            )


    # 12. Reuse V8 features when row order is verified

    oldx = globals().get("cat_X", None)
    oldt = globals().get("cat_T", None)

    if not (
        isinstance(oldx, pd.DataFrame)
        and isinstance(oldt, pd.DataFrame)
        and len(oldx) == V10_N
        and len(oldt) == len(te0)
    ):
        oldx = globals().get(
            "X_train_v4_ready", None
        )
        oldt = globals().get(
            "X_test_v4_ready", None
        )

    if (
        isinstance(oldx, pd.DataFrame)
        and isinstance(oldt, pd.DataFrame)
        and len(oldx) == V10_N
        and len(oldt) == len(te0)
    ):
        if (
            "train" in globals()
            and isinstance(train, pd.DataFrame)
            and "encounter_id" in train
        ):
            aligned = (
                train.encounter_id.astype(str)
                .reset_index(drop=True)
                .equals(
                    tr0.encounter_id.astype(str)
                    .reset_index(drop=True)
                )
            )
        else:
            aligned = False

        if aligned:
            old = pd.concat(
                [
                    oldx.reset_index(drop=True),
                    oldt.reset_index(drop=True)
                ],
                ignore_index=True
            )

            old = old.loc[
                :, ~old.columns.duplicated()
            ]

            old = old.rename(
                columns=lambda c: "v8_" + str(c)
            )

            old = old.loc[
                :, [safe(c) for c in old.columns]
            ]

            A = pd.concat(
                [
                    A.reset_index(drop=True),
                    old.reset_index(drop=True)
                ],
                axis=1
            )

            print(
                "REUSED PREVIOUS FEATURES:",
                len(old.columns), flush=True
            )

        else:
            print(
                "OLD FEATURES NOT REUSED:",
                "train row order not verified",
                flush=True
            )


    # 13. Clean the master feature table

    remove = [
        c for c in A
        if c in ["encounter_id", "patient_id"]
        or not safe(c)
        or re.search(
            r"(^|_)(admit_dt|discharge_dt|birth_date|dob|date_of_birth)(_|$)",
            c, re.I
        )
    ]

    A = A.drop(
        columns=remove, errors="ignore"
    )

    A = A.loc[
        :, ~A.columns.duplicated()
    ]

    for c in list(A.columns):
        if pd.api.types.is_datetime64_any_dtype(A[c]):
            A = A.drop(columns=c)
            continue

        if not pd.api.types.is_numeric_dtype(A[c]):
            A[c] = (
                A[c].fillna("Missing")
                .astype(str)
            )

            if (
                A[c].nunique(dropna=False) > 1000
                or A[c].nunique(dropna=False) > 0.4 * len(A)
            ):
                A = A.drop(columns=c)
            else:
                A[c] = A[c].astype("category")

        else:
            A[c] = (
                pd.to_numeric(A[c], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .astype("float32")
            )

    A["v10_missing_count"] = (
        A.isna().sum(axis=1).astype("float32")
    )

    A = A.loc[
        :, A.notna().any(axis=0)
    ]

    A.columns = [
        re.sub("[^a-zA-Z0-9_]", "_", str(c))
        for c in A.columns
    ]

    A = A.loc[
        :, ~A.columns.duplicated()
    ]

    V10_X = (
        A.iloc[:V10_N]
        .reset_index(drop=True)
    )

    V10_T = (
        A.iloc[V10_N:]
        .reset_index(drop=True)
    )

    assert V10_X.columns.equals(V10_T.columns)
    assert len(V10_X) == len(tr0)
    assert len(V10_T) == len(te0)
    assert V10_X.columns.is_unique

    audit = pd.DataFrame(
        AUDIT,
        columns=[
            "source", "matched_encounters",
            "total_encounters", "added_features"
        ]
    )

    audit.to_csv(
        "/kaggle/working/V10_DATA_AUDIT.csv",
        index=False
    )

    print(
        "\nV10 MASTER READY:",
        V10_X.shape,
        "TEST:", V10_T.shape,
        flush=True
    )

    print(
        "FILES FOUND:", len(FILES),
        "| SOURCES ADDED:", len(AUDIT),
        flush=True
    )

    print(audit.to_string(index=False))

    del A
    gc.collect()


### Archive M39 — V10 validation and stress tests

**Original source cell 39; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import base64, gc
    import numpy as np
    import pandas as pd

    from IPython.display import display, HTML
    from sklearn.model_selection import (
        StratifiedKFold, GroupShuffleSplit
    )
    from sklearn.metrics import (
        roc_auc_score, average_precision_score
    )
    from catboost import CatBoostClassifier
    from lightgbm import (
        LGBMClassifier, early_stopping, log_evaluation
    )

    assert (
        "V10_X" in globals()
        and "V10_T" in globals()
    ), "Run V10 Cell 1 first."

    X = V10_X.copy()
    T = V10_T.copy()
    y = V10_Y

    cats = X.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for c in cats:
        combined = pd.concat(
            [X[c], T[c]],
            ignore_index=True
        ).astype("object").fillna("Missing").astype(str)

        categories = pd.Categorical(combined).categories

        X[c] = pd.Categorical(
            X[c].astype("object")
            .fillna("Missing").astype(str),
            categories=categories
        )

        T[c] = pd.Categorical(
            T[c].astype("object")
            .fillna("Missing").astype(str),
            categories=categories
        )

    # CatBoost uses strings; LightGBM uses categories.
    CX = X.copy()
    CT = T.copy()

    for c in cats:
        CX[c] = CX[c].astype(str)
        CT[c] = CT[c].astype(str)

    folds = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    co = np.zeros(len(y))
    lo = np.zeros(len(y))

    ct = np.zeros(len(T))
    lt = np.zeros(len(T))

    rows = []
    best_iters = []
    cb_best_iters = []

    # 1. Three-fold cross-validation

    for k, (tr, va) in enumerate(
        folds.split(X, y), 1
    ):
        print(
            "V10 FOLD", k, "/ 3: CatBoost",
            flush=True
        )

        cb = CatBoostClassifier(
            iterations=1200,
            depth=6,
            learning_rate=0.035,
            l2_leaf_reg=9,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=100 + k,
            thread_count=4,
            verbose=False,
            allow_writing_files=False
        )

        cb.fit(
            CX.iloc[tr],
            y[tr],
            cat_features=cats,
            eval_set=(CX.iloc[va], y[va]),
            early_stopping_rounds=110,
            verbose=False
        )

        co[va] = cb.predict_proba(
            CX.iloc[va]
        )[:, 1]

        ct += cb.predict_proba(
            CT
        )[:, 1] / 3

        cb_best_iters.append(
            max(100, cb.get_best_iteration() + 1)
        )

        cb_train = roc_auc_score(
            y[tr],
            cb.predict_proba(CX.iloc[tr])[:, 1]
        )

        cb_val = roc_auc_score(
            y[va], co[va]
        )

        rows.append((
            k, "CatBoost",
            cb_train, cb_val,
            cb_train - cb_val
        ))

        del cb
        gc.collect()

        print(
            "V10 FOLD", k, "/ 3: LightGBM",
            flush=True
        )

        lg = LGBMClassifier(
            n_estimators=1500,
            learning_rate=0.03,
            num_leaves=15,
            min_child_samples=85,
            colsample_bytree=0.82,
            subsample=0.85,
            subsample_freq=1,
            reg_lambda=10,
            reg_alpha=1,
            random_state=200 + k,
            n_jobs=4,
            verbosity=-1
        )

        lg.fit(
            X.iloc[tr],
            y[tr],
            eval_set=[(X.iloc[va], y[va])],
            eval_metric="auc",
            callbacks=[
                early_stopping(
                    110, verbose=False
                ),
                log_evaluation(0)
            ]
        )

        lo[va] = lg.predict_proba(
            X.iloc[va]
        )[:, 1]

        lt += lg.predict_proba(
            T
        )[:, 1] / 3

        lg_train = roc_auc_score(
            y[tr],
            lg.predict_proba(X.iloc[tr])[:, 1]
        )

        lg_val = roc_auc_score(
            y[va], lo[va]
        )

        rows.append((
            k, "LightGBM",
            lg_train, lg_val,
            lg_train - lg_val
        ))

        best_iters.append(lg.best_iteration_)

        print(
            "FOLD", k,
            "CatBoost", round(cb_val, 6),
            "LightGBM", round(lg_val, 6),
            flush=True
        )

        del lg
        gc.collect()


    # 2. Training vs validation AUC

    fold_table = pd.DataFrame(
        rows,
        columns=[
            "fold", "model", "train_auc",
            "validation_auc", "overfit_gap"
        ]
    )

    print("\nFOLD VALIDATION AND TRAINING GAP")

    print(
        fold_table.round(5).to_string(index=False)
    )


    # 3. Compare models and ensembles

    new_oof = 0.5 * co + 0.5 * lo
    new_test = 0.5 * ct + 0.5 * lt

    candidates = {
        "v10_catboost": (co, ct),
        "v10_lightgbm": (lo, lt),
        "v10_ensemble": (
            new_oof, new_test
        )
    }

    # Only reuse V8 predictions if rows align.
    prior_oof = globals().get("v8_oof", None)
    prior_test = globals().get("v8_test", None)

    old_train = globals().get("train", None)
    old_test = globals().get("test", None)

    aligned = (
        isinstance(old_train, pd.DataFrame)
        and isinstance(old_test, pd.DataFrame)
        and "encounter_id" in old_train
        and "encounter_id" in old_test
        and old_train.encounter_id.astype(str)
            .reset_index(drop=True)
            .equals(
                tr0.encounter_id.astype(str)
                .reset_index(drop=True)
            )
        and old_test.encounter_id.astype(str)
            .reset_index(drop=True)
            .equals(
                te0.encounter_id.astype(str)
                .reset_index(drop=True)
            )
    )

    if (
        aligned
        and prior_oof is not None
        and prior_test is not None
    ):
        old_oof = np.asarray(
            prior_oof
        ).reshape(-1)

        old_pred = np.asarray(
            prior_test
        ).reshape(-1)

        if (
            len(old_oof) == len(y)
            and len(old_pred) == len(T)
        ):
            candidates["v8_reference"] = (
                old_oof, old_pred
            )

            candidates["v10_v8_30_new70"] = (
                0.3 * old_oof + 0.7 * new_oof,
                0.3 * old_pred + 0.7 * new_test
            )

            candidates["v10_v8_50_new50"] = (
                0.5 * old_oof + 0.5 * new_oof,
                0.5 * old_pred + 0.5 * new_test
            )

            print(
                "VERIFIED: V8 predictions aligned."
            )

    else:
        print(
            "V8 predictions unavailable or unverified. "
            "Historical V8 OOF ROC-AUC: 0.748035."
        )

    ranked = sorted(
        [
            (
                roc_auc_score(y, p[0]),
                name
            )
            for name, p in candidates.items()
        ],
        reverse=True
    )

    print("\nV10 THREE-FOLD ROC-AUC COMPARISON")

    for auc, name in ranked:
        ap = average_precision_score(
            y, candidates[name][0]
        )

        print(
            f"{name:24s} "
            f"ROC-AUC={auc:.6f} "
            f"PR-AUC={ap:.6f}"
        )


    # 4. Feature ablation: old vs comprehensive

    v8_cols = [
        c for c in X
        if c.startswith("v8_")
    ]

    if len(v8_cols) > 100:
        tr, va = next(folds.split(X, y))

        baseline = LGBMClassifier(
            n_estimators=1000,
            learning_rate=0.03,
            num_leaves=15,
            min_child_samples=85,
            reg_lambda=10,
            n_jobs=4,
            verbosity=-1
        )

        baseline.fit(
            X.iloc[tr][v8_cols],
            y[tr],
            eval_set=[
                (X.iloc[va][v8_cols], y[va])
            ],
            callbacks=[
                early_stopping(
                    90, verbose=False
                ),
                log_evaluation(0)
            ]
        )

        baseline_auc = roc_auc_score(
            y[va],
            baseline.predict_proba(
                X.iloc[va][v8_cols]
            )[:, 1]
        )

        full_auc = roc_auc_score(
            y[va], lo[va]
        )

        print(
            "\nABLATION (LightGBM fold 1):",
            "V8 features only:",
            round(baseline_auc, 6),
            "| V10 comprehensive:",
            round(full_auc, 6),
            "| gain:",
            round(full_auc - baseline_auc, 6)
        )

        del baseline
        gc.collect()


    # 5. Generalization to later admissions and hospitals

    fixed_trees = max(
        100, int(np.median(best_iters))
    )

    fixed_cb = int(
        np.median(cb_best_iters)
    )

    print(
        "\nGENERALIZATION STRESS TESTS",
        "| LightGBM trees:", fixed_trees,
        "| CatBoost trees:", fixed_cb
    )

    def stress(name, tr, va):
        if (
            len(tr) < 1000
            or len(va) < 1000
            or len(np.unique(y[va])) < 2
        ):
            print(
                name, "SKIPPED: insufficient data"
            )
            return

        m = LGBMClassifier(
            n_estimators=fixed_trees,
            learning_rate=0.03,
            num_leaves=15,
            min_child_samples=85,
            colsample_bytree=0.82,
            reg_lambda=10,
            reg_alpha=1,
            n_jobs=4,
            verbosity=-1,
            random_state=999
        )

        m.fit(
            X.iloc[tr], y[tr]
        )

        p_train = m.predict_proba(
            X.iloc[tr]
        )[:, 1]

        p_val = m.predict_proba(
            X.iloc[va]
        )[:, 1]

        print(
            f"{name} LightGBM: "
            f"train={roc_auc_score(y[tr], p_train):.6f} "
            f"holdout={roc_auc_score(y[va], p_val):.6f}",
            flush=True
        )

        del m
        gc.collect()

        c = CatBoostClassifier(
            iterations=fixed_cb,
            depth=6,
            learning_rate=0.035,
            l2_leaf_reg=9,
            loss_function="Logloss",
            random_seed=999,
            thread_count=4,
            verbose=False,
            allow_writing_files=False
        )

        c.fit(
            CX.iloc[tr],
            y[tr],
            cat_features=cats,
            verbose=False
        )

        q_train = c.predict_proba(
            CX.iloc[tr]
        )[:, 1]

        q_val = c.predict_proba(
            CX.iloc[va]
        )[:, 1]

        print(
            f"{name} CatBoost: "
            f"train={roc_auc_score(y[tr], q_train):.6f} "
            f"holdout={roc_auc_score(y[va], q_val):.6f}",
            flush=True
        )

        blend_train = 0.5 * (
            p_train + q_train
        )

        blend_val = 0.5 * (
            p_val + q_val
        )

        a = roc_auc_score(
            y[tr], blend_train
        )

        b = roc_auc_score(
            y[va], blend_val
        )

        print(
            f"{name} ENSEMBLE: "
            f"train={a:.6f} "
            f"holdout={b:.6f} "
            f"gap={a-b:.6f} "
            f"holdout_n={len(va)} "
            f"holdout_prevalence={y[va].mean():.3f}",
            flush=True
        )

        del c
        gc.collect()


    # Temporal holdout: train on earlier admissions.

    admit = V10_ADMIT
    valid = admit.notna().to_numpy()

    if (
        valid.mean() > 0.80
        and admit[valid].nunique() > 20
    ):
        order = np.where(valid)[0][
            np.argsort(
                admit[valid].to_numpy()
            )
        ]

        cut = int(
            len(order) * 0.8
        )

        stress(
            "TEMPORAL (earlier -> later admissions)",
            order[:cut],
            order[cut:]
        )

    else:
        print(
            "TEMPORAL: skipped "
            "(dates unavailable or insufficient variation)"
        )


    # Hospital holdout: train without some hospitals.

    if (
        "V10_HOSP" in globals()
        and V10_HOSP.notna().sum() > 0
        and V10_HOSP.nunique() > 5
    ):
        grp = (
            V10_HOSP.fillna("Missing")
            .astype(str).to_numpy()
        )

        tr, va = next(
            GroupShuffleSplit(
                n_splits=1,
                test_size=0.2,
                random_state=42
            ).split(
                X, y, groups=grp
            )
        )

        stress(
            "HOSPITAL (unseen hospitals)",
            tr, va
        )

    else:
        print(
            "HOSPITAL: skipped "
            "(hospital identifiers unavailable)"
        )


    # 6. Select candidate and bootstrap AUC difference

    new_ranked = [
        (auc, name)
        for auc, name in ranked
        if name != "v8_reference"
    ]

    best_auc, best_name = new_ranked[0]

    if "v8_reference" in candidates:
        old_ref = candidates[
            "v8_reference"
        ][0]

        chosen = candidates[
            best_name
        ][0]

        rng = np.random.default_rng(2026)
        diffs = []

        for _ in range(150):
            ix = rng.integers(
                0, len(y), len(y)
            )

            if len(np.unique(y[ix])) < 2:
                continue

            diffs.append(
                roc_auc_score(
                    y[ix], chosen[ix]
                )
                -
                roc_auc_score(
                    y[ix], old_ref[ix]
                )
            )

        print(
            "\n",
            best_name,
            "minus V8 bootstrap AUC gain",
            "(5th, 50th, 95th percentiles):",
            np.round(
                np.percentile(
                    diffs, [5, 50, 95]
                ),
                6
            )
        )


    # 7. Save submission, but do not submit automatically

    output = pd.DataFrame({
        "encounter_id": V10_TEST_IDS,
        "readmit_30d": candidates[
            best_name
        ][1]
    })

    assert len(output) == len(te0)
    assert output.encounter_id.is_unique
    assert output.readmit_30d.notna().all()
    assert output.readmit_30d.between(0, 1).all()

    path = "/kaggle/working/SUBMIT_V10.csv"

    output.to_csv(
        path, index=False
    )

    print(
        "\nSAVED:", path,
        "|", best_name,
        "| CV ROC-AUC", round(best_auc, 6)
    )

    print(
        "V8 historical offline ROC-AUC: 0.748035",
        "| V8 public ROC-AUC: 0.741920"
    )

    if best_auc <= 0.748035:
        print(
            "WARNING: V10 did not beat historical V8 "
            "offline. DO NOT SUBMIT yet."
        )
    else:
        print(
            "V10 improved offline. Review fold consistency, "
            "stress tests, and bootstrap BEFORE submitting."
        )

    with open(path, "rb") as f:
        encoded = base64.b64encode(
            f.read()
        ).decode("ascii")

    display(HTML(
        '<a download="SUBMIT_V10.csv" '
        'href="data:text/csv;base64,'
        + encoded
        + '">DOWNLOAD V10 CSV</a>'
    ))


### Archive M40 — V11 ensemble models

**Original source cell 40; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    from pathlib import Path
    import gc, re, base64
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier
    from IPython.display import HTML, display

    # Check previous results are available
    needed = [
        "X9", "T9", "V10_X", "V10_T",
        "tr0", "te0", "train", "test",
        "v8_oof", "v8_test",
        "xgb_oof", "xgb_test"
    ]

    missing = [s for s in needed if s not in globals()]

    if missing:
        raise RuntimeError(
            "Missing: " + ", ".join(missing)
            + ". Use the same notebook session after V9 and V10."
        )

    # Verify that all three versions use identical encounter order
    assert train.encounter_id.astype(str).reset_index(
        drop=True
    ).equals(
        tr0.encounter_id.astype(str).reset_index(drop=True)
    )

    assert test.encounter_id.astype(str).reset_index(
        drop=True
    ).equals(
        te0.encounter_id.astype(str).reset_index(drop=True)
    )

    assert len(X9) == len(V10_X) == len(tr0)
    assert len(T9) == len(V10_T) == len(te0)

    y11 = tr0.readmit_30d.to_numpy(dtype=int)
    n11 = len(y11)

    ids11 = pd.Index(
        pd.concat(
            [tr0.encounter_id, te0.encounter_id],
            ignore_index=True
        ).astype(str)
    )

    # Keep the complete V9 feature representation
    base = pd.concat(
        [X9, T9], ignore_index=True
    ).copy()

    base = base.loc[
        :, ~base.columns.duplicated()
    ].reset_index(drop=True)

    base.columns = [
        "base_" + str(c) for c in base.columns
    ]

    # Add selected clinical information from V10
    extra = pd.concat(
        [V10_X, V10_T],
        ignore_index=True
    ).reset_index(drop=True)

    prefixes = (
        "v10_history_",
        "v10_prior_type_",
        "v10_vital_",
        "v10_lab_",
        "v10_cross_",
        "v10_plan_",
        "v10_icu_",
        "v10_transfer_",
        "v10_proc_",
        "v10_patient_",
        "v10_hospital_",
        "v10_dx_icd_chapter_"
    )

    cols = [
        c for c in extra.columns
        if c.startswith(prefixes)
        and pd.api.types.is_numeric_dtype(extra[c])
    ]

    extra = extra[cols].copy()

    extra.columns = [
        "extra_" + str(c) for c in extra.columns
    ]

    print(
        "V11: existing features", base.shape[1],
        "| additional clinical features", extra.shape[1],
        flush=True
    )

    # Restore encounter-level medication information.
    # These are encounter-scoped and discharge-time attributes;
    # no post-discharge outcomes are used.
    medpaths = list(
        Path("/kaggle/input").rglob("medications.csv")
    )

    if medpaths:
        m = pd.read_csv(
            medpaths[0],
            usecols=lambda c: c in [
                "encounter_id",
                "drug_class",
                "continued_at_discharge"
            ],
            low_memory=False
        )

        m = m.dropna(subset=["encounter_id"])
        m["encounter_id"] = m.encounter_id.astype(str)

        g = m.groupby("encounter_id")

        medf = g.size().to_frame("v11_med_count")

        if "drug_class" in m:
            cls = (
                m.drug_class.fillna("Missing")
                .astype(str).str.upper().str.strip()
            )

            keep = cls.value_counts().head(25).index

            t = pd.crosstab(
                m.encounter_id,
                cls.where(cls.isin(keep), "OTHER")
            )

            t.columns = [
                "v11_med_" +
                re.sub("[^A-Z0-9]+", "_", str(c))[:32]
                for c in t.columns
            ]

            medf = medf.join(t)

        if "continued_at_discharge" in m:
            continued = (
                m.continued_at_discharge
                .astype(str).str.lower().str.strip()
                .isin(["true", "yes", "y", "1"])
            )

            medf["v11_med_continued"] = (
                continued.groupby(m.encounter_id).sum()
            )

        medf.index = medf.index.astype(str)

        medf = medf.reindex(ids11).reset_index(drop=True)
        medf = medf.add_prefix("extra_")

        extra = pd.concat(
            [extra, medf], axis=1
        )

        print(
            "Restored medication features:",
            medf.shape[1],
            flush=True
        )

        del m, medf

    else:
        print("WARNING: medications.csv not found", flush=True)

    # Prepare numerical training data
    Z = pd.concat([base, extra], axis=1)
    Z = Z.loc[:, ~Z.columns.duplicated()]

    for c in Z.columns:
        Z[c] = pd.to_numeric(Z[c], errors="coerce")

    Z = Z.replace([np.inf, -np.inf], np.nan)

    Z = Z.loc[
        :, Z.iloc[:n11].notna().sum() >= 100
    ]

    Z = Z.loc[
        :, Z.iloc[:n11].nunique(dropna=True) > 1
    ]

    Z.columns = [
        re.sub("[^A-Za-z0-9_]", "_", str(c))
        for c in Z.columns
    ]

    Z = Z.loc[
        :, ~Z.columns.duplicated()
    ].astype("float32")

    Ztr = Z.iloc[:n11].reset_index(drop=True)
    Zte = Z.iloc[n11:].reset_index(drop=True)

    assert Ztr.columns.equals(Zte.columns)

    print(
        "V11 TRAINING:", Ztr.shape,
        "TEST:", Zte.shape,
        flush=True
    )

    # Same folds as V8/V9 for a fair comparison
    splits = list(
        StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=42
        ).split(Ztr, y11)
    )

    o_xgb = np.zeros(n11)
    t_xgb = np.zeros(len(Zte))

    o_cb = np.zeros(n11)
    t_cb = np.zeros(len(Zte))

    train_gaps = []

    for k, (tr, va) in enumerate(splits, 1):

        print(
            "V11 FOLD", k, "/ 3 XGBOOST",
            flush=True
        )

        xgb = XGBClassifier(
            n_estimators=1250,
            max_depth=3,
            learning_rate=0.035,
            min_child_weight=18,
            subsample=0.82,
            colsample_bytree=0.82,
            reg_lambda=14,
            reg_alpha=1,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            n_jobs=4,
            random_state=400 + k,
            early_stopping_rounds=90
        )

        xgb.fit(
            Ztr.iloc[tr],
            y11[tr],
            eval_set=[(Ztr.iloc[va], y11[va])],
            verbose=False
        )

        o_xgb[va] = xgb.predict_proba(
            Ztr.iloc[va]
        )[:, 1]

        t_xgb += xgb.predict_proba(
            Zte
        )[:, 1] / 3

        train_auc = roc_auc_score(
            y11[tr],
            xgb.predict_proba(Ztr.iloc[tr])[:, 1]
        )

        val_auc = roc_auc_score(
            y11[va], o_xgb[va]
        )

        train_gaps.append((
            "XGBoost", k,
            round(train_auc, 6),
            round(val_auc, 6),
            round(train_auc - val_auc, 6)
        ))

        print(
            "XGBoost validation:", round(val_auc, 6),
            "| overfit gap:",
            round(train_auc - val_auc, 6),
            flush=True
        )

        del xgb
        gc.collect()

        print(
            "V11 FOLD", k, "/ 3 CATBOOST",
            flush=True
        )

        cb = CatBoostClassifier(
            iterations=1050,
            depth=5,
            learning_rate=0.04,
            l2_leaf_reg=12,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=500 + k,
            thread_count=4,
            verbose=False,
            allow_writing_files=False
        )

        cb.fit(
            Ztr.iloc[tr],
            y11[tr],
            eval_set=(Ztr.iloc[va], y11[va]),
            early_stopping_rounds=90,
            verbose=False
        )

        o_cb[va] = cb.predict_proba(
            Ztr.iloc[va]
        )[:, 1]

        t_cb += cb.predict_proba(
            Zte
        )[:, 1] / 3

        train_auc = roc_auc_score(
            y11[tr],
            cb.predict_proba(Ztr.iloc[tr])[:, 1]
        )

        val_auc = roc_auc_score(
            y11[va], o_cb[va]
        )

        train_gaps.append((
            "CatBoost", k,
            round(train_auc, 6),
            round(val_auc, 6),
            round(train_auc - val_auc, 6)
        ))

        print(
            "CatBoost validation:", round(val_auc, 6),
            "| overfit gap:",
            round(train_auc - val_auc, 6),
            flush=True
        )

        del cb
        gc.collect()

    # Recreate the known V8 and V9 baselines
    v8o = np.asarray(v8_oof)
    v8t = np.asarray(v8_test)

    v9o = 0.8 * v8o + 0.2 * np.asarray(xgb_oof)
    v9t = 0.8 * v8t + 0.2 * np.asarray(xgb_test)

    v11o = 0.5 * o_xgb + 0.5 * o_cb
    v11t = 0.5 * t_xgb + 0.5 * t_cb

    options = {
        "V8_reference": (v8o, v8t),
        "V9_reference": (v9o, v9t),
        "V11_XGBoost": (o_xgb, t_xgb),
        "V11_CatBoost": (o_cb, t_cb),
        "V11_ensemble": (v11o, v11t),

        "V11_V9_80_new20": (
            0.8*v9o + 0.2*v11o,
            0.8*v9t + 0.2*v11t
        ),

        "V11_V9_60_new40": (
            0.6*v9o + 0.4*v11o,
            0.6*v9t + 0.4*v11t
        ),

        "V11_V8_60_new40": (
            0.6*v8o + 0.4*v11o,
            0.6*v8t + 0.4*v11t
        ),

        "V11_V9_80_cb20": (
            0.8*v9o + 0.2*o_cb,
            0.8*v9t + 0.2*t_cb
        ),

        "V11_V9_80_xgb20": (
            0.8*v9o + 0.2*o_xgb,
            0.8*v9t + 0.2*t_xgb
        )
    }

    ranked11 = sorted(
        [
            (roc_auc_score(y11, p[0]), name)
            for name, p in options.items()
        ],
        reverse=True
    )

    print("\nV11 TRAIN vs VALIDATION GAPS")

    for record in train_gaps:
        print(record)

    print("\nV11 THREE-FOLD AUC COMPARISON")

    for auc, name in ranked11:
        gains = [
            roc_auc_score(
                y11[va], options[name][0][va]
            )
            -
            roc_auc_score(
                y11[va], v9o[va]
            )
            for _, va in splits
        ]

        print(
            name,
            round(auc, 6),
            "| gains vs V9:",
            [round(g, 6) for g in gains],
            flush=True
        )

    bestscore, bestname = ranked11[0]

    best_gains = [
        roc_auc_score(
            y11[va], options[bestname][0][va]
        )
        -
        roc_auc_score(
            y11[va], v9o[va]
        )
        for _, va in splits
    ]

    v9score = roc_auc_score(y11, v9o)

    # Avoid wasting submissions on tiny improvements
    if (
        bestscore <= v9score + 0.0010
        or sum(g > 0 for g in best_gains) < 2
        or bestname in ["V8_reference", "V9_reference"]
    ):
        print(
            "\nSTOP: No convincing V11 improvement.",
            "Keep V9/V8. Do not submit V11.",
            flush=True
        )

    else:
        pred = np.asarray(
            options[bestname][1]
        )

        assert len(pred) == len(te0)
        assert np.isfinite(pred).all()
        assert ((pred >= 0) & (pred <= 1)).all()

        out = pd.DataFrame({
            "encounter_id": te0.encounter_id,
            "readmit_30d": pred
        })

        assert out.encounter_id.is_unique

        path = "/kaggle/working/SUBMIT_V11.csv"

        out.to_csv(path, index=False)

        print(
            "\nV11 CANDIDATE SAVED:",
            bestname,
            "ROC-AUC:", round(bestscore, 6),
            flush=True
        )

        with open(path, "rb") as f:
            encoded = base64.b64encode(
                f.read()
            ).decode()

        display(HTML(
            '<a download="SUBMIT_V11.csv" '
            'href="data:text/csv;base64,'
            + encoded
            + '">DOWNLOAD V11 CSV</a>'
        ))


### Archive M41 — V11 final submissions

**Original source cell 41; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import numpy as np
    import pandas as pd
    import base64
    from pathlib import Path
    from sklearn.metrics import roc_auc_score
    from IPython.display import HTML, display

    # Three final candidates from the completed V11 run
    final_choices = {
        "FINAL_A_V9_XGB.csv": "V11_V9_80_xgb20",
        "FINAL_B_V8_NEW40.csv": "V11_V8_60_new40",
        "FINAL_C_V9_NEW40.csv": "V11_V9_60_new40"
    }

    assert "options" in globals(), (
        "Run this in the completed V11 notebook."
    )

    assert "test" in globals()
    assert "train" in globals()

    missing = [
        name for name in final_choices.values()
        if name not in options
    ]

    assert not missing, (
        "Missing V11 candidates: " + str(missing)
    )

    y_final = train["readmit_30d"].to_numpy()
    test_ids_final = test["encounter_id"].to_numpy()

    print("FINAL SUBMISSION CHECK")
    print("Train:", len(y_final))
    print("Test:", len(test_ids_final))

    for filename, model_name in final_choices.items():

        oof_pred, test_pred = options[model_name]

        oof_pred = np.asarray(oof_pred).reshape(-1)
        test_pred = np.asarray(test_pred).reshape(-1)

        assert len(oof_pred) == len(y_final)
        assert len(test_pred) == len(test_ids_final)
        assert np.isfinite(oof_pred).all()
        assert np.isfinite(test_pred).all()
        assert ((test_pred >= 0) & (test_pred <= 1)).all()

        score = roc_auc_score(y_final, oof_pred)

        output = pd.DataFrame({
            "encounter_id": test_ids_final,
            "readmit_30d": test_pred
        })

        assert output["encounter_id"].is_unique

        path = Path("/kaggle/working") / filename
        output.to_csv(path, index=False)

        print(
            "\nSAVED:", filename,
            "| OOF AUC:", round(score, 6),
            "| ROWS:", len(output),
            flush=True
        )

        with open(path, "rb") as f:
            encoded = base64.b64encode(
                f.read()
            ).decode("ascii")

        display(HTML(
            '<a download="' + filename +
            '" href="data:text/csv;base64,' +
            encoded +
            '">DOWNLOAD ' + filename + '</a>'
        ))

    print("\nALL THREE FILES ARE READY.")


### Archive M42 — V12 fast blends

**Original source cell 42; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import numpy as np
    import pandas as pd
    import base64

    from scipy.stats import rankdata
    from sklearn.metrics import roc_auc_score
    from IPython.display import display, HTML

    assert "options" in globals(), "Run in completed V11 notebook."
    assert "train" in globals() and "test" in globals()

    y12 = train["readmit_30d"].to_numpy(dtype=int)
    ids12 = test["encounter_id"].to_numpy()

    def v12_get(name):
        assert name in options, "Missing V11 option: " + name
        a, b = options[name]
        a = np.asarray(a, dtype=float).reshape(-1)
        b = np.asarray(b, dtype=float).reshape(-1)
        assert len(a) == len(y12)
        assert len(b) == len(ids12)
        assert np.isfinite(a).all() and np.isfinite(b).all()
        return a, b

    v9o, v9t = v12_get("V9_reference")
    v8o, v8t = v12_get("V8_reference")
    v11o, v11t = v12_get("V11_V9_80_xgb20")

    def v12_rank(x):
        return rankdata(x, method="average") / len(x)

    def v12_mix(parts, ranked=False):
        a = np.zeros(len(y12))
        b = np.zeros(len(ids12))
        for weight, (o, t) in parts:
            if ranked:
                o, t = v12_rank(o), v12_rank(t)
            a += weight * o
            b += weight * t
        return a, b

    V12_candidates = {
        "V11_BASELINE": (v11o, v11t),

        "V12_PROB_70V11_30V8": v12_mix([
            (.70, (v11o, v11t)),
            (.30, (v8o, v8t))
        ]),

        "V12_PROB_70V11_30V9": v12_mix([
            (.70, (v11o, v11t)),
            (.30, (v9o, v9t))
        ]),

        "V12_RANK_80V11_20V8": v12_mix([
            (.80, (v11o, v11t)),
            (.20, (v8o, v8t))
        ], ranked=True),

        "V12_RANK_70V11_30V9": v12_mix([
            (.70, (v11o, v11t)),
            (.30, (v9o, v9t))
        ], ranked=True),

        "V12_RANK_50V11_30V9_20V8": v12_mix([
            (.50, (v11o, v11t)),
            (.30, (v9o, v9t)),
            (.20, (v8o, v8t))
        ], ranked=True)
    }

    V12_results = sorted(
        [
            (roc_auc_score(y12, pred[0]), name)
            for name, pred in V12_candidates.items()
        ],
        reverse=True
    )

    print("V12 FAST ENSEMBLE COMPARISON")
    for auc, name in V12_results:
        print(f"{name:32s} ROC-AUC={auc:.6f}")

    v12_fast_auc, v12_fast_name = V12_results[0]
    v12_fast_oof, v12_fast_test = V12_candidates[v12_fast_name]

    def v12_save(filename, pred):
        pred = np.asarray(pred).reshape(-1)
        assert len(pred) == len(ids12)
        assert np.isfinite(pred).all()
        assert ((pred >= 0) & (pred <= 1)).all()

        out = pd.DataFrame({
            "encounter_id": ids12,
            "readmit_30d": pred
        })
        assert out.encounter_id.is_unique

        path = "/kaggle/working/" + filename
        out.to_csv(path, index=False)

        with open(path, "rb") as f:
            encoded = base64.b64encode(f.read()).decode()

        display(HTML(
            '<a download="' + filename +
            '" href="data:text/csv;base64,' + encoded +
            '">DOWNLOAD ' + filename + '</a>'
        ))

        print("SAVED:", path)

    v12_save("V12_FAST_BLEND.csv", v12_fast_test)

    print("BEST FAST V12:", v12_fast_name, round(v12_fast_auc, 6))
    print("TARGET: 0.750000")

    if v12_fast_auc >= 0.7500:
        print("TARGET REACHED OFFLINE — verify fold consistency.")
    else:
        print("Below target. Continue to V12 Cell 2.")


### Archive M43 — V12 LightGBM variants

**Original source cell 43; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import gc
    import numpy as np
    import pandas as pd

    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from lightgbm import LGBMClassifier, early_stopping, log_evaluation

    assert "Ztr" in globals() and "Zte" in globals(), (
        "Run in completed V11 notebook."
    )
    assert "V12_candidates" in globals(), "Run V12 Cell 1 first."

    X12 = pd.DataFrame(Ztr).reset_index(drop=True).copy()
    T12 = pd.DataFrame(Zte).reset_index(drop=True).copy()

    assert len(X12) == len(y12)
    assert len(T12) == len(ids12)
    assert X12.shape[1] == T12.shape[1]

    for c in X12.columns:
        X12[c] = pd.to_numeric(X12[c], errors="coerce")
        T12[c] = pd.to_numeric(T12[c], errors="coerce")

    X12 = X12.replace([np.inf, -np.inf], np.nan)
    T12 = T12.replace([np.inf, -np.inf], np.nan)

    keep12 = (
        (X12.notna().sum() >= 100)
        & (X12.nunique(dropna=True) > 1)
    )

    X12 = X12.loc[:, keep12].astype("float32")
    T12 = T12.loc[:, keep12].astype("float32")

    # LightGBM requires unique feature names
    X12.columns = [f"v12f_{i}" for i in range(X12.shape[1])]
    T12.columns = X12.columns

    print("V12 TRAIN:", X12.shape, "TEST:", T12.shape, flush=True)

    folds12 = list(
        StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=42
        ).split(X12, y12)
    )

    v12_oof = {}
    v12_test = {}
    v12_report = []

    configs = {
        "V12_SMALL": dict(
            num_leaves=9,
            min_child_samples=110,
            reg_lambda=18,
            reg_alpha=1.0,
            max_depth=5
        ),
        "V12_MEDIUM": dict(
            num_leaves=17,
            min_child_samples=85,
            reg_lambda=14,
            reg_alpha=0.8,
            max_depth=-1
        )
    }

    for name, params in configs.items():

        v12_oof[name] = np.zeros(len(y12))
        v12_test[name] = np.zeros(len(ids12))

        for k, (tr, va) in enumerate(folds12, 1):

            print(name, "FOLD", k, "/3", flush=True)

            model = LGBMClassifier(
                n_estimators=850,
                learning_rate=0.035,
                objective="binary",
                colsample_bytree=0.85,
                subsample=0.85,
                subsample_freq=1,
                max_bin=127,
                n_jobs=4,
                random_state=1200 + k,
                verbosity=-1,
                **params
            )

            model.fit(
                X12.iloc[tr],
                y12[tr],
                eval_set=[(X12.iloc[va], y12[va])],
                eval_metric="auc",
                callbacks=[
                    early_stopping(70, verbose=False),
                    log_evaluation(0)
                ]
            )

            vp = model.predict_proba(X12.iloc[va])[:, 1]
            tp = model.predict_proba(T12)[:, 1]

            v12_oof[name][va] = vp
            v12_test[name] += tp / 3

            train_auc = roc_auc_score(
                y12[tr],
                model.predict_proba(X12.iloc[tr])[:, 1]
            )

            val_auc = roc_auc_score(y12[va], vp)

            v12_report.append((
                name, k, train_auc, val_auc,
                train_auc - val_auc
            ))

            print(
                "VALIDATION AUC:", round(val_auc, 6),
                "OVERFIT GAP:", round(train_auc - val_auc, 6),
                flush=True
            )

            del model
            gc.collect()

    # New LightGBM ensemble
    new_oof = (
        0.5 * v12_oof["V12_SMALL"]
        + 0.5 * v12_oof["V12_MEDIUM"]
    )

    new_test = (
        0.5 * v12_test["V12_SMALL"]
        + 0.5 * v12_test["V12_MEDIUM"]
    )

    # Compare fixed ensemble weights against V11
    final12 = dict(V12_candidates)

    final12.update({
        "V12_SMALL_ONLY": (
            v12_oof["V12_SMALL"],
            v12_test["V12_SMALL"]
        ),

        "V12_MEDIUM_ONLY": (
            v12_oof["V12_MEDIUM"],
            v12_test["V12_MEDIUM"]
        ),

        "V12_LGB_ENSEMBLE": (new_oof, new_test),

        "V12_V11_85_LGB15": v12_mix([
            (.85, (v11o, v11t)),
            (.15, (new_oof, new_test))
        ]),

        "V12_V11_70_LGB30": v12_mix([
            (.70, (v11o, v11t)),
            (.30, (new_oof, new_test))
        ]),

        "V12_V9_60_V11_25_LGB15": v12_mix([
            (.60, (v9o, v9t)),
            (.25, (v11o, v11t)),
            (.15, (new_oof, new_test))
        ]),

        "V12_RANK_V11_80_LGB20": v12_mix([
            (.80, (v11o, v11t)),
            (.20, (new_oof, new_test))
        ], ranked=True)
    })

    ranked12 = sorted(
        [
            (roc_auc_score(y12, p[0]), name)
            for name, p in final12.items()
        ],
        reverse=True
    )

    print("\nFINAL V12 THREE-FOLD ROC-AUC COMPARISON")

    for auc, name in ranked12:
        print(f"{name:34s} {auc:.6f}")

    best_auc12, best_name12 = ranked12[0]
    best_oof12, best_test12 = final12[best_name12]

    print("\nFOLD GENERALIZATION CHECK")

    for k, (tr, va) in enumerate(folds12, 1):
        new_auc = roc_auc_score(y12[va], best_oof12[va])
        old_auc = roc_auc_score(y12[va], v11o[va])

        print(
            f"Fold {k}: V12={new_auc:.6f}",
            f"V11={old_auc:.6f}",
            f"GAIN={new_auc-old_auc:+.6f}"
        )

    print("\nV12 TRAINING GAPS")

    print(
        pd.DataFrame(
            v12_report,
            columns=[
                "model", "fold",
                "train_auc", "validation_auc",
                "overfit_gap"
            ]
        ).round(5).to_string(index=False)
    )

    v12_save("V12_FINAL.csv", best_test12)

    print("\nV12 WINNER:", best_name12)
    print("V12 OFFLINE ROC-AUC:", round(best_auc12, 6))
    print("V11 BEST OFFLINE ROC-AUC: 0.748671")

    if best_auc12 >= 0.7500:
        print("TARGET 0.7500 REACHED OFFLINE.")
    elif best_auc12 > 0.748671:
        print("IMPROVED OVER V11, BUT BELOW 0.7500.")
    else:
        print("NO IMPROVEMENT. KEEP EXISTING V11/V9.")


### Archive M44 — V12 logistic stacking

**Original source cell 44; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import numpy as np
    import pandas as pd
    import base64

    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from IPython.display import HTML, display

    assert "options" in globals()
    assert "v12_oof" in globals()
    assert "v12_test" in globals()
    assert "y12" in globals()
    assert "ids12" in globals()

    # Collect different models already trained
    sources = {}

    for name in [
        "V9_reference",
        "V8_reference",
        "V11_XGBoost",
        "V11_CatBoost",
        "V11_ensemble"
    ]:
        if name in options:
            a, b = options[name]
            sources[name] = (
                np.asarray(a).reshape(-1),
                np.asarray(b).reshape(-1)
            )

    for name in ["V12_SMALL", "V12_MEDIUM"]:
        if name in v12_oof and name in v12_test:
            sources[name] = (
                np.asarray(v12_oof[name]).reshape(-1),
                np.asarray(v12_test[name]).reshape(-1)
            )

    assert len(sources) >= 3, (
        "Not enough existing models for stacking."
    )

    # Use the best validated V11 model as baseline
    base_oof, base_test = options[
        "V11_V9_80_xgb20"
    ]

    base_oof = np.asarray(base_oof).reshape(-1)
    base_test = np.asarray(base_test).reshape(-1)

    print("STACKING MODELS:", list(sources))

    def logit12(x):
        x = np.clip(np.asarray(x), 1e-5, 1-1e-5)
        return np.log(x / (1-x))

    # Base-model probability scores become meta-features
    stack_X = np.column_stack([
        logit12(p[0]) for p in sources.values()
    ])

    stack_T = np.column_stack([
        logit12(p[1]) for p in sources.values()
    ])

    assert stack_X.shape[0] == len(y12)
    assert stack_T.shape[0] == len(ids12)
    assert np.isfinite(stack_X).all()
    assert np.isfinite(stack_T).all()

    folds = list(
        StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=42
        ).split(stack_X, y12)
    )

    stack_oof = np.zeros(len(y12))
    stack_test = np.zeros(len(ids12))

    for k, (tr, va) in enumerate(folds, 1):

        scaler = StandardScaler()

        Xtr = scaler.fit_transform(stack_X[tr])
        Xva = scaler.transform(stack_X[va])
        Xt = scaler.transform(stack_T)

        # Strong regularization limits meta-model overfitting
        meta = LogisticRegression(
            C=0.05,
            max_iter=300,
            solver="lbfgs",
            random_state=42
        )

        meta.fit(Xtr, y12[tr])

        stack_oof[va] = meta.predict_proba(Xva)[:, 1]
        stack_test += meta.predict_proba(Xt)[:, 1] / 3

        print(
            "STACK FOLD", k,
            "AUC:",
            round(
                roc_auc_score(y12[va], stack_oof[va]),
                6
            ),
            flush=True
        )

    # Evaluate different weights
    candidates = {
        "V11_BASELINE": (
            base_oof,
            base_test
        ),
        "V12_STACK_ONLY": (
            stack_oof,
            stack_test
        ),
        "V12_STACK_25": (
            .75*base_oof + .25*stack_oof,
            .75*base_test + .25*stack_test
        ),
        "V12_STACK_50": (
            .50*base_oof + .50*stack_oof,
            .50*base_test + .50*stack_test
        ),
        "V12_STACK_75": (
            .25*base_oof + .75*stack_oof,
            .25*base_test + .75*stack_test
        )
    }

    results = sorted(
        [
            (roc_auc_score(y12, p[0]), name)
            for name, p in candidates.items()
        ],
        reverse=True
    )

    print("\nFINAL V12 STACKING COMPARISON")

    for auc, name in results:
        print(f"{name:24s} {auc:.6f}")

    # Find the best NEW candidate
    challengers = [
        (auc, name)
        for auc, name in results
        if name != "V11_BASELINE"
    ]

    challenger_auc, challenger_name = challengers[0]
    challenger_oof, challenger_test = candidates[
        challenger_name
    ]

    print("\nFOLD-BY-FOLD COMPARISON")

    gains = []

    for k, (tr, va) in enumerate(folds, 1):
        old_auc = roc_auc_score(y12[va], base_oof[va])
        new_auc = roc_auc_score(y12[va], challenger_oof[va])
        gain = new_auc - old_auc
        gains.append(gain)

        print(
            f"Fold {k}: "
            f"V11={old_auc:.6f} "
            f"V12={new_auc:.6f} "
            f"gain={gain:+.6f}"
        )

    print("\nBEST NEW CHALLENGER:", challenger_name)
    print("CHALLENGER AUC:", round(challenger_auc, 6))
    print("V11 BASELINE:", round(
        roc_auc_score(y12, base_oof), 6
    ))

    # Save challenger, without overwriting V11/V12 files
    filename = "V12_STACK_CHALLENGER.csv"

    out = pd.DataFrame({
        "encounter_id": ids12,
        "readmit_30d": challenger_test
    })

    assert len(out) == len(ids12)
    assert out.encounter_id.is_unique
    assert np.isfinite(out.readmit_30d).all()
    assert out.readmit_30d.between(0, 1).all()

    path = "/kaggle/working/" + filename
    out.to_csv(path, index=False)

    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode()

    display(HTML(
        '<a download="' + filename +
        '" href="data:text/csv;base64,' + encoded +
        '">DOWNLOAD ' + filename + '</a>'
    ))

    if (
        challenger_auc > 0.748671
        and sum(g > 0 for g in gains) >= 2
    ):
        print("\nPOTENTIAL IMPROVEMENT — review before submitting.")
    else:
        print("\nNO CONVINCING IMPROVEMENT — retain V11/V9.")

    if challenger_auc >= 0.7600:
        print("0.7600 TARGET REACHED OFFLINE.")
    else:
        print("0.7600 TARGET NOT REACHED.")


### Archive M45 — V12 nonlinear blends

**Original source cell 45; disabled for automatic execution.** This preserves the historical implementation, not a new claim of validation or a recommended final model.

In [ ]:
if False:  # ARCHIVED ORIGINAL SOURCE — not part of Run All execution

    import numpy as np
    import pandas as pd
    import base64
    from scipy.special import expit, logit
    from sklearn.metrics import roc_auc_score
    from IPython.display import HTML, display

    assert "options" in globals()
    assert "y12" in globals()
    assert "ids12" in globals()

    names = {
        "base": "V11_V9_80_xgb20",
        "v9": "V9_reference",
        "v8": "V8_reference",
        "xgb": "V11_XGBoost",
        "cat": "V11_CatBoost"
    }

    assert all(n in options for n in names.values()), (
        "Missing predictions in V11 notebook."
    )

    P = {}
    for short, name in names.items():
        a, b = options[name]
        P[short] = (
            np.asarray(a, dtype=float).ravel(),
            np.asarray(b, dtype=float).ravel()
        )

    def nonlinear_blend(weights, method="logit"):
        train_p = np.zeros(len(y12))
        test_p = np.zeros(len(ids12))

        for key, weight in weights.items():
            a, b = P[key]
            a = np.clip(a, 1e-6, 1 - 1e-6)
            b = np.clip(b, 1e-6, 1 - 1e-6)

            if method == "logit":
                train_p += weight * logit(a)
                test_p += weight * logit(b)
            else:
                train_p += weight * np.log(a)
                test_p += weight * np.log(b)

        if method == "logit":
            return expit(train_p), expit(test_p)

        return np.exp(train_p), np.exp(test_p)

    candidates = {
        "V12_LOGIT_BASE_CAT": nonlinear_blend(
            {"base": .75, "cat": .25}
        ),
        "V12_LOGIT_BASE_XGB": nonlinear_blend(
            {"base": .75, "xgb": .25}
        ),
        "V12_LOGIT_BASE_V9": nonlinear_blend(
            {"base": .75, "v9": .25}
        ),
        "V12_LOGIT_BASE_V8": nonlinear_blend(
            {"base": .75, "v8": .25}
        ),
        "V12_LOGIT_THREE_MODELS": nonlinear_blend(
            {"base": .60, "cat": .20, "xgb": .20}
        ),
        "V12_GEOMETRIC_BASE_CAT": nonlinear_blend(
            {"base": .80, "cat": .20},
            method="geometric"
        ),
        "V12_GEOMETRIC_BASE_XGB": nonlinear_blend(
            {"base": .80, "xgb": .20},
            method="geometric"
        )
    }

    baseline_auc = roc_auc_score(
        y12, P["base"][0]
    )

    results = sorted(
        [
            (roc_auc_score(y12, pair[0]), name)
            for name, pair in candidates.items()
        ],
        reverse=True
    )

    print("\nV12 LAST-CHANCE RESULTS")
    print("BASELINE:", round(baseline_auc, 6))

    for auc, name in results:
        print(
            name,
            "AUC:", round(auc, 6),
            "GAIN:", round(auc - baseline_auc, 6)
        )

    best_auc, best_name = results[0]

    if best_auc > baseline_auc:
        print("\nNEW OFFLINE WINNER:", best_name)

        predictions = candidates[best_name][1]

        output = pd.DataFrame({
            "encounter_id": ids12,
            "readmit_30d": predictions
        })

        assert output.encounter_id.is_unique
        assert np.isfinite(predictions).all()
        assert np.all(
            (predictions >= 0) & (predictions <= 1)
        )

        filename = "V12_LAST_CHALLENGER.csv"
        path = "/kaggle/working/" + filename
        output.to_csv(path, index=False)

        with open(path, "rb") as f:
            encoded = base64.b64encode(
                f.read()
            ).decode()

        display(HTML(
            '<a download="' + filename +
            '" href="data:text/csv;base64,' +
            encoded +
            '">DOWNLOAD ' + filename + '</a>'
        ))

    else:
        print(
            "\nNO IMPROVEMENT. "
            "DO NOT SPEND A SUBMISSION ON V12."
        )

    print(
        "\nTARGET 0.7600:",
        "REACHED" if best_auc >= .7600
        else "NOT REACHED"
    )


# Appendix B — Complete original NOVA notebook source (archived, not re-executed)

NOVA was an independent feature-engineering/model-comparison branch. It should **not** be concatenated into the active V11 global namespace, because it redefines data and model variables and was not used in the selected final Kaggle submission. Its original code is preserved below, cell-by-cell, with explanations.

### Archive N00 — Kaggle starter environment

**Original NOVA source cell 0; disabled for automatic execution.** Its results should be reported as a separate experiment, not as the final selected V11/V12 model.

In [ ]:
if False:  # ARCHIVED NOVA SOURCE — not part of Run All execution
    # This Python 3 environment comes with many helpful analytics libraries installed
    # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
    # For example, here's several helpful packages to load

    import numpy as np # linear algebra
    import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

    # Input data files are available in the read-only "../input/" directory
    # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

    import os
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

    # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
    # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

    # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
    # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

    import kagglehub
    # kagglehub.dataset_download('<owner>/<dataset-slug>')


### Archive N01 — NOVA one-row-per-encounter comprehensive data pipeline

**Original NOVA source cell 1; disabled for automatic execution.** Its results should be reported as a separate experiment, not as the final selected V11/V12 model.

In [ ]:
if False:  # ARCHIVED NOVA SOURCE — not part of Run All execution

    import re, gc
    from pathlib import Path
    import numpy as np
    import pandas as pd

    NOVA_ROOT = Path('/kaggle/input')
    NOVA_FILES = {
        p.name.lower(): p
        for p in NOVA_ROOT.rglob('*.csv')
    }

    def nova_read(filename, wanted=None):
        p = NOVA_FILES.get(filename.lower())
        if p is None:
            print('NOT FOUND:', filename, flush=True)
            return None

        if wanted is None:
            d = pd.read_csv(p, low_memory=False)
        else:
            d = pd.read_csv(
                p,
                usecols=lambda c: c.lower() in wanted,
                low_memory=False
            )

        d.columns = d.columns.str.strip().str.lower()
        print('READ:', filename, d.shape, flush=True)
        return d

    def nova_col(d, *names):
        if d is None:
            return None
        return next(
            (n for n in names if n in d.columns),
            None
        )

    def nova_dt(s):
        try:
            return pd.to_datetime(
                s, errors='coerce', format='mixed'
            )
        except (TypeError, ValueError):
            return pd.to_datetime(s, errors='coerce')

    def nova_safe(c):
        return not any(
            w in str(c).lower()
            for w in (
                'readmit', 'target', 'label',
                'outcome', 'future_', 'post_discharge',
                'next_admission', 'mortality',
                'date_of_death', 'death_date',
                'days_to_readm', 'days_until_readm',
                'readmission'
            )
        )

    NOVA_train = nova_read('train.csv')
    NOVA_test = nova_read('test.csv')

    assert NOVA_train is not None and NOVA_test is not None
    assert {
        'encounter_id', 'readmit_30d'
    }.issubset(NOVA_train.columns)
    assert 'encounter_id' in NOVA_test.columns

    NOVA_y = NOVA_train[
        'readmit_30d'
    ].to_numpy(dtype=np.int8)

    NOVA_n = len(NOVA_train)
    NOVA_test_ids = NOVA_test['encounter_id'].copy()

    NOVA_ids = pd.Index(
        pd.concat(
            [
                NOVA_train['encounter_id'],
                NOVA_test['encounter_id']
            ],
            ignore_index=True
        ).astype(str),
        name='encounter_id'
    )

    assert NOVA_ids.is_unique

    NOVA_F = pd.DataFrame(index=NOVA_ids)
    NOVA_AUDIT = []

    def nova_add(frame, name):
        global NOVA_F

        if frame is None or frame.empty:
            print('EMPTY:', name, flush=True)
            return

        frame = frame.copy()
        frame.index = frame.index.astype(str)

        if not frame.index.is_unique:
            raise ValueError(
                'Non-unique encounter aggregate: ' + name
            )

        frame = frame.loc[
            :, ~frame.columns.duplicated()
        ]

        frame.columns = [
            re.sub('[^a-zA-Z0-9_]', '_', str(c))
            for c in frame.columns
        ]

        frame = frame.loc[
            :, ~frame.columns.isin(NOVA_F.columns)
        ]

        if not len(frame.columns):
            return

        coverage = int(
            NOVA_ids.isin(frame.index).sum()
        )

        NOVA_AUDIT.append((
            name, len(frame.columns), coverage
        ))

        NOVA_F = pd.concat(
            [
                NOVA_F,
                frame.reindex(NOVA_ids)
            ],
            axis=1
        )

        print(
            'ADDED', name, len(frame.columns),
            'columns, coverage', coverage,
            flush=True
        )

    def nova_crosstab(d, key, field, prefix, top=30):
        if field is None or field not in d or d.empty:
            return

        v = (
            d[field].fillna('Missing')
            .astype(str).str.upper().str.strip()
        )

        top_vals = v.value_counts().head(top).index
        v = v.where(v.isin(top_vals), 'OTHER')

        tab = pd.crosstab(
            d[key].astype(str), v
        )

        tab.columns = [
            prefix + '_' + str(i)
            for i in range(len(tab.columns))
        ]

        nova_add(
            tab.astype(np.float32), prefix
        )

    def nova_event(
        name, fields, time_names=(),
        required_time=False
    ):
        d = nova_read(name, fields)

        if d is None or 'encounter_id' not in d:
            return None

        d = d.dropna(
            subset=['encounter_id']
        ).copy()

        d['encounter_id'] = (
            d['encounter_id'].astype(str)
        )

        d = d[
            d['encounter_id'].isin(NOVA_ids)
        ]

        tc = nova_col(d, *time_names)

        if tc is not None:
            t = nova_dt(d[tc])

            admit = d['encounter_id'].map(
                NOVA_admit
            )

            discharge = d['encounter_id'].map(
                NOVA_discharge
            )

            good = (
                t.notna()
                & admit.notna()
                & discharge.notna()
                & t.ge(admit)
                & t.le(discharge)
            )

            print(
                'TIME-CHECK', name,
                'retained', int(good.sum()),
                'of', len(d),
                flush=True
            )

            d = d.loc[good].copy()
            d['_time'] = t.loc[good]
            d['_admit'] = admit.loc[good]
            d['_discharge'] = discharge.loc[good]

        elif required_time:
            print(
                'SKIP', name,
                ': no usable timestamp',
                flush=True
            )
            return None

        else:
            print(
                'ENCOUNTER-SCOPED (no timestamp):',
                name, flush=True
            )

        return d


    # 1. Encounter, patient and hospital information

    NOVA_idx = nova_read('index_encounters.csv')

    assert (
        NOVA_idx is not None
        and 'encounter_id' in NOVA_idx
    )

    NOVA_idx['encounter_id'] = (
        NOVA_idx['encounter_id'].astype(str)
    )

    assert NOVA_idx['encounter_id'].is_unique

    NOVA_idx = (
        NOVA_idx.set_index('encounter_id')
        .reindex(NOVA_ids)
    )

    NOVA_admit_col = nova_col(
        NOVA_idx,
        'admit_dt', 'admission_dt',
        'admit_date', 'admission_date'
    )

    NOVA_discharge_col = nova_col(
        NOVA_idx,
        'discharge_dt', 'discharge_date',
        'discharged_dt'
    )

    assert NOVA_admit_col and NOVA_discharge_col, (
        'Admission/discharge date columns not found'
    )

    NOVA_admit = nova_dt(
        NOVA_idx[NOVA_admit_col]
    )
    NOVA_admit.index = NOVA_ids

    NOVA_discharge = nova_dt(
        NOVA_idx[NOVA_discharge_col]
    )
    NOVA_discharge.index = NOVA_ids

    NOVA_F['nova_stay_days'] = (
        NOVA_discharge - NOVA_admit
    ).dt.total_seconds() / 86400

    NOVA_F['nova_admit_month'] = (
        NOVA_admit.dt.month
    )
    NOVA_F['nova_admit_weekday'] = (
        NOVA_admit.dt.dayofweek
    )
    NOVA_F['nova_discharge_weekday'] = (
        NOVA_discharge.dt.dayofweek
    )
    NOVA_F['nova_discharge_hour'] = (
        NOVA_discharge.dt.hour
    )
    NOVA_F['nova_weekend_discharge'] = (
        NOVA_discharge.dt.dayofweek >= 5
    ).astype('float32')

    NOVA_hosp_groups = (
        NOVA_idx['hospital_id']
        .iloc[:NOVA_n]
        .reset_index(drop=True)
        if 'hospital_id' in NOVA_idx
        else None
    )

    NOVA_pid = (
        NOVA_idx['patient_id'].astype(str)
        if 'patient_id' in NOVA_idx
        else None
    )

    for c in NOVA_idx:
        if (
            c in (
                NOVA_admit_col,
                NOVA_discharge_col,
                'patient_id'
            )
            or not nova_safe(c)
        ):
            continue

        if (
            'date' in c
            or c.endswith('_dt')
            or 'timestamp' in c
        ):
            continue

        NOVA_F['nova_index_' + c] = (
            NOVA_idx[c].to_numpy()
        )

    for fname, joincol, prefix in [
        ('patients.csv', 'patient_id', 'patient'),
        ('hospitals.csv', 'hospital_id', 'hospital')
    ]:
        d = nova_read(fname)

        if (
            d is None
            or joincol not in d
            or joincol not in NOVA_idx
        ):
            continue

        d[joincol] = d[joincol].astype(str)

        d = (
            d.drop_duplicates(joincol)
            .set_index(joincol)
        )

        for c in d.columns:
            if (
                not nova_safe(c)
                or 'date' in c
                or c.endswith('_dt')
                or 'timestamp' in c
                or 'birth' in c
                or c == 'dob'
            ):
                continue

            NOVA_F[
                'nova_' + prefix + '_' + c
            ] = (
                NOVA_idx[joincol]
                .astype(str)
                .map(d[c])
                .to_numpy()
            )

        if prefix == 'patient':
            dob = nova_col(
                d, 'dob', 'birth_date',
                'date_of_birth'
            )

            if dob:
                birth = nova_dt(
                    NOVA_idx[joincol]
                    .astype(str)
                    .map(d[dob])
                )

                age = (
                    NOVA_admit - birth
                ).dt.days / 365.25

                NOVA_F['nova_age_from_birth'] = (
                    age.where(age.between(0, 115))
                )

        del d


    # 2. Diagnosis codes and clinical conditions

    NOVA_d = nova_event(
        'diagnoses.csv',
        {
            'encounter_id', 'icd10_code',
            'icd_10_code', 'diagnosis_code',
            'code', 'diagnosis_type', 'is_primary'
        }
    )

    if NOVA_d is not None and len(NOVA_d):
        g = NOVA_d.groupby('encounter_id')

        nova_add(
            g.size().to_frame('nova_dx_count'),
            'diagnoses_count'
        )

        c = nova_col(
            NOVA_d, 'icd10_code',
            'icd_10_code', 'diagnosis_code', 'code'
        )

        if c:
            NOVA_d['_code'] = (
                NOVA_d[c].fillna('UNK')
                .astype(str).str.upper()
                .str.replace('.', '', regex=False)
            )

            NOVA_d['_three'] = (
                NOVA_d['_code'].str[:3]
            )

            NOVA_d['_chapter'] = (
                NOVA_d['_code'].str[:1]
            )

            nova_add(
                g['_three'].nunique().to_frame(
                    'nova_dx_unique'
                ),
                'diagnoses_unique'
            )

            nova_crosstab(
                NOVA_d, 'encounter_id',
                '_three', 'nova_dx3', 65
            )

            nova_crosstab(
                NOVA_d, 'encounter_id',
                '_chapter', 'nova_dxchapter', 26
            )

            for label, regex in [
                ('diabetes', r'^E1[0-4]'),
                ('heart_failure', r'^I50'),
                ('kidney', r'^N18'),
                ('copd', r'^J44'),
                ('malignancy', r'^C'),
                ('sepsis', r'^A4[01]')
            ]:
                f = NOVA_d['_three'].str.contains(
                    regex, regex=True, na=False
                )

                nova_add(
                    f.groupby(
                        NOVA_d.encounter_id
                    ).max().astype('float32').to_frame(
                        'nova_dx_' + label
                    ),
                    'dx_' + label
                )

        del NOVA_d


    # 3. Medications (fixes V10's lost records)

    NOVA_d = nova_event(
        'medications.csv',
        {
            'encounter_id', 'drug_class',
            'med_class', 'medication_class',
            'drug_name', 'medication_name',
            'continued_at_discharge'
        }
    )

    if NOVA_d is not None and len(NOVA_d):
        g = NOVA_d.groupby('encounter_id')

        nova_add(
            g.size().to_frame('nova_med_count'),
            'med_count'
        )

        c = nova_col(
            NOVA_d, 'drug_class',
            'med_class', 'medication_class'
        )

        if c:
            nova_add(
                g[c].nunique().to_frame(
                    'nova_med_unique_classes'
                ),
                'med_classes'
            )

            nova_crosstab(
                NOVA_d, 'encounter_id',
                c, 'nova_medclass', 45
            )

        c = nova_col(
            NOVA_d, 'drug_name',
            'medication_name'
        )

        if c:
            nova_crosstab(
                NOVA_d, 'encounter_id',
                c, 'nova_medname', 18
            )

        c = nova_col(
            NOVA_d, 'continued_at_discharge'
        )

        if c:
            z = (
                NOVA_d[c].fillna('')
                .astype(str).str.lower()
                .isin(['1', 'true', 'yes', 'y'])
            )

            nova_add(
                z.groupby(
                    NOVA_d.encounter_id
                ).sum().to_frame(
                    'nova_med_continued'
                ),
                'med_discharge'
            )

        del NOVA_d


    # 4. Procedures and unit transfers

    for fname, pref, options, top in [
        (
            'procedures.csv', 'nova_proc',
            [
                'procedure_code',
                'procedure_type',
                'procedure_name', 'code'
            ],
            30
        ),
        (
            'unit_transfers.csv', 'nova_transfer',
            [
                'unit_type', 'to_unit',
                'unit_name', 'care_unit'
            ],
            18
        )
    ]:
        NOVA_d = nova_event(
            fname, {'encounter_id', *options}
        )

        if NOVA_d is None or NOVA_d.empty:
            continue

        nova_add(
            NOVA_d.groupby(
                'encounter_id'
            ).size().to_frame(pref + '_count'),
            pref + '_count'
        )

        c = nova_col(NOVA_d, *options)

        if c:
            nova_crosstab(
                NOVA_d, 'encounter_id',
                c, pref, top
            )

            if 'transfer' in pref:
                icu = (
                    NOVA_d[c].fillna('')
                    .astype(str)
                    .str.contains(
                        'ICU|INTENSIVE|CRITICAL',
                        case=False, regex=True
                    )
                )

                nova_add(
                    icu.groupby(
                        NOVA_d.encounter_id
                    ).sum().to_frame(
                        'nova_icu_count'
                    ),
                    'icu_count'
                )

        del NOVA_d


    # 5. Discharge planning

    NOVA_d = nova_read(
        'discharge_planning.csv'
    )

    if (
        NOVA_d is not None
        and 'encounter_id' in NOVA_d
    ):
        NOVA_d['encounter_id'] = (
            NOVA_d.encounter_id.astype(str)
        )

        NOVA_d = NOVA_d[
            NOVA_d.encounter_id.isin(NOVA_ids)
        ]

        nova_add(
            NOVA_d.groupby(
                'encounter_id'
            ).size().to_frame(
                'nova_plan_count'
            ),
            'plan_count'
        )

        plan_cols = [
            c for c in NOVA_d.columns
            if c != 'encounter_id'
            and nova_safe(c)
            and not any(
                w in c
                for w in [
                    'date', '_dt', 'time',
                    'note', 'text',
                    'description', '_id'
                ]
            )
        ]

        for c in plan_cols[:12]:
            if NOVA_d[c].nunique(
                dropna=True
            ) <= 25:
                nova_crosstab(
                    NOVA_d, 'encounter_id',
                    c, 'nova_plan_' + c, 12
                )

        del NOVA_d


    # 6. Historical utilization before admission

    NOVA_d = nova_read(
        'prior_utilization.csv',
        {
            'patient_id', 'visit_dt',
            'visit_date', 'visit_type',
            'encounter_type', 'care_setting'
        }
    )

    if (
        NOVA_d is not None
        and NOVA_pid is not None
        and 'patient_id' in NOVA_d
    ):
        c = nova_col(
            NOVA_d, 'visit_dt', 'visit_date'
        )

        if c:
            NOVA_d['patient_id'] = (
                NOVA_d.patient_id.astype(str)
            )

            NOVA_d['_visit'] = nova_dt(
                NOVA_d[c]
            )

            NOVA_map = pd.DataFrame({
                'encounter_id': NOVA_ids.to_numpy(),
                'patient_id': NOVA_pid.to_numpy(),
                'admit': NOVA_admit.to_numpy()
            })

            NOVA_d = NOVA_d.merge(
                NOVA_map,
                on='patient_id',
                how='inner',
                validate='many_to_many'
            )

            NOVA_d['_days'] = (
                NOVA_d.admit - NOVA_d._visit
            ).dt.total_seconds() / 86400

            NOVA_d = NOVA_d[
                NOVA_d._days.gt(0)
                & NOVA_d._days.le(3650)
            ]

            g = NOVA_d.groupby('encounter_id')

            f = pd.DataFrame({
                'nova_prior_count': g.size(),
                'nova_prior_recency_days': g._days.min()
            })

            for days in (
                7, 30, 90, 180, 365, 730
            ):
                f[
                    'nova_prior_' + str(days) + 'd'
                ] = (
                    NOVA_d._days.le(days)
                    .groupby(NOVA_d.encounter_id)
                    .sum()
                )

            nova_add(f, 'prior_visits')

            c = nova_col(
                NOVA_d, 'visit_type',
                'encounter_type', 'care_setting'
            )

            if c:
                nova_crosstab(
                    NOVA_d, 'encounter_id',
                    c, 'nova_prior_type', 12
                )

            del NOVA_map

        del NOVA_d


    # 7. Laboratory results and abnormality trends

    NOVA_d = nova_event(
        'lab_results.csv',
        {
            'encounter_id', 'collected_dt',
            'collection_dt', 'result_dt',
            'lab_dt', 'charttime',
            'lab_name', 'test_name',
            'lab_test', 'analyte',
            'value', 'result_value',
            'numeric_value', 'lab_value',
            'result', 'abnormal_flag',
            'flag', 'abnormality'
        },
        (
            'collected_dt', 'collection_dt',
            'result_dt', 'lab_dt', 'charttime'
        ),
        required_time=True
    )

    if NOVA_d is not None and len(NOVA_d):
        g = NOVA_d.groupby('encounter_id')

        nova_add(
            g.size().to_frame(
                'nova_lab_count'
            ),
            'lab_count'
        )

        c = nova_col(
            NOVA_d,
            'abnormal_flag', 'flag',
            'abnormality'
        )

        if c:
            flag = (
                NOVA_d[c].fillna('')
                .astype(str).str.upper().str.strip()
            )

            bad = flag.isin([
                'H', 'L', 'HH', 'LL',
                'HIGH', 'LOW',
                'ABNORMAL', 'ABN', 'A',
                'CRITICAL', 'CRIT'
            ])

            f = pd.DataFrame({
                'nova_lab_abnormal_count':
                    bad.groupby(
                        NOVA_d.encounter_id
                    ).sum(),
                'nova_lab_abnormal_rate':
                    bad.groupby(
                        NOVA_d.encounter_id
                    ).mean()
            })

            nova_add(f, 'lab_abnormal')

            last24 = (
                NOVA_d._discharge - NOVA_d._time
            ).dt.total_seconds().between(
                0, 86400
            )

            f2 = pd.DataFrame({
                'nova_lab_last24_count':
                    last24.groupby(
                        NOVA_d.encounter_id
                    ).sum(),
                'nova_lab_last24_abnormal':
                    (last24 & bad).groupby(
                        NOVA_d.encounter_id
                    ).sum()
            })

            nova_add(f2, 'lab_last24')

        name = nova_col(
            NOVA_d,
            'lab_name', 'test_name',
            'lab_test', 'analyte'
        )

        value = nova_col(
            NOVA_d,
            'value', 'result_value',
            'numeric_value', 'lab_value',
            'result'
        )

        if name:
            nova_crosstab(
                NOVA_d, 'encounter_id',
                name, 'nova_labtest', 28
            )

        if name and value:
            NOVA_d['_num'] = pd.to_numeric(
                NOVA_d[value], errors='coerce'
            )

            NOVA_d['_kind'] = (
                NOVA_d[name].fillna('Unknown')
                .astype(str).str.upper().str.strip()
            )

            top = (
                NOVA_d['_kind']
                .value_counts()
                .head(20).index
            )

            sm = NOVA_d[
                NOVA_d._kind.isin(top)
                & NOVA_d._num.notna()
            ].sort_values('_time')

            if len(sm):
                stat = sm.groupby(
                    ['encounter_id', '_kind']
                )['_num'].agg([
                    'mean', 'std', 'min',
                    'max', 'first', 'last'
                ])

                stat['delta'] = (
                    stat['last'] - stat['first']
                )

                wide = stat.unstack('_kind')

                wide.columns = [
                    'nova_labnum_' + str(i)
                    for i in range(wide.shape[1])
                ]

                nova_add(
                    wide, 'lab_values'
                )

            del sm

        del NOVA_d
        gc.collect()


    # 8. Vital signs and clinical trajectories

    NOVA_d = nova_event(
        'vitals.csv',
        {
            'encounter_id', 'charttime',
            'recorded_dt', 'vital_dt',
            'event_dt', 'vital_type',
            'vital_name', 'measurement_type',
            'value', 'vital_value',
            'numeric_value'
        },
        (
            'charttime', 'recorded_dt',
            'vital_dt', 'event_dt'
        ),
        required_time=True
    )

    if NOVA_d is not None and len(NOVA_d):
        name = nova_col(
            NOVA_d,
            'vital_type', 'vital_name',
            'measurement_type'
        )

        value = nova_col(
            NOVA_d,
            'value', 'vital_value',
            'numeric_value'
        )

        if name and value:
            NOVA_d['_kind'] = (
                NOVA_d[name].fillna('UNK')
                .astype(str).str.upper().str.strip()
            )

            NOVA_d['_num'] = pd.to_numeric(
                NOVA_d[value], errors='coerce'
            )

            temp = (
                NOVA_d['_kind'].isin([
                    'TEMP', 'TEMPERATURE'
                ])
                & NOVA_d._num.between(45, 115)
            )

            NOVA_d.loc[temp, '_num'] = (
                NOVA_d.loc[temp, '_num'] - 32
            ) * 5 / 9

            NOVA_d = NOVA_d.sort_values('_time')

            top = (
                NOVA_d['_kind']
                .value_counts()
                .head(10).index
            )

            NOVA_d = NOVA_d[
                NOVA_d._kind.isin(top)
            ]

            stat = NOVA_d.groupby(
                ['encounter_id', '_kind']
            )['_num'].agg([
                'mean', 'std', 'min',
                'max', 'first', 'last', 'count'
            ])

            stat['delta'] = (
                stat['last'] - stat['first']
            )

            wide = stat.unstack('_kind')

            wide.columns = [
                'nova_vital_' + str(i)
                for i in range(wide.shape[1])
            ]

            nova_add(
                wide, 'vital_stats'
            )

            last24 = (
                NOVA_d._discharge - NOVA_d._time
            ).dt.total_seconds().between(
                0, 86400
            )

            last = (
                NOVA_d[last24]
                .groupby(
                    ['encounter_id', '_kind']
                )['_num']
                .mean()
                .unstack('_kind')
            )

            last.columns = [
                'nova_vital_last24_' + str(i)
                for i in range(last.shape[1])
            ]

            nova_add(
                last, 'vital_last24'
            )

        del NOVA_d
        gc.collect()


    # 9. Cross-table clinical interactions

    for a, b, c in [
        (
            'nova_lab_abnormal_count',
            'nova_lab_count',
            'nova_abnormal_fraction'
        ),
        (
            'nova_med_count',
            'nova_stay_days',
            'nova_med_per_day'
        ),
        (
            'nova_dx_count',
            'nova_stay_days',
            'nova_dx_per_day'
        ),
        (
            'nova_transfer_count',
            'nova_stay_days',
            'nova_transfers_per_day'
        ),
        (
            'nova_prior_30d',
            'nova_prior_365d',
            'nova_prior_recent_share'
        )
    ]:
        if a in NOVA_F and b in NOVA_F:
            NOVA_F[c] = (
                pd.to_numeric(
                    NOVA_F[a], errors='coerce'
                )
                /
                (
                    1 + pd.to_numeric(
                        NOVA_F[b], errors='coerce'
                    )
                )
            )


    # 10. Final master dataset

    for c in list(NOVA_F.columns):
        if (
            not nova_safe(c)
            or pd.api.types.is_datetime64_any_dtype(
                NOVA_F[c]
            )
        ):
            NOVA_F.drop(
                columns=c, inplace=True
            )

        elif NOVA_F[c].dtype == 'object':
            NOVA_F[c] = (
                NOVA_F[c].fillna('Missing')
                .astype(str)
            )

            if (
                NOVA_F[c].nunique() > 200
                or NOVA_F[c].nunique()
                > len(NOVA_F) * 0.2
            ):
                NOVA_F.drop(
                    columns=c, inplace=True
                )

    NOVA_F = NOVA_F.replace(
        [np.inf, -np.inf], np.nan
    )

    NOVA_F['nova_missing_count'] = (
        NOVA_F.isna()
        .sum(axis=1)
        .astype(np.float32)
    )

    NOVA_F = NOVA_F.loc[
        :, ~NOVA_F.columns.duplicated()
    ]

    NOVA_X = (
        NOVA_F.iloc[:NOVA_n]
        .reset_index(drop=True)
    )

    NOVA_T = (
        NOVA_F.iloc[NOVA_n:]
        .reset_index(drop=True)
    )

    assert len(NOVA_X) == len(NOVA_train)
    assert len(NOVA_T) == len(NOVA_test)
    assert NOVA_X.columns.equals(NOVA_T.columns)

    print(
        '\nNOVA DATA READY:',
        NOVA_X.shape,
        NOVA_T.shape,
        flush=True
    )

    print(
        pd.DataFrame(
            NOVA_AUDIT,
            columns=[
                'source',
                'features',
                'matched_encounters'
            ]
        ).to_string(index=False)
    )


### Archive N02 — NOVA small/medium LightGBM, linear model and course baselines

**Original NOVA source cell 2; disabled for automatic execution.** Its results should be reported as a separate experiment, not as the final selected V11/V12 model.

In [ ]:
if False:  # ARCHIVED NOVA SOURCE — not part of Run All execution

    import gc
    import numpy as np
    import pandas as pd

    from scipy.sparse import csr_matrix, hstack
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score
    )

    from lightgbm import (
        LGBMClassifier,
        early_stopping,
        log_evaluation
    )

    assert (
        'NOVA_X' in globals()
        and 'NOVA_T' in globals()
    ), 'Run NOVA Cell 1 first'

    NOVA_X2 = NOVA_X.copy()
    NOVA_T2 = NOVA_T.copy()

    # Remove constant and almost-empty features

    NOVA_keep = [
        c for c in NOVA_X2.columns
        if (
            NOVA_X2[c].notna().sum()
            >= max(25, int(.001 * len(NOVA_X2)))
            and NOVA_X2[c].nunique(
                dropna=True
            ) > 1
        )
    ]

    NOVA_X2 = NOVA_X2[NOVA_keep]
    NOVA_T2 = NOVA_T2[NOVA_keep]

    NOVA_cat = NOVA_X2.select_dtypes(
        exclude='number'
    ).columns.tolist()

    NOVA_num = [
        c for c in NOVA_X2.columns
        if c not in NOVA_cat
    ]

    for c in NOVA_cat:
        vals = pd.concat(
            [
                NOVA_X2[c],
                NOVA_T2[c]
            ],
            ignore_index=True
        ).fillna('Missing').astype(str)

        levels = pd.Categorical(
            vals
        ).categories

        NOVA_X2[c] = pd.Categorical(
            NOVA_X2[c].fillna('Missing')
            .astype(str),
            categories=levels
        )

        NOVA_T2[c] = pd.Categorical(
            NOVA_T2[c].fillna('Missing')
            .astype(str),
            categories=levels
        )

    for c in NOVA_num:
        NOVA_X2[c] = pd.to_numeric(
            NOVA_X2[c], errors='coerce'
        ).astype(np.float32)

        NOVA_T2[c] = pd.to_numeric(
            NOVA_T2[c], errors='coerce'
        ).astype(np.float32)

    NOVA_folds = list(
        StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=42
        ).split(NOVA_X2, NOVA_y)
    )

    NOVA_oof = {}
    NOVA_pred = {}
    NOVA_fold_report = []

    # Two decision-tree boosting configurations

    NOVA_models = {
        'TREE_SMALL': dict(
            num_leaves=9,
            min_child_samples=120,
            reg_lambda=18,
            reg_alpha=1.0,
            max_depth=5
        ),
        'TREE_MEDIUM': dict(
            num_leaves=19,
            min_child_samples=75,
            reg_lambda=12,
            reg_alpha=0.6,
            max_depth=-1
        )
    }

    for label, params in NOVA_models.items():
        NOVA_oof[label] = np.zeros(
            len(NOVA_y)
        )

        NOVA_pred[label] = np.zeros(
            len(NOVA_T2)
        )

        for k, (tr, va) in enumerate(
            NOVA_folds, 1
        ):
            print(
                'NOVA', label,
                'FOLD', k, '/3',
                flush=True
            )

            m = LGBMClassifier(
                n_estimators=1200,
                learning_rate=0.035,
                objective='binary',
                colsample_bytree=0.82,
                subsample=0.85,
                subsample_freq=1,
                random_state=500 + k,
                n_jobs=4,
                verbosity=-1,
                **params
            )

            m.fit(
                NOVA_X2.iloc[tr],
                NOVA_y[tr],
                eval_set=[
                    (
                        NOVA_X2.iloc[va],
                        NOVA_y[va]
                    )
                ],
                eval_metric='auc',
                callbacks=[
                    early_stopping(
                        95, verbose=False
                    ),
                    log_evaluation(0)
                ]
            )

            NOVA_oof[label][va] = (
                m.predict_proba(
                    NOVA_X2.iloc[va]
                )[:, 1]
            )

            NOVA_pred[label] += (
                m.predict_proba(
                    NOVA_T2
                )[:, 1] / 3
            )

            auc = roc_auc_score(
                NOVA_y[va],
                NOVA_oof[label][va]
            )

            trainauc = roc_auc_score(
                NOVA_y[tr],
                m.predict_proba(
                    NOVA_X2.iloc[tr]
                )[:, 1]
            )

            NOVA_fold_report.append((
                label,
                k,
                auc,
                trainauc,
                trainauc - auc,
                m.best_iteration_
            ))

            print(
                'AUC:', round(auc, 6),
                'TRAIN-VALID GAP:',
                round(trainauc - auc, 5),
                flush=True
            )

            del m
            gc.collect()


    # Prepare sparse data for the linear model

    NOVA_median = (
        NOVA_X2[NOVA_num]
        .median()
        .fillna(0)
    )

    NOVA_lo = NOVA_X2[
        NOVA_num
    ].quantile(.05)

    NOVA_hi = NOVA_X2[
        NOVA_num
    ].quantile(.95)

    NOVA_scale = (
        (NOVA_hi - NOVA_lo)
        .replace(0, 1)
        .fillna(1)
    )

    def nova_numeric_matrix(df):
        a = (
            df[NOVA_num].fillna(
                NOVA_median
            ) - NOVA_median
        ) / NOVA_scale

        return csr_matrix(
            np.clip(
                a.to_numpy(
                    dtype=np.float32
                ),
                -10, 10
            )
        )

    NOVA_linear_x = nova_numeric_matrix(
        NOVA_X2
    )

    NOVA_linear_t = nova_numeric_matrix(
        NOVA_T2
    )

    if NOVA_cat:
        enc = OneHotEncoder(
            handle_unknown='ignore',
            min_frequency=20,
            dtype=np.float32
        )

        both = pd.concat(
            [
                NOVA_X2[NOVA_cat],
                NOVA_T2[NOVA_cat]
            ],
            ignore_index=True
        ).astype(str)

        enc.fit(both)

        NOVA_linear_x = hstack(
            [
                NOVA_linear_x,
                enc.transform(
                    NOVA_X2[NOVA_cat].astype(str)
                )
            ],
            format='csr'
        )

        NOVA_linear_t = hstack(
            [
                NOVA_linear_t,
                enc.transform(
                    NOVA_T2[NOVA_cat].astype(str)
                )
            ],
            format='csr'
        )

        del both, enc


    # Classroom Decision Tree with entropy

    tr0, va0 = NOVA_folds[0]

    NOVA_dt_model = DecisionTreeClassifier(
        criterion='entropy',
        max_depth=6,
        min_samples_leaf=100,
        random_state=42
    )

    NOVA_dt_model.fit(
        NOVA_linear_x[tr0],
        NOVA_y[tr0]
    )

    print(
        'NOVA ENTROPY DECISION TREE (fold 1):',
        round(
            roc_auc_score(
                NOVA_y[va0],
                NOVA_dt_model.predict_proba(
                    NOVA_linear_x[va0]
                )[:, 1]
            ),
            6
        ),
        flush=True
    )

    del NOVA_dt_model


    # Regularized linear classifier

    NOVA_oof['LINEAR_L2'] = np.zeros(
        len(NOVA_y)
    )

    NOVA_pred['LINEAR_L2'] = np.zeros(
        len(NOVA_T2)
    )

    for k, (tr, va) in enumerate(
        NOVA_folds, 1
    ):
        print(
            'NOVA LINEAR_L2 FOLD',
            k, '/3',
            flush=True
        )

        lr = LogisticRegression(
            C=0.12,
            solver='liblinear',
            max_iter=250,
            random_state=42
        )

        lr.fit(
            NOVA_linear_x[tr],
            NOVA_y[tr]
        )

        NOVA_oof['LINEAR_L2'][va] = (
            lr.predict_proba(
                NOVA_linear_x[va]
            )[:, 1]
        )

        NOVA_pred['LINEAR_L2'] += (
            lr.predict_proba(
                NOVA_linear_t
            )[:, 1] / 3
        )

        valauc = roc_auc_score(
            NOVA_y[va],
            NOVA_oof['LINEAR_L2'][va]
        )

        trainauc = roc_auc_score(
            NOVA_y[tr],
            lr.predict_proba(
                NOVA_linear_x[tr]
            )[:, 1]
        )

        NOVA_fold_report.append((
            'LINEAR_L2',
            k,
            valauc,
            trainauc,
            trainauc - valauc,
            None
        ))

        print(
            'AUC:', round(valauc, 6),
            flush=True
        )

        del lr
        gc.collect()


    # Compare independent models and ensembles

    NOVA_candidates = {
        'TREE_SMALL': ('TREE_SMALL',),
        'TREE_MEDIUM': ('TREE_MEDIUM',),
        'LINEAR_L2': ('LINEAR_L2',),
        'TREE_50_50': (
            'TREE_SMALL',
            'TREE_MEDIUM'
        ),
        'TREE_80_LINEAR20': (
            'TREE_SMALL',
            'TREE_MEDIUM',
            'LINEAR_L2'
        ),
        'TREE_60_LINEAR40': (
            'TREE_SMALL',
            'TREE_MEDIUM',
            'LINEAR_L2',
            'LINEAR_L2'
        )
    }

    def nova_blend(name, predictions):
        parts = NOVA_candidates[name]

        if name == 'TREE_80_LINEAR20':
            return (
                .4 * predictions['TREE_SMALL']
                + .4 * predictions['TREE_MEDIUM']
                + .2 * predictions['LINEAR_L2']
            )

        if name == 'TREE_60_LINEAR40':
            return (
                .3 * predictions['TREE_SMALL']
                + .3 * predictions['TREE_MEDIUM']
                + .4 * predictions['LINEAR_L2']
            )

        return np.mean(
            [
                predictions[p]
                for p in parts
            ],
            axis=0
        )

    NOVA_scores = []

    for name in NOVA_candidates:
        p = nova_blend(
            name, NOVA_oof
        )

        NOVA_scores.append((
            roc_auc_score(
                NOVA_y, p
            ),
            average_precision_score(
                NOVA_y, p
            ),
            name
        ))

    NOVA_scores.sort(reverse=True)

    print(
        '\nNOVA MODEL COMPARISON (3-fold OOF):',
        flush=True
    )

    for auc, ap, name in NOVA_scores:
        print(
            f'{name:24s} '
            f'ROC-AUC={auc:.6f} '
            f'PR-AUC={ap:.6f}',
            flush=True
        )

    print(
        '\nNOVA FOLD REPORT:',
        flush=True
    )

    print(
        pd.DataFrame(
            NOVA_fold_report,
            columns=[
                'model', 'fold',
                'validation_auc',
                'training_auc',
                'overfit_gap',
                'best_iteration'
            ]
        ).round(5).to_string(index=False)
    )

    print(
        '\nV9 historical OOF: 0.748383; '
        'public: 0.74214',
        flush=True
    )


### Archive N03 — NOVA ensemble selection, temporal and hospital checks, CSV outputs

**Original NOVA source cell 3; disabled for automatic execution.** Its results should be reported as a separate experiment, not as the final selected V11/V12 model.

In [ ]:
if False:  # ARCHIVED NOVA SOURCE — not part of Run All execution

    import base64
    from IPython.display import HTML, display
    from sklearn.model_selection import GroupShuffleSplit

    NOVA_best_auc, NOVA_best_ap, NOVA_best_name = (
        NOVA_scores[0]
    )

    print(
        'NOVA WINNER:',
        NOVA_best_name,
        'CV ROC-AUC:',
        round(NOVA_best_auc, 6),
        flush=True
    )

    def nova_stress(title, tr, va):
        if (
            len(tr) < 1000
            or len(va) < 500
            or len(np.unique(NOVA_y[va])) < 2
        ):
            print(
                title, ': insufficient samples'
            )
            return

        m = LGBMClassifier(
            n_estimators=220,
            learning_rate=0.035,
            num_leaves=9,
            min_child_samples=120,
            reg_lambda=18,
            reg_alpha=1,
            colsample_bytree=.82,
            subsample=.85,
            subsample_freq=1,
            random_state=42,
            n_jobs=4,
            verbosity=-1
        )

        m.fit(
            NOVA_X2.iloc[tr],
            NOVA_y[tr]
        )

        trainauc = roc_auc_score(
            NOVA_y[tr],
            m.predict_proba(
                NOVA_X2.iloc[tr]
            )[:, 1]
        )

        valauc = roc_auc_score(
            NOVA_y[va],
            m.predict_proba(
                NOVA_X2.iloc[va]
            )[:, 1]
        )

        print(
            f'{title}: '
            f'train={trainauc:.5f} '
            f'holdout={valauc:.5f} '
            f'gap={trainauc-valauc:.5f} '
            f'n={len(va)}',
            flush=True
        )

    print(
        '\nNOVA GENERALIZATION CHECKS:',
        flush=True
    )

    # Later discharge dates

    NOVA_end = (
        NOVA_discharge.iloc[:NOVA_n]
        .reset_index(drop=True)
    )

    valid = NOVA_end.notna().to_numpy()

    if (
        valid.sum() >= 10000
        and NOVA_end[valid].nunique() > 20
    ):
        order = np.flatnonzero(valid)[
            np.argsort(
                NOVA_end[valid].to_numpy()
            )
        ]

        cut = int(len(order) * .8)

        nova_stress(
            'LATER DISCHARGES',
            order[:cut],
            order[cut:]
        )
    else:
        print(
            'LATER DISCHARGES: unavailable'
        )

    # Hospitals not seen during training

    if (
        NOVA_hosp_groups is not None
        and NOVA_hosp_groups.nunique() > 5
    ):
        groups = (
            NOVA_hosp_groups.fillna('Missing')
            .astype(str).to_numpy()
        )

        tr, va = next(
            GroupShuffleSplit(
                n_splits=1,
                test_size=.2,
                random_state=42
            ).split(
                NOVA_X2,
                NOVA_y,
                groups=groups
            )
        )

        nova_stress(
            'UNSEEN HOSPITALS',
            tr, va
        )
    else:
        print(
            'UNSEEN HOSPITALS: unavailable'
        )


    # Save two independently named candidates

    NOVA_tree_name = max(
        (
            t for t in NOVA_scores
            if 'LINEAR' not in t[2]
        ),
        key=lambda t: t[0]
    )[2]

    NOVA_exports = [
        ('NOVA_BEST', NOVA_best_name),
        ('NOVA_TREE_ONLY', NOVA_tree_name)
    ]

    for filename, name in NOVA_exports:
        pred = np.asarray(
            nova_blend(
                name, NOVA_pred
            )
        )

        assert len(pred) == len(NOVA_test_ids)
        assert np.isfinite(pred).all()
        assert ((pred >= 0) & (pred <= 1)).all()

        out = pd.DataFrame({
            'encounter_id':
                NOVA_test_ids.to_numpy(),
            'readmit_30d': pred
        })

        assert out.encounter_id.is_unique

        path = (
            '/kaggle/working/'
            + filename + '.csv'
        )

        out.to_csv(
            path, index=False
        )

        print(
            'SAVED:', path,
            'candidate:', name,
            flush=True
        )

        with open(path, 'rb') as f:
            encoded = base64.b64encode(
                f.read()
            ).decode('ascii')

        display(
            HTML(
                '<a download="'
                + filename + '.csv" '
                + 'href="data:text/csv;base64,'
                + encoded
                + '">DOWNLOAD '
                + filename + '.csv</a>'
            )
        )

    pd.DataFrame(
        NOVA_AUDIT,
        columns=[
            'source',
            'features',
            'matched_encounters'
        ]
    ).to_csv(
        '/kaggle/working/NOVA_SOURCE_AUDIT.csv',
        index=False
    )

    pd.DataFrame(
        NOVA_fold_report,
        columns=[
            'model', 'fold',
            'validation_auc',
            'training_auc',
            'overfit_gap',
            'best_iteration'
        ]
    ).to_csv(
        '/kaggle/working/NOVA_FOLD_REPORT.csv',
        index=False
    )

    print(
        '\nHistorical V9 OOF: 0.748383; '
        'public: 0.74214. '
        'Review results before submitting.',
        flush=True
    )


### Archive N04 — NOVA-BOOST exact diagnosis and last-24-hour laboratory features

**Original NOVA source cell 4; disabled for automatic execution.** Its results should be reported as a separate experiment, not as the final selected V11/V12 model.

In [ ]:
if False:  # ARCHIVED NOVA SOURCE — not part of Run All execution

    import numpy as np
    import pandas as pd
    import re

    needed = [
        'NOVA_X2', 'NOVA_T2', 'NOVA_y',
        'NOVA_ids', 'NOVA_n', 'NOVA_admit',
        'NOVA_discharge', 'nova_read', 'nova_dt'
    ]

    assert all(k in globals() for k in needed), (
        "Run NOVA Cells 1 and 2 first."
    )

    NB_all = pd.concat(
        [NOVA_X2, NOVA_T2],
        ignore_index=True
    ).copy()

    NB_added = []

    def nb_add(frame, label):
        global NB_all

        if frame is None or frame.empty:
            return

        frame = frame.copy()
        frame.index = frame.index.astype(str)

        frame = frame[
            ~frame.index.duplicated(keep='first')
        ]

        frame = frame.reindex(
            NOVA_ids
        ).reset_index(drop=True)

        frame.columns = [
            f'nb_{label}_{i}'
            for i in range(frame.shape[1])
        ]

        NB_all = pd.concat(
            [NB_all, frame],
            axis=1
        )

        NB_added.append((
            label,
            len(frame.columns),
            int(frame.notna().any(axis=1).sum())
        ))

        print(
            'ADDED', label,
            frame.shape[1], 'features',
            flush=True
        )


    # 1. Exact diagnosis codes, not only broad groups

    dx = nova_read(
        'diagnoses.csv',
        {
            'encounter_id',
            'icd10_code',
            'icd_10_code',
            'diagnosis_code',
            'diagnosis_type',
            'is_primary'
        }
    )

    if dx is not None and 'encounter_id' in dx:

        codecol = next(
            (
                c for c in [
                    'icd10_code',
                    'icd_10_code',
                    'diagnosis_code'
                ]
                if c in dx
            ),
            None
        )

        if codecol:
            dx = dx.dropna(
                subset=['encounter_id']
            ).copy()

            dx['encounter_id'] = (
                dx.encounter_id.astype(str)
            )

            dx['_code'] = (
                dx[codecol]
                .fillna('UNKNOWN')
                .astype(str)
                .str.upper()
                .str.replace(
                    r'[^A-Z0-9]', '',
                    regex=True
                )
            )

            keep = (
                dx['_code']
                .value_counts()
                .head(140)
                .index
            )

            subset = dx[
                dx['_code'].isin(keep)
            ]

            counts = pd.crosstab(
                subset.encounter_id,
                subset['_code']
            ).astype('float32')

            nb_add(
                counts,
                'exact_icd'
            )

            primary = pd.Series(
                False, index=dx.index
            )

            if 'is_primary' in dx:
                primary = (
                    dx.is_primary
                    .astype(str)
                    .str.lower()
                    .isin([
                        '1', 'true',
                        'yes', 'y'
                    ])
                )

            elif 'diagnosis_type' in dx:
                primary = (
                    dx.diagnosis_type
                    .astype(str)
                    .str.lower()
                    .str.contains(
                        'primary|principal',
                        regex=True
                    )
                )

            if primary.any():
                pp = dx.loc[primary].copy()

                pkeep = (
                    pp['_code']
                    .value_counts()
                    .head(45)
                    .index
                )

                pp = pp[
                    pp['_code'].isin(pkeep)
                ]

                nb_add(
                    pd.crosstab(
                        pp.encounter_id,
                        pp['_code']
                    ).astype('float32'),
                    'primary_icd'
                )

            del counts, subset

        del dx


    # 2. Last 24-hour numerical laboratory measurements

    lab = nova_read(
        'lab_results.csv',
        {
            'encounter_id',
            'collected_dt',
            'collection_dt',
            'result_dt',
            'lab_dt',
            'charttime',
            'lab_name',
            'test_name',
            'lab_test',
            'analyte',
            'value',
            'result_value',
            'numeric_value',
            'lab_value',
            'result'
        }
    )

    if lab is not None and 'encounter_id' in lab:

        tc = next(
            (
                c for c in [
                    'collected_dt',
                    'collection_dt',
                    'result_dt',
                    'lab_dt',
                    'charttime'
                ]
                if c in lab
            ),
            None
        )

        nc = next(
            (
                c for c in [
                    'lab_name',
                    'test_name',
                    'lab_test',
                    'analyte'
                ]
                if c in lab
            ),
            None
        )

        vc = next(
            (
                c for c in [
                    'value',
                    'result_value',
                    'numeric_value',
                    'lab_value',
                    'result'
                ]
                if c in lab
            ),
            None
        )

        if tc and nc and vc:

            lab['encounter_id'] = (
                lab.encounter_id.astype(str)
            )

            lab['_t'] = nova_dt(
                lab[tc]
            )

            lab['_end'] = (
                lab.encounter_id.map(
                    NOVA_discharge
                )
            )

            lab['_start'] = (
                lab.encounter_id.map(
                    NOVA_admit
                )
            )

            h = (
                lab['_end'] - lab['_t']
            ).dt.total_seconds() / 3600

            lab = lab[
                h.between(0, 24)
                & lab['_t'].ge(
                    lab['_start']
                )
            ].copy()

            lab['_val'] = pd.to_numeric(
                lab[vc],
                errors='coerce'
            )

            lab['_test'] = (
                lab[nc]
                .fillna('UNKNOWN')
                .astype(str)
                .str.upper()
                .str.strip()
            )

            lab = lab[
                lab['_val'].notna()
            ]

            if len(lab):
                top = (
                    lab['_test']
                    .value_counts()
                    .head(25)
                    .index
                )

                lab = lab[
                    lab['_test'].isin(top)
                ].sort_values('_t')

                latest = lab.drop_duplicates(
                    ['encounter_id', '_test'],
                    keep='last'
                )

                wide = latest.pivot(
                    index='encounter_id',
                    columns='_test',
                    values='_val'
                )

                nb_add(
                    wide.astype('float32'),
                    'last24_lab_value'
                )

                print(
                    'LAST-24H LAB RECORDS:',
                    len(lab),
                    flush=True
                )

        del lab


    # 3. Clinical interactions

    interactions = [
        (
            'nova_med_count',
            'nova_stay_days',
            'meds_per_day'
        ),
        (
            'nova_dx_count',
            'nova_stay_days',
            'dx_per_day'
        ),
        (
            'nova_lab_abnormal_count',
            'nova_lab_count',
            'lab_abnormal_rate2'
        ),
        (
            'nova_prior_30d',
            'nova_stay_days',
            'prior30_by_stay'
        ),
        (
            'nova_prior_90d',
            'nova_prior_365d',
            'recent_prior_fraction'
        ),
        (
            'nova_transfer_count',
            'nova_stay_days',
            'transfers_per_day'
        ),
        (
            'nova_icu_count',
            'nova_stay_days',
            'icu_per_day'
        ),
        (
            'nova_lab_last24_abnormal',
            'nova_lab_last24_count',
            'last24_abnormal_rate'
        )
    ]

    for a, b, name in interactions:

        if a in NB_all and b in NB_all:

            x = pd.to_numeric(
                NB_all[a],
                errors='coerce'
            )

            z = pd.to_numeric(
                NB_all[b],
                errors='coerce'
            )

            NB_all[
                'nb_ratio_' + name
            ] = x / (
                1 + z.clip(lower=0)
            )


    NB_all = NB_all.replace(
        [np.inf, -np.inf],
        np.nan
    )

    NB_X = (
        NB_all.iloc[:NOVA_n]
        .reset_index(drop=True)
    )

    NB_T = (
        NB_all.iloc[NOVA_n:]
        .reset_index(drop=True)
    )

    assert len(NB_X) == len(NOVA_y)
    assert len(NB_T) == len(NOVA_T2)

    print(
        '\nNOVA-BOOST FEATURES READY:',
        NB_X.shape,
        'TEST:', NB_T.shape,
        flush=True
    )

    print(
        'NEW SOURCE SUMMARY:',
        NB_added
    )


### Archive N05 — NOVA-BOOST XGBoost/LightGBM and ensemble evaluation

**Original NOVA source cell 5; disabled for automatic execution.** Its results should be reported as a separate experiment, not as the final selected V11/V12 model.

In [ ]:
if False:  # ARCHIVED NOVA SOURCE — not part of Run All execution

    import gc
    import base64
    import numpy as np
    import pandas as pd

    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from xgboost import XGBClassifier
    from lightgbm import (
        LGBMClassifier,
        early_stopping,
        log_evaluation
    )
    from IPython.display import HTML, display

    needed = [
        'NB_X', 'NB_T',
        'NOVA_y', 'NOVA_test_ids',
        'NOVA_oof', 'NOVA_pred',
        'NOVA_scores', 'nova_blend'
    ]

    assert all(
        k in globals() for k in needed
    ), "Run NOVA Cells 1-2 and BOOST Cell A first."


    # Prepare consistent numerical features

    NB_raw = pd.concat(
        [NB_X, NB_T],
        ignore_index=True
    )

    NB_cats = NB_raw.select_dtypes(
        exclude='number'
    ).columns.tolist()

    NB_raw = pd.get_dummies(
        NB_raw,
        columns=NB_cats,
        dummy_na=True,
        dtype=np.uint8
    )

    NB_raw = NB_raw.loc[
        :, ~NB_raw.columns.duplicated()
    ]

    NB_raw = NB_raw.replace(
        [np.inf, -np.inf],
        np.nan
    )

    trainpart = NB_raw.iloc[
        :len(NOVA_y)
    ]

    keep = (
        (trainpart.notna().sum() >= 80)
        &
        (trainpart.nunique(
            dropna=True
        ) > 1)
    )

    NB_raw = NB_raw.loc[:, keep]

    NB_raw.columns = [
        f'nbf_{i}'
        for i in range(
            NB_raw.shape[1]
        )
    ]

    NB_raw = NB_raw.astype(
        np.float32
    )

    NB_train = (
        NB_raw.iloc[:len(NOVA_y)]
        .reset_index(drop=True)
    )

    NB_test = (
        NB_raw.iloc[len(NOVA_y):]
        .reset_index(drop=True)
    )

    print(
        'BOOST TRAIN:', NB_train.shape,
        'TEST:', NB_test.shape,
        flush=True
    )


    # Same folds as the earlier models

    folds = list(
        StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=42
        ).split(
            NB_train, NOVA_y
        )
    )

    NB_oof = {}
    NB_testpred = {}
    NB_report = []


    for model_name in [
        'XGBOOST',
        'LIGHTGBM'
    ]:

        NB_oof[model_name] = np.zeros(
            len(NOVA_y)
        )

        NB_testpred[model_name] = np.zeros(
            len(NB_test)
        )

        for k, (tr, va) in enumerate(
            folds, 1
        ):

            print(
                'NOVA-BOOST',
                model_name,
                'FOLD', k, '/3',
                flush=True
            )

            if model_name == 'XGBOOST':

                model = XGBClassifier(
                    n_estimators=950,
                    max_depth=3,
                    learning_rate=0.035,
                    min_child_weight=18,
                    subsample=0.85,
                    colsample_bytree=0.83,
                    reg_lambda=12,
                    reg_alpha=1,
                    tree_method='hist',
                    objective='binary:logistic',
                    eval_metric='auc',
                    early_stopping_rounds=75,
                    n_jobs=4,
                    random_state=300 + k
                )

                model.fit(
                    NB_train.iloc[tr],
                    NOVA_y[tr],
                    eval_set=[
                        (
                            NB_train.iloc[va],
                            NOVA_y[va]
                        )
                    ],
                    verbose=False
                )

                best_iter = (
                    model.best_iteration + 1
                )

            else:

                model = LGBMClassifier(
                    n_estimators=1050,
                    learning_rate=0.035,
                    num_leaves=11,
                    min_child_samples=100,
                    colsample_bytree=0.83,
                    subsample=0.85,
                    subsample_freq=1,
                    reg_lambda=14,
                    reg_alpha=1,
                    n_jobs=4,
                    verbosity=-1,
                    random_state=400 + k
                )

                model.fit(
                    NB_train.iloc[tr],
                    NOVA_y[tr],
                    eval_set=[
                        (
                            NB_train.iloc[va],
                            NOVA_y[va]
                        )
                    ],
                    eval_metric='auc',
                    callbacks=[
                        early_stopping(
                            85, verbose=False
                        ),
                        log_evaluation(0)
                    ]
                )

                best_iter = (
                    model.best_iteration_
                )

            NB_oof[model_name][va] = (
                model.predict_proba(
                    NB_train.iloc[va]
                )[:, 1]
            )

            NB_testpred[model_name] += (
                model.predict_proba(
                    NB_test
                )[:, 1] / 3
            )

            train_auc = roc_auc_score(
                NOVA_y[tr],
                model.predict_proba(
                    NB_train.iloc[tr]
                )[:, 1]
            )

            val_auc = roc_auc_score(
                NOVA_y[va],
                NB_oof[model_name][va]
            )

            NB_report.append((
                model_name,
                k,
                train_auc,
                val_auc,
                train_auc - val_auc,
                best_iter
            ))

            print(
                'VAL:', round(val_auc, 6),
                'GAP:',
                round(
                    train_auc - val_auc,
                    6
                ),
                'TREES:', best_iter,
                flush=True
            )

            del model
            gc.collect()


    # Blend against the original NOVA winner

    NB_oldname = NOVA_scores[0][2]

    NB_old_oof = nova_blend(
        NB_oldname,
        NOVA_oof
    )

    NB_old_test = nova_blend(
        NB_oldname,
        NOVA_pred
    )

    NB_new_oof = (
        0.5 * NB_oof['XGBOOST']
        + 0.5 * NB_oof['LIGHTGBM']
    )

    NB_new_test = (
        0.5 * NB_testpred['XGBOOST']
        + 0.5 * NB_testpred['LIGHTGBM']
    )


    NB_candidates = {
        'NOVA_ORIGINAL': (
            NB_old_oof,
            NB_old_test
        ),

        'BOOST_XGBOOST': (
            NB_oof['XGBOOST'],
            NB_testpred['XGBOOST']
        ),

        'BOOST_LIGHTGBM': (
            NB_oof['LIGHTGBM'],
            NB_testpred['LIGHTGBM']
        ),

        'BOOST_NEW_50_50': (
            NB_new_oof,
            NB_new_test
        ),

        'BOOST_OLD20_NEW80': (
            0.2 * NB_old_oof
            + 0.8 * NB_new_oof,

            0.2 * NB_old_test
            + 0.8 * NB_new_test
        ),

        'BOOST_OLD50_NEW50': (
            0.5 * NB_old_oof
            + 0.5 * NB_new_oof,

            0.5 * NB_old_test
            + 0.5 * NB_new_test
        ),

        'BOOST_OLD70_XGB30': (
            0.7 * NB_old_oof
            + 0.3 * NB_oof['XGBOOST'],

            0.7 * NB_old_test
            + 0.3 * NB_testpred['XGBOOST']
        )
    }


    NB_ranking = sorted(
        [
            (
                roc_auc_score(
                    NOVA_y,
                    p[0]
                ),
                name
            )
            for name, p in NB_candidates.items()
        ],
        reverse=True
    )


    print(
        '\nNOVA-BOOST OOF COMPARISON:'
    )

    for auc, name in NB_ranking:

        fold_gains = [
            roc_auc_score(
                NOVA_y[va],
                NB_candidates[name][0][va]
            )
            -
            roc_auc_score(
                NOVA_y[va],
                NB_old_oof[va]
            )
            for tr, va in folds
        ]

        print(
            name,
            'AUC=', round(auc, 6),
            'FOLD GAINS=',
            np.round(
                fold_gains, 6
            ),
            flush=True
        )


    print(
        '\nTRAINING VS VALIDATION GAP'
    )

    print(
        pd.DataFrame(
            NB_report,
            columns=[
                'model',
                'fold',
                'train_auc',
                'val_auc',
                'gap',
                'trees'
            ]
        ).round(5).to_string(
            index=False
        )
    )

    print(
        '\nV9 HISTORICAL OFFLINE: 0.748383'
    )

    print(
        'V9 PUBLIC: 0.74214'
    )


    best_auc, best_name = (
        NB_ranking[0]
    )

    if best_auc < 0.748383:
        print(
            '\nWARNING: Still below V9 offline.',
            'Do not submit automatically.'
        )
    else:
        print(
            '\nAbove V9 offline.',
            'Review fold consistency before submission.'
        )


    # Save both candidates

    for filename, name in [
        (
            'NOVA_BOOST_BEST.csv',
            best_name
        ),
        (
            'NOVA_BOOST_XGB.csv',
            'BOOST_XGBOOST'
        )
    ]:

        pred = np.asarray(
            NB_candidates[name][1]
        )

        assert len(pred) == len(
            NOVA_test_ids
        )

        assert np.isfinite(
            pred
        ).all()

        assert (
            (pred >= 0) & (pred <= 1)
        ).all()

        out = pd.DataFrame({
            'encounter_id':
                NOVA_test_ids.to_numpy(),

            'readmit_30d': pred
        })

        assert out.encounter_id.is_unique

        path = (
            '/kaggle/working/'
            + filename
        )

        out.to_csv(
            path,
            index=False
        )

        with open(path, 'rb') as f:
            encoded = base64.b64encode(
                f.read()
            ).decode('ascii')

        display(
            HTML(
                '<a download="'
                + filename
                + '" href="data:text/csv;base64,'
                + encoded
                + '">DOWNLOAD '
                + filename
                + '</a>'
            )
        )

        print(
            'SAVED:', path,
            'MODEL:', name
        )


# Appendix C — Reproducibility and academic-integrity checklist

1. Attach the **original** competition CSV dataset to Kaggle; confirm that all required files are found in Section 1.
2. Select **Restart Session → Run All** and allow the original V5/V8/V9/V10/V11/V12 models to train. This may take a long time; the archived source cells do not execute.
3. Confirm the three final CSVs and `submission.csv` exist and contain exactly the test encounter IDs in the original order, unique IDs and probabilities in [0,1].
4. Confirm the V11 feature count and export `FINAL_FEATURE_COLUMNS.csv`; manually check ambiguous lineage mappings. A full list of 855 feature names is produced **only when the original data and feature pipeline yield 855 columns**.
5. Save the executed notebook and the audit/feature/model CSV outputs. Do **not** report a fresh run as successful if any cell fails.
6. In the written report, distinguish **historical public/private Kaggle scores** from **recomputed OOF/stress-test metrics**.
7. State that original exploratory experiments included some unsupervised train/test-wide category discovery, and that full fold-isolated preprocessing and fully nested stacking would be needed for a strict deployment-grade estimate.
8. The professor requires training-only fit of imputers/scalers/encoders; the original V5 XGBoost preprocessing meets this for each fold, while some later original feature-vocabulary steps do **not**. This must not be concealed or described as perfectly fold-isolated.
9. There is no original V0/V7/V13 code in the supplied materials; do not claim these experiments were executed.

**Conclusion:** This notebook consolidates the actual historical work and defines an executable Kaggle reproduction path. Independent execution against the original raw CSVs is still required to verify the full path and to substantiate any claim of bit-for-bit reproduction.